

---

<!-- Source: README.md -->



# _docs — Module Documentation

Flat file structure with compound numbering. Main sections: `NN_module.md`, subsections: `NNSS_topic.md`.

See `.agent/skills/docs_skill.md` for conventions and the doc file template.

**Ordering rule: within every topic block, methodology/concept docs have lower numbers than
their corresponding code docs.**

---

## Structure

```
_doc_/
  0000_project_overview.md

  # Database Infrastructure (1xxx)
  1000_database.md
  1001_database_module.md
  1100_store.md + 1110–1150 (duckdb_store, query, stats, validate, toolkit)
  1200_sync_tables.md
  1210_sync_ohlcv.md
  1230_sync_predictions.md
  1300_tests.md + 1310–1320 (store, pipeline)

  # Features (2xxx) — methodology BEFORE code
  2000_features.md           ← feature layer: 25 groups, lag, warmup (METHODOLOGY)
  2010_feature_engineering.md ← feature selection: quality, redundancy, stability (METHODOLOGY)
  2100_sync_features.md      ← sync_features.py (CODE)
  2200_features_polars.md    ← _features_polars.py (CODE)

  # Targets (3xxx) — methodology BEFORE code
  3000_targets.md            ← fw60 logreturn outcomes, MFE (METHODOLOGY)
  3100_sync_targets.md       ← sync_targets.py (CODE)

  # Quant Train (4xxx) — methodology BEFORE code
  4000_quant_train.md        ← INNER JOIN handoff, rebuild semantics (METHODOLOGY)
  4100_quant_train.md        ← schema, rebuild modes, CLI (CODE)

  # Sampling / Modelling (5xxx)
  5000_modelling.md          ← modeling domain overview / TOC
  5010_sampling_yearly.md    ← yearly random-hour sampling (METHODOLOGY, active)
  5100_sampling_config.md    ← YearlySamplingConfig dataclass
  5200_sampling_artifacts.md ← write_yearly_artifacts / load_yearly_sample
  5300_create_sample.md      ← create_yearly_sample orchestrator + CLI
  5400_sampling.md           ← ARCHÍV: expanding window CV
  5410_sampling_splits.md    ← ARCHÍV: expanding window splits
  5420_sampling_audit.md     ← ARCHÍV: feature table audit

  analysis/                  ← analyst_agent: EDA, specs, sample quality notebooks
```




---

<!-- Source: 0000_project_overview.md -->



﻿# ChronoQuant — Project Overview

Single source of truth for the orchestrator. Agents load their own module docs
from `_doc_/<module>/`; this file is for cross-domain context only.

---

## Business Goal

ChronoQuant is an algorithmic crypto trading system targeting **SOLUSDT perpetual
futures on Binance**. The core loop:

1. Sync 1-minute OHLCV candles from Binance into a local DuckDB store.
2. Compute quantitative features (momentum, volume, volatility) over that data.
3. Train LightGBM regressors to predict the next 60-bar forward MFE (Maximum
   Favorable Excursion) — a continuous logreturn outcome — for long and short direction.
4. The trading module calibrates entry/exit rules against out-of-sample model
   output, producing a strategy artifact.
5. A Streamlit dashboard consumes live predictions and strategy rules, shows
   signals, and exposes live trading controls.

Elliott wave analysis (`research/elliott/`) is a parallel research module —
it does not feed the live trading pipeline.

---

## Module Architecture

Four production modules with clear, non-overlapping responsibilities:

### `src/data_handling/` — Operational data layer
Owns all live data: ingestion, storage, sync, and validation.
- Syncs raw OHLCV from Binance
- Computes and syncs features, targets, predictions into DuckDB
- No modeling logic; no strategy logic

### `src/modeling/` — Offline ML development
Produces model artifacts from historical data. Runs ad-hoc, not in production.
- Creates yearly samples (parquet), searches hyperparameters, fits final models
- Outputs: `model.pkl`, `features.json`, `params.json`, `sample_oos.parquet`
- Does not operate live; does not decide thresholds or rules

### `src/trading/` — Strategy + live operations
Consumes model artifacts to calibrate strategy rules, then runs them live.
- Measures strategy performance on OOF predictions → produces strategy artifact
- Runs the live trading loop: reads `predictions` table, applies rules, journals trades
- Owns: thresholds, cut-offs, position sizing, cooldowns, hold times

### `src/ui/` — Display layer
Reads from the database and from artifacts; does not write business data.
- Shows live predictions, signals, equity curve, trade journal
- Exposes live trading controls (start/stop, parameter overrides)

### `research/` — Sandbox
Exploratory code, prototypes, and ideas not yet ready for production.
- `research/elliott/` — Elliott wave parser, validators, scanners (isolated)
- No code here feeds the live pipeline

---

## Data Flow

```
Binance API
    │
    ▼
database/  ──sync──▶  ohlcv  ──sync──▶  feat_ohlcv_quant  ──sync──▶  target
                                                │                         │
                                                └──────────┬─────────────┘
                                                           │
modeling/                                           quant_train (ad-hoc build)
  0_create_sample   ◀──────────────────────────────────────┘
  1_feature_engineering
  2_hyper_param_search
  3_fit_model  ──────────▶  model artifact (model.pkl + features.json)
               ──────────▶  sample_oos.parquet

trading/
  0_measure_strategy  ◀──  sample_oos.parquet
                      ──▶  strategy artifact (thresholds, rules)
  live loop           ◀──  predictions table (database sync)
                      ◀──  strategy artifact
                      ──▶  trade journal

database/  ◀──  prediction sync uses model artifact  ──▶  predictions table

ui/  ◀──  predictions table + strategy artifact + trade journal
```

---

## Persistence Rules

| Data type | Where | Reason |
|-----------|-------|--------|
| Live OHLCV, features, targets, predictions | DuckDB | Synced, queryable, updatable |
| `quant_train` | DuckDB | Ad-hoc join table, rebuilt before training |
| Yearly samples | Parquet (`database/<asset>/samples/<id>/`) | Static snapshots, not synced |
| OOS predictions | Parquet (`database/<asset>/samples/<sample_id>/sample_oos.parquet`) | Static, per-model OOS output |
| Feature engineering analysis | `artifacts/<model_id>/feature_engineering/` (`.ipynb`, `.html`, `feature_set.json`) | Per-model, analyst output |
| Model artifacts | `artifacts/<model_id>/` (`manifest.json`, `model.pkl`, `features.json`, `params.json`, `search/`) | Runtime use by prediction sync |
| Strategy artifacts | `artifacts/<model_id>/strategy/` | Runtime use by trading loop |

**Rule:** if a table needs to be synced incrementally → DuckDB. If it is a static
snapshot produced by a modeling or analysis run → Parquet.

---

## Repository Layout

```
src/
  data_handling/    Operational data layer
    store/            DuckDB store, queries, validation, stats
    sync_tables/      OHLCV sync, feature sync, prediction sync, target sync
    tests/            Tests (store/, sync_tables/ — smoke, sanity, perf, integration)
    01_validate_stats.py
    02_sync_pipeline.py

  modeling/         Offline ML development
    sampling/           Yearly sample creation: config, sampler, audit, artifacts
    training/           LightGBM trainer, CV, datasets, metrics, reports, artifacts
    evaluation/         Backtest runner, metrics
    search/             Hyperparameter search (LightGBM + Optuna)
    feature_engineering/  Feature quality, target-relation, redundancy, stability library
    text/               Future placeholder
    00_create_sample.py
    01_feature_engineering.ipynb
    02_hyper_param_search.py
    03_fit_model.py

  trading/          Strategy calibration + live operations
    calibration/        Backtest engine, calibration orchestrator, strategy artifacts
    live/               TradingService, exchange client, journal, state machine, strategy evaluator
    00_calibrate_strategy.py
    01_sweep_strategy.py
    02_run_service.py

  ui/               Streamlit dashboard (pages, components, data loading)
  utils.py          All config loading — single entry point, never read JSON directly

research/           Sandbox — explorations not yet production-ready
  elliott/            Elliott wave parser, validators, scanners, backtest

src/analyst/        Analyst Python segédmodulok (table_formatting, plot_utils, db_utils, CSS, _quarto.yml)
_doc_/              Module documentation + analyst notebooks
  XXXX_*.ipynb        Analyst notebookok (közvetlenül itt)
  XXXX_*.html         Quarto-rendered HTML output
_jira_/             Local task tracking (epics → tasks → stories); jira.json = epic counter
.agent/             Agent rules, skills, tool docs
config/             JSON config files (assets, features, models, strategies, trading…)
artifacts/          Model development artifacts — one folder per model_id
  <model_id>/
    manifest.json                   Pipeline state + model summary
    model.pkl / features.json / params.json
    sample_oos.parquet              OOS predictions (produced by train step)
    search/                         Hyperparameter search results
    feature_engineering/            01_feature_engineering.ipynb + .html + feature_set.json
    strategy/                       strategy_artifact.json + sweep_results.csv + report.html
database/           DuckDB files + static sample snapshots (read-only source for training)
  solusdt/
    solusdt.duckdb
    samples/<sample_id>/            metadata.json, audit.json, sample_train_valid.parquet
```

---

## Database (DuckDB)

**One DuckDB file per asset:** `database/<asset_id>/<asset_id>.duckdb`

Currently only one active asset: **solusdt** (SOLUSDT, 1m, futures).

### Tables

| Table | Primary Key | Purpose |
|-------|-------------|---------|
| `ohlcv` | `open_time` TIMESTAMP | Raw 1-minute candles from Binance |
| `target` | `open_time` TIMESTAMP | fw60 forward outcomes: `long_mfe_fw60`, `short_mfe_fw60` + 8 further columns |
| `feat_ohlcv_quant` | `open_time` TIMESTAMP | Quantitative features (`feat_` prefix) |
| `predictions` | `open_time` TIMESTAMP | Model probability scores + signals |
| `quant_train` | `open_time` TIMESTAMP | Ad-hoc join: all `feat_*` + `long_mfe_fw60` + `short_mfe_fw60`; NULL targets excluded |

`quant_train` is rebuilt ad-hoc before training via `src/data_handling/03_build_quant_train.py`.
Full rebuild = `CREATE OR REPLACE TABLE`; range rebuild = DELETE + INSERT.

All timestamps are **UTC, format `YYYY-MM-DD HH:MM:SS`** (naive strings treated as UTC).
Epoch milliseconds used internally as `open_time_ms`.

All sync operations are **idempotent upserts keyed on `open_time`** — safe to re-run.

Config is always accessed via `src/utils.py` — never read JSON config files directly.

---

## Modeling Pipeline

### Model naming convention

```
lgbm_{asset}_{direction}_fw{horizon}_{year}
```
pl. `lgbm_solusdt_l_fw60_2021`, `lgbm_solusdt_s_fw60_2023`

**Active models:** 10 éves modell (2021–2025 × long + short) — mind `active: false` amíg nem kerül kiválasztásra élő kereskedésre. Config: `config/models.json` (schema v4).

- **Target semantics:** `fw60` = 60-bar forward window; `long_mfe_fw60` = log(max upside / close[t]); `short_mfe_fw60` = log(min downside / close[t]). Folytonos regressziós target — nincs percentilis küszöb, nincs binarizálás.
- **Feature prefix:** `feat_` | **t-1 lag mandatory** on all features (prevents data leakage).
- **Feature engineering target:** only the model's own direction target is used (`l` → `long_mfe_fw60`, `s` → `short_mfe_fw60`).
- **Artifacts:** `artifacts/<model_id>/` — `manifest.json`, `model.pkl`, `features.json`, `params.json`, `search/`, `feature_engineering/`.
- **Samples:** read-only forrás `database/solusdt/samples/solusdt_fw60_yearly_{year}/`; nem másolódik az artifact-ba, csak hivatkozik rá (`sampling.sample_dir`).

### Pipeline (runs offline, in order)

```bash
# Teljes pipeline egy modellre:
uv run python src/modeling/pipeline.py --model lgbm_solusdt_l_fw60_2021

# Egyes lépések:
uv run python src/modeling/pipeline.py --model lgbm_solusdt_l_fw60_2021 --step setup
uv run python src/modeling/pipeline.py --model lgbm_solusdt_l_fw60_2021 --step feature_engineering
uv run python src/modeling/pipeline.py --model lgbm_solusdt_l_fw60_2021 --step search --stage smoke
uv run python src/modeling/pipeline.py --model lgbm_solusdt_l_fw60_2021 --step train
```

| Lépés | Input | Output (artifact-ban) |
|-------|-------|----------------------|
| `setup` | `config/models.json` | `manifest.json` |
| `feature_engineering` | `samples/{sample_id}/sample_train_valid.parquet` (via DuckDB) | `feature_engineering/01_fe.ipynb`, `.html`, `feature_set.json` |
| `search` | sample parquet + feature_set.json | `search/search_best.json`, `search_trials.jsonl` |
| `train` | sample parquet + search results | `model.pkl`, `features.json`, `params.json`, `sample_oos.parquet` |

### Yearly sample model

One sample = one calendar year. Sample ID: `{asset_id}_fw60_yearly_{year}`.

**Segments:** `train` / `valid` / `purge` — no test holdout within the sample.
Test evaluation uses a separate future-year OOS (see OOS Evaluation section).

```
sample_train_valid.parquet columns:
  open_time | long_mfe_fw60 | short_mfe_fw60 | segment

sample_oos.parquet columns (artifacts/<model_id>/sample_oos.parquet):
  l-irányú model: open_time | pred_long  | long_mfe_fw60 | short_mfe_fw60
  s-irányú model: open_time | pred_short | long_mfe_fw60 | short_mfe_fw60
```

Features are NOT stored in the sample — loaded from `quant_train` at training time.
Samples are parquet only — no DuckDB materialization.
Methodology details: `_doc_/5010_sampling_yearly.md`.

### OOS evaluation

OOS (out-of-sample) is always a **separate, future calendar year** — not a holdout
month from the training year. This ensures all seasonal effects are represented in
the training data, and the OOS is a genuinely unseen period.

```
2021 model  →  trained on 2021 sample  →  OOS scored on 2022
2022 model  →  trained on 2022 sample  →  OOS scored on 2023
...
```

`uv run python src/modeling/03_fit_model.py --model lgbm_solusdt_l_fw60_2021` produces:
1. Final model refitted on all train+valid rows of the 2021 sample.
2. `sample_oos.parquet` — `predict()` (continuous regression output) applied to the full `oos_year` (2022) dataset.

The trading module uses `sample_oos.parquet` for strategy calibration.

---

## Trading Strategy

The trading module calibrates strategy rules offline, then runs them live.

**Offline calibration (`00_calibrate_strategy.py --model <model_id>`):**
- Reads `artifacts/<model_id>/sample_oos.parquet` (OOS predictions + targets)
- Runs backtest with default parameters, produces `strategy_artifact.json`
- Sweep variant: `01_sweep_strategy.py` — grid search over entry/hold/TP combinations

**Live state machine (`src/trading/live/strategy.py`):**
- **States:** FLAT → LONG / SHORT → COOLDOWN → FLAT
- **Entry:** `pred_long >= entry_threshold` → ENTER_LONG (long has priority if both fire)
- **Exit:** max hold time elapsed OR stop-loss triggered → EXIT, enter COOLDOWN
- **Rearm:** both model probs must cool below `rearm_threshold` before next entry
- All thresholds and cooldowns come from `strategy_artifact.json`

---

## Testing Rules

| What to run | When |
|-------------|------|
| `uv run pyright src/<module>/` | After any type-annotated change |
| `ruff check src/<module>/ --fix` | Before committing any Python file |
| `uv run pytest src/data_handling/tests/ -v` | Store or pipeline changes |
| `uv run pytest src/modeling/ -v` | Modeling changes |
| `uv run pytest src/trading/tests/ -v` | Trading calibration or live service changes |
| `STREAMLIT_CONFIG_DIR=src/ui uv run streamlit run src/ui/main.py` | UI changes (manual smoke test) |

Always run pyright and ruff for the affected module. Never skip for non-trivial changes.

---

## Key Conventions

- **Config gateway:** all config through `src/utils.py` — no raw JSON reads in `src/`
- **Active asset:** `solusdt` — do not spend time on inactive asset paths
- **Polars for features:** feature computation uses Polars; pandas allowed elsewhere
- **No print() in library code** — use `logging` or `st.*`
- **Upserts only** — no delete/truncate patterns in sync operations
- **DuckDB = synced/live, Parquet = static snapshots** — never invert this
- **Elliott and research are isolated** — nothing in `research/` feeds the live pipeline

---

## Agent Ownership

| Agent | Owns |
|-------|------|
| Database Agent | `src/data_handling/`, `config/assets.json`, DuckDB schema |
| Modeling Agent | `src/modeling/`, feature computation, model artifacts |
| UI Agent | `src/ui/`, `src/trading/` |
| Code Doc Agent | `.agent/`, tooling, infra, dependencies; `_doc_/` X110+ code reference files |
| Analyst Agent | `_doc_/XXXX_*.ipynb` (elemzési notebookok), `src/analyst/` (Python segédmodulok: `table_formatting`, `plot_utils`, `db_utils`, CSS, `_quarto.yml`); user-célból indul, nem spec-ből; interaktív session |
| Methodology Agent | `_doc_/` X000, X100 levels — business rationale, methodological decisions, parameter justification |
| Validator Agent | `pr_` ticket validation: ruff + pyright + pytest, then `done_` promotion |




---

<!-- Source: 1000_database.md -->



﻿# Database — DuckDB Schema

A ChronoQuant egyetlen DuckDB fájlban tárolja az összes piaci adatot, feature-t és predikciót egy assetenként.

---

## Áttekintés

```{mermaid}
erDiagram
    ohlcv {
        TIMESTAMP open_time PK
        DOUBLE open
        DOUBLE high
        DOUBLE low
        DOUBLE close
        DOUBLE volume
        DOUBLE quote_volume
        BIGINT trades
        DOUBLE taker_buy_base
        DOUBLE taker_buy_quote
    }

    target {
        TIMESTAMP open_time PK
        DOUBLE close
        DOUBLE long_mfe_fw60 "log(max_fw60/close) — LONG TARGET"
        DOUBLE short_mfe_fw60 "log(min_fw60/close) — SHORT TARGET"
        DOUBLE fw60_close "close[t+60]"
        DOUBLE fw60_max "max(close[t+1..t+60])"
        DOUBLE fw60_min "min(close[t+1..t+60])"
        DOUBLE fw60_close_ret "simple return"
        DOUBLE fw60_close_logret "log return"
        DOUBLE fw60_max_ratio "max/close"
        DOUBLE fw60_min_ratio "min/close"
    }

    feat_ohlcv_quant {
        TIMESTAMP open_time PK
        DOUBLE close
        TIMESTAMP available_ts
        TIMESTAMP lookback_end_ts
        DOUBLE feat_cols "feat_* oszlopok (config-driven, ~100+)"
    }

    predictions {
        TIMESTAMP open_time PK
        DOUBLE close
        TIMESTAMP label_end_ts
        DOUBLE long_pred "long model prediction score"
        DOUBLE short_pred "short model prediction score"
    }

    quant_train {
        TIMESTAMP open_time PK
        DOUBLE feat_cols "összes feat_* oszlop (feat_ohlcv_quant-ból)"
        DOUBLE long_mfe_fw60 "fw60 long outcome"
        DOUBLE short_mfe_fw60 "fw60 short outcome"
    }

    ohlcv ||--o{ target : "open_time"
    ohlcv ||--o{ feat_ohlcv_quant : "open_time"
    feat_ohlcv_quant ||--o{ predictions : "available_ts ASOF join"
    feat_ohlcv_quant ||--o{ quant_train : "INNER JOIN open_time"
    target ||--o{ quant_train : "INNER JOIN open_time"
```

Minden tábla `open_time` TIMESTAMP primary key-en alapul. Az összes timestamp **UTC**, `YYYY-MM-DD HH:MM:SS` formátumban tárolva.

**DuckDB fájl helye:** `database/<asset_id>/<asset_id>.duckdb`

Aktív asset: `solusdt` → `database/solusdt/solusdt.duckdb`

Az elérési út mindig a `config/assets.json` → `utils.load_asset_config(asset_id)` → `db_path` mezőjéből jön.

---

## Táblák

### ohlcv

**Cél:** Nyers, változatlan Binance 1-perces kline adatok. A pipeline alapja — minden downstream tábla ebből épül fel.

**Beírási mód:** append-only. Csak a tárolt `MAX(open_time)`-nál újabb sorok kerülnek be (`_insert_append_only`). Nincs upsert, nincs törlés.

| Oszlop | Típus | Leírás |
|--------|-------|--------|
| `open_time` | `TIMESTAMP` (PK) | Gyertya nyitásának időpontja, UTC. Minden sor egyedi. |
| `open` | `DOUBLE` | Nyitóár USDT-ben |
| `high` | `DOUBLE` | Legmagasabb ár az 1 perces ablakban |
| `low` | `DOUBLE` | Legalacsonyabb ár az 1 perces ablakban |
| `close` | `DOUBLE` | Záróár USDT-ben |
| `volume` | `DOUBLE` | Forgalom base asset-ben (SOL) |
| `quote_volume` | `DOUBLE` | Forgalom quote asset-ben (USDT) |
| `trades` | `BIGINT` | Kötések száma az 1 perces ablakban |
| `taker_buy_base` | `DOUBLE` | Taker vevő forgalom base asset-ben (SOL) |
| `taker_buy_quote` | `DOUBLE` | Taker vevő forgalom quote asset-ben (USDT) |

**Nem tárolt Binance mezők:** `close_time` (redundáns), `ignore` (deprecated).

```sql
CREATE TABLE IF NOT EXISTS ohlcv (
    open_time       TIMESTAMP PRIMARY KEY,
    open            DOUBLE,
    high            DOUBLE,
    low             DOUBLE,
    close           DOUBLE,
    volume          DOUBLE,
    quote_volume    DOUBLE,
    trades          BIGINT,
    taker_buy_base  DOUBLE,
    taker_buy_quote DOUBLE
)
```

---

### target

**Cél:** Folytonos forward logreturn outcome-ok (fw60). Az `ohlcv.close` alapján, DuckDB SQL ablakfüggvényekkel számítva. Teljes rebuild minden `sync_targets` híváskor.

**Beírási mód:** DELETE + INSERT a teljes tartományra (`insert_target`). Az előre definiált időablakban (`ROWS BETWEEN 1 FOLLOWING AND 60 FOLLOWING`) az aktuális bar (`t`) NEM szerepel a forward window-ban.

**NULL sorok:** Az utolsó 60 sor minden fw60 outcome oszlopban `NULL` — nincs elegendő jövőbeli adat.

**Kód referencia:** [`_doc_/3100_sync_targets.md`](_doc_/3100_sync_targets.md) | **Metodológia:** [`_doc_/3000_targets.md`](_doc_/3000_targets.md)

| Oszlop | Típus | Leírás |
|--------|-------|--------|
| `open_time` | `TIMESTAMP` (PK) | Bar nyitási ideje, UTC |
| `close` | `DOUBLE` | Bar záróára (referencia close[t]) |
| `fw60_close` | `DOUBLE` | close[t+60] — nyers forward close |
| `fw60_max` | `DOUBLE` | max(close[t+1:t+60]) |
| `fw60_min` | `DOUBLE` | min(close[t+1:t+60]) |
| `fw60_close_ret` | `DOUBLE` | close[t+60] / close[t] − 1 |
| `fw60_close_logret` | `DOUBLE` | log(close[t+60] / close[t]) |
| `fw60_max_ratio` | `DOUBLE` | max(close[t+1:t+60]) / close[t] |
| `fw60_min_ratio` | `DOUBLE` | min(close[t+1:t+60]) / close[t] |
| **`long_mfe_fw60`** | **`DOUBLE`** | **log(max(close[t+1:t+60]) / close[t]) — LONG TARGET** |
| **`short_mfe_fw60`** | **`DOUBLE`** | **log(min(close[t+1:t+60]) / close[t]) — SHORT TARGET** |

```sql
CREATE TABLE IF NOT EXISTS target (
    open_time        TIMESTAMP PRIMARY KEY,
    close            DOUBLE,
    fw60_close       DOUBLE,
    fw60_max         DOUBLE,
    fw60_min         DOUBLE,
    fw60_close_ret   DOUBLE,
    fw60_close_logret DOUBLE,
    fw60_max_ratio   DOUBLE,
    fw60_min_ratio   DOUBLE,
    long_mfe_fw60    DOUBLE,
    short_mfe_fw60   DOUBLE
)
```

---

### feat_ohlcv_quant

**Cél:** Technikai indikátorok és feature-ök (Polars LazyFrame pipeline). Séma config-driven — az oszlopok száma és neve a `config/features.json` indikátor konfigurációjától függ (~100+ `feat_*` oszlop).

**Beírási mód:** append-only (`_insert_append_only`). A séma automatikusan bővül az első `insert_feat_ohlcv_quant` hívásnál (tábla létrehozása) és ha új feature oszlopok jelennek meg.

**t-1 lag:** Minden OHLCV-alapú feature 1 barral el van tolva (`shift(1)`) a `compute_features_polars` hívásban. A P2 (időindexes) feature-ök kivételek — ezek nem kerülnek eltolásra (`T_MINUS_1_SKIP`).

| Oszlop | Típus | Leírás |
|--------|-------|--------|
| `open_time` | `TIMESTAMP` (PK) | Bar nyitási ideje, UTC |
| `close` | `DOUBLE` | Bar záróára (referencia) |
| `available_ts` | `TIMESTAMP` | Mikor válik elérhetővé a feature (== `open_time`, t-1 lag garantált) |
| `lookback_end_ts` | `TIMESTAMP` | A lookback ablak vége (== `open_time`) |
| `feat_*` | `DOUBLE` / `BOOLEAN` | Config-driven feature oszlopok, `feat_` prefix |

Az ASOF join (`predictions` ↔ `feat_ohlcv_quant`) az `available_ts` oszlopon alapul: `p.open_time >= f.available_ts`.

---

### predictions

**Cél:** Champion modellek által generált predikciós pontszámok. Egy sor per `open_time`, unified long+short output.

**Beírási mód:** append-only (`_insert_append_only`). A séma rögzített — `ensure_tables` hozza létre.

| Oszlop | Típus | Leírás |
|--------|-------|--------|
| `open_time` | `TIMESTAMP` (PK) | Bar nyitási ideje, UTC |
| `close` | `DOUBLE` | Bar záróára (az `ohlcv.close`-val egyezik) |
| `label_end_ts` | `TIMESTAMP` | A forward window vége: `open_time + fw_minutes` |
| `long_pred` | `DOUBLE` | Long modell predikciós értéke (`predict_proba` vagy `predict`, config szerint) |
| `short_pred` | `DOUBLE` | Short modell predikciós értéke (`predict_proba` vagy `predict`, config szerint) |

```sql
CREATE TABLE IF NOT EXISTS predictions (
    open_time    TIMESTAMP PRIMARY KEY,
    close        DOUBLE,
    label_end_ts TIMESTAMP,
    long_pred    DOUBLE,
    short_pred   DOUBLE
)
```

**Legacy oszlopok:** `dataset_split`, `fold_id`, valamint a régi `trg_*` bináris target oszlopok — ha jelen vannak, az `ensure_tables` migráció során `ALTER TABLE DROP COLUMN`-nal törlődnek.

---

### quant_train

**Cél:** Model-ready join tábla — az összes `feat_*` feature és a két aktív fw60 target oszlop (`long_mfe_fw60`, `short_mfe_fw60`) egyetlen lekérdezhető táblaként. A feature engineering, sampling és LightGBM tanítás kiindulópontja.

**Forrás:** `feat_ohlcv_quant` INNER JOIN `target` ON `open_time`. NULL target sorok kizárva.

**Beírási mód:** Ad-hoc rebuild, NEM a live sync pipeline része. Tanítás előtt futtatandó:
- Full rebuild: `CREATE OR REPLACE TABLE` (determinisztikus)
- Range rebuild: `DELETE + INSERT` a megadott `open_time` ablakra

**CLI:** `uv run python src/database/03_build_quant_train.py [--start YYYY-MM-DD] [--end YYYY-MM-DD]`

**Kód referencia:** [`_doc_/4100_quant_train.md`](_doc_/4100_quant_train.md)

| Oszlop | Típus | Leírás |
|--------|-------|--------|
| `open_time` | `TIMESTAMP` (PK) | Bar nyitási ideje, UTC. Egyedi — INNER JOIN garantálja. |
| `feat_*` | `DOUBLE` | Az összes `feat_ohlcv_quant`-ban szereplő feature oszlop (t-1 lag már alkalmazva) |
| `long_mfe_fw60` | `DOUBLE` | Fw60 long outcome: `log(max_fw60 / close[t])`. NULL sorok kizárva. |
| `short_mfe_fw60` | `DOUBLE` | Fw60 short outcome: `log(min_fw60 / close[t])`. NULL sorok kizárva. |

> **Megjegyzés:** A `quant_train` nem tartalmaz `close`, `available_ts`, `label_end_ts` vagy predikció oszlopokat.
> A legacy `trg_*` boolean target elnevezés NEM kerül felhasználásra ebben a rétegben.

---

## Általános konvenciók

| Szabály | Részlet |
|---------|---------|
| **Timestamp formátum** | UTC, `YYYY-MM-DD HH:MM:SS` (naiv string, UTC-ként értelmezve) |
| **Epoch ms** | Csak Binance API-val való kommunikációban, nem tárolva |
| **Config gateway** | Mindig `utils.load_asset_config(asset_id)` → `db_path` |
| **Idempotens upsert** | Minden sync operáció biztonságosan újrafuttatható |
| **Zonemap** | Sorok `open_time` szerint rendezve kerülnek be (DuckDB range query optimalizálás) |




---

<!-- Source: 1001_database_module.md -->



# src/database/ — Database Module

A `src/database/` modul kezeli az összes piaci adatot: Binance OHLCV szinkront, feature számítást, target labeleket, predikciók beírását és a DuckDB store réteget. Ez a fő adatvezeték, amelyből a modeling és a UI olvas.

---

## Modul struktúra

```
src/database/
├── store/                  DuckDB store réteg (írás, olvasás, validáció, statisztikák)
├── sync_tables/            Sync pipeline (ohlcv → features → targets → predictions)
├── tests/                  Pytest tesztek (smoke, sanity, perf, integration)
├── 01_validate_stats.py    CLI: DB stat riport
└── 02_sync_pipeline.py     CLI: Unified sync belépési pont (OHLCV + derived táblák)
```

Részletes dokumentáció:
- Store réteg → [1100_store.md](1100_store.md)
- Sync tables → [1200_sync_tables.md](1200_sync_tables.md)
- Tesztek → [1300_tests.md](1300_tests.md)
- DuckDB schema → [1000_database.md](1000_database.md)

---

## Adatfolyam

```{mermaid}
flowchart TD
    BINANCE["Binance API\n(klines)"]
    OHLCV["ohlcv\ntábla"]
    FEAT["feat_ohlcv_quant\ntábla"]
    TARGET["target\ntábla"]
    PRED["predictions\ntábla"]

    BINANCE -->|sync_pipeline\nsync_ohlcv| OHLCV
    OHLCV -->|sync_pipeline\nsync_features| FEAT
    OHLCV -->|sync_pipeline\nsync_targets| TARGET
    FEAT -->|sync_pipeline\nsync_predictions| PRED
    TARGET -.-> PRED
```

Minden réteg az előző réteg `MAX(open_time)` értékétől indul — az operációk egymásra épülnek és idempotensek.

---

## Entry point scriptek

### `01_validate_stats.py`

**Célja:** Gyors DB egészség-ellenőrzés — megjeleníti az összes tábla sorát, időtartományát, null arányát és lekérdezési teljesítményét.

```bash
uv run python src/database/01_validate_stats.py
```

Belső hívása: `collect_duckdb_stats_report(db_path, tables)` → `format_duckdb_stats_report(report)` → stdout.

**Tábla lista:** `["ohlcv", "target", "feat_ohlcv_quant", "predictions"]`

---

### `02_sync_pipeline.py`

**Célja:** Unified CLI belépési pont — OHLCV Binance szinkron és derived tábla rebuild (targets, features, predictions) egy scriptből.

```bash
# Teljes sync (OHLCV + összes derived tábla)
uv run python src/database/02_sync_pipeline.py

# Csak derived táblák (Binance fetch kihagyva)
uv run python src/database/02_sync_pipeline.py --skip-ohlcv

# OHLCV szinkron konkrét kezdőponttól
uv run python src/database/02_sync_pipeline.py --start "2024-01-01 00:00:00" --asset-id solusdt

# Csak features és predictions egy dátumtartományra
uv run python src/database/02_sync_pipeline.py \
    --tables features,predictions \
    --start "2025-01-01 00:00:00" \
    --end   "2025-06-01 00:00:00" \
    --chunk-months 1
```

| Argument | Alap | Leírás |
|----------|------|--------|
| `--asset-id` | config default | Asset azonosító (`solusdt`) |
| `--start` | legkorábbi OHLCV sor | Derived rebuild / OHLCV fetch kezdete (`YYYY-MM-DD HH:MM:SS`) |
| `--end` | legújabb OHLCV sor | Derived rebuild vége (`YYYY-MM-DD HH:MM:SS`) |
| `--chunk-months` | `3` | Hónapos chunk méret features/predictions-hoz |
| `--skip-ohlcv` | — | Binance fetch kihagyása, csak derived rebuild |
| `--tables` | `ohlcv,targets,features,predictions` | Vesszővel elválasztott tábla szűkítés |

**Függőségi sorrend** (mindig betartva): `ohlcv` → `targets` → `features` → `predictions`.

**Log fájl:** `database/<asset_id>/logs/sync_pipeline_<timestamp>.log`

---

## Konfiguráció

A modul **csak** a `src/utils.py` API-n keresztül fér hozzá konfigurációhoz:

| Függvény | Mit ad vissza |
|----------|---------------|
| `utils.load_asset_config(asset_id)` | DB elérési út, feature profil neve |
| `utils.load_features_config(asset_id)` | Indikátor konfigurációk, target config |
| `utils.load_models_config()` | Champion modellek, path-ok |
| `utils.champion_models_for_asset(model_cfg, asset_id)` | Aktív long+short modell ID-k és metaadatok |

Közvetlen JSON olvasás tiltott a `src/database/` teljes kódbázisában.




---

<!-- Source: 1100_store.md -->



# store/ — DuckDB Store Réteg

A `src/database/store/` könyvtár kezeli az összes alacsony szintű DuckDB interakciót: séma létrehozást, adatbeírást, lekérdezést, validációt és statisztikákat.

---

## Áttekintés

```{mermaid}
flowchart TD
    APP["Hívó kód\n(sync_*, 02_sync_pipeline, UI)"]
    STORE["duckdb_store.py\nírás, séma, migráció"]
    QUERY["duckdb_query.py\nolvasás, range, ASOF join"]
    STATS["duckdb_stats.py\nstat gyűjtés, audit, formázás"]
    VALID["validate.py\nintegritás ellenőrzés"]
    TOOL["toolkit.py\nDS inspekciós segédek"]
    DB[("solusdt.duckdb")]

    APP --> STORE & QUERY & TOOL
    STORE --> DB
    QUERY --> DB
    STATS --> DB
    VALID --> DB
    TOOL --> QUERY
```

---

## Fájlok

### [duckdb_store.py](1110_duckdb_store.md)

DuckDB kapcsolat kezelés, séma inicializálás és adatbeírás.

**Kulcs funkciók:**
- `get_connection(db_path)` — DuckDB kapcsolat megnyitása, parent dir létrehozása
- `ensure_tables(conn)` — Táblák létrehozása + legacy migráció (`dataset_split`, `fold_id` drop)
- `_ensure_feat_ohlcv_quant_table(conn, df)` — Dinamikus séma: tábla létrehozása vagy oszlopbővítés
- `_insert_append_only(conn, table, df)` — Core append logika `MAX(open_time)` alapon
- `insert_ohlcv(conn, df)` — OHLCV beírás (10 oszlop szűrés + append)
- `insert_feat_ohlcv_quant(conn, df)` — Feature beírás (dinamikus séma + append)
- `insert_target(conn, df)` — Target beírás (DELETE+INSERT, teljes rebuild szemantika)
- `insert_predictions(conn, df)` — Prediction beírás (séma DB-ből, append)

---

### [duckdb_query.py](1120_duckdb_query.md)

Read-only lekérdezések pandas és Polars DataFrame kimenettel.

**Kulcs funkciók:**
- `_connect(db_path)` — Read-only kapcsolat, `None` ha a fájl hiányzik
- `query_range(db_path, dataset, start, end, columns)` → pandas DataFrame
- `query_range_pl(db_path, dataset, start, end, columns)` → Polars DataFrame (zero-copy)
- `dataset_columns(db_path, dataset)` → oszlopnév lista
- `dataset_exists(db_path, dataset)` → bool (tábla létezik ÉS van benne sor)
- `asof_join_predictions_features(db_path, feature_cols, start, end)` → ASOF LEFT JOIN
- `latest_open_time(db_path, dataset)` → `pd.Timestamp | None`
- OHLCV shortcut-ok: `ohlcv_dataset_exists`, `ohlcv_row_count`, `ohlcv_latest_open_time`, `ohlcv_time_stats`

---

### [duckdb_stats.py](1130_duckdb_stats.md)

DB egészség-statisztikák gyűjtése, formázása és dataset audit.

**Dataclass-ok:** `TableStats`, `TimedMetric`, `DuckDBStatsReport`

**Kulcs funkciók:**
- `collect_duckdb_stats_report(db_path, tables)` — Sorok, időtartomány, null arányok, timing smoke (1d/1w/1mo/full), GROUP BY year
- `format_duckdb_stats_report(report)` → szöveg riport stdout-ra
- `raw_manifest_audit(db_path, dataset)` — Nyers integritás audit: sorok, tartomány, null_ts, dup_ts logolása
- `log_dataset_check(db_path, dataset)` — Sor szám + időtartomány + `raw_manifest_audit` logolása

---

### [validate.py](1140_validate.md)

Integritás invariánsok ellenőrzése AssertionError-ral.

**Kulcs funkciók:**
- `assert_zero(con, sql, msg)` — SQL futtat, AssertionError ha `count > 0`
- `check_no_future_features(db_path)` — `available_ts <= open_time` mindenhol
- `check_target_no_current_bar(db_path)` — NULL tail sorok megléte

---

### [toolkit.py](1150_toolkit.md)

DS workflow segédek — dataset inspekció és összefoglalók.

**Kulcs funkciók:**
- `resolve_db_path(asset_id)` — db_path lekérés config-ból
- `list_datasets(asset_id)` — Adatot tartalmazó dataset-ek listája
- `get_dataset_columns`, `get_row_count`, `get_time_range`, `print_summary`

---

## Írási módok összefoglalója

| Tábla | Írási mód | Függvény |
|-------|-----------|----------|
| `ohlcv` | Append-only (`MAX(open_time)` alap) | `insert_ohlcv` |
| `feat_ohlcv_quant` | Append-only + dinamikus séma | `insert_feat_ohlcv_quant` |
| `target` | DELETE+INSERT (teljes rebuild) | `insert_target` |
| `predictions` | Append-only | `insert_predictions` |




---

<!-- Source: 1110_duckdb_store.md -->



# duckdb_store.py — DuckDB Írás és Séma

`src/database/store/duckdb_store.py`

Kapcsolat kezelés, táblák inicializálása, migráció és adatbeírás. Ez az egyetlen modul, amely írási hozzáféréssel rendelkezik a DuckDB-hez.

---

## Függvény áttekintés

```{mermaid}
flowchart TD
    GC["get_connection(db_path)"]
    ET["ensure_tables(conn)"]
    EFQ["_ensure_feat_ohlcv_quant_table(conn, df)"]
    IAO["_insert_append_only(conn, table, df)"]
    IO["insert_ohlcv(conn, df)"]
    IF["insert_feat_ohlcv_quant(conn, df)"]
    IT["insert_target(conn, df)"]
    IP["insert_predictions(conn, df)"]

    GC --> ET
    GC --> IO & IF & IT & IP
    IO --> IAO
    IF --> EFQ --> IAO
    IT --> |DELETE range + INSERT| IAO
    IP --> IAO
```

---

## `get_connection(db_path)`

**Célja:** DuckDB kapcsolat megnyitása írás-olvasás módban.

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `db_path` | `str` | DuckDB fájl elérési útja |

**Visszatérési érték:** `duckdb.DuckDBPyConnection`

**Mellékhatások:** Létrehozza a szülőkönyvtárat (`Path(db_path).parent.mkdir(parents=True, exist_ok=True)`).

---

## `ensure_tables(conn)`

**Célja:** Mind a négy tábla létrehozása, ha nem léteznek. Legacy migráció: `dataset_split` és `fold_id` oszlopok eltávolítása a `predictions` táblából ha jelen vannak.

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `conn` | `duckdb.DuckDBPyConnection` | Nyitott írható kapcsolat |

**Létrehozott táblák:** `ohlcv`, `target`, `predictions` (rögzített séma). A `feat_ohlcv_quant` létrehozása dinamikus — lásd `_ensure_feat_ohlcv_quant_table`.

**Migráció:** Ha a `feat_ohlcv_quant` vagy `predictions` tábla tartalmaz `dataset_split` vagy `fold_id` oszlopot, `ALTER TABLE DROP COLUMN` törli őket.

---

## `_ensure_feat_ohlcv_quant_table(conn, df)`

**Célja:** Dinamikus séma kezelés a `feat_ohlcv_quant` táblához.

- **Ha a tábla nem létezik:** `CREATE TABLE` a DataFrame oszlopaiból (DuckDB típus inferencia)
- **Ha a tábla létezik:** hiányzó oszlopokat `ALTER TABLE ADD COLUMN`-nal adja hozzá

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `conn` | `duckdb.DuckDBPyConnection` | Nyitott írható kapcsolat |
| `df` | `pl.DataFrame` | Az újonnan beírandó feature DataFrame |

**Fontos:** A config-driven feature szám változhat modellek között. Ez a függvény teszi lehetővé az online sémabővítést anélkül, hogy a teljes táblát újra kellene építeni.

---

## `_insert_append_only(conn, table, df)`

**Célja:** Core append logika — csak az eddig nem tárolt sorok beírása.

**Működés:**
1. `conn.register("_ins_batch", df)` — DataFrame DuckDB view-ként regisztrálva
2. `MAX(open_time)` lekérdezése az adott táblából (`last_ts`)
3. `INSERT INTO table SELECT * FROM _ins_batch WHERE open_time > last_ts`

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `conn` | `duckdb.DuckDBPyConnection` | Nyitott írható kapcsolat |
| `table` | `str` | Céltábla neve |
| `df` | `pl.DataFrame` | Beírandó adatok |

**Invariáns:** Ha a tábla üres (`last_ts = None`), az összes sor bekerül. Ha minden sor már korábban be volt írva, 0 sor kerül beírásra (idempotens).

---

## `insert_ohlcv(conn, df)`

**Célja:** OHLCV adatok beírása append-only módban.

**Előfeldolgozás:** A DataFrame-ből csak a 10 OHLCV oszlopot tartja meg (szűri a Binance `close_time` és `ignore` mezőket, ha véletlenül jelen vannak).

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `conn` | `duckdb.DuckDBPyConnection` | Nyitott írható kapcsolat |
| `df` | `pl.DataFrame` | OHLCV sorok (`open_time`, `open`, `high`, `low`, `close`, `volume`, `quote_volume`, `trades`, `taker_buy_base`, `taker_buy_quote`) |

---

## `insert_feat_ohlcv_quant(conn, df)`

**Célja:** Feature adatok beírása dinamikus sémával.

**Lépések:** `_ensure_feat_ohlcv_quant_table(conn, df)` → `_insert_append_only(conn, "feat_ohlcv_quant", df)`

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `conn` | `duckdb.DuckDBPyConnection` | Nyitott írható kapcsolat |
| `df` | `pl.DataFrame` | Feature sorok (`open_time`, `close`, `available_ts`, `lookback_end_ts`, `feat_*`) |

---

## `insert_target(conn, df)`

**Célja:** Target labelek beírása. **Teljes rebuild szemantika** — nem append-only.

**Lépések:**
1. `DELETE FROM target WHERE open_time BETWEEN df.min AND df.max`
2. `INSERT INTO target SELECT * FROM _ins_batch`

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `conn` | `duckdb.DuckDBPyConnection` | Nyitott írható kapcsolat |
| `df` | `pl.DataFrame` (Polars) | Target sorok (`open_time`, `long_mfe_fw60`, `short_mfe_fw60` és auxiliary fw60 oszlopok) |

**Fontos:** A target kvantilis küszöbök a teljes history alapján újraszámítódnak minden `sync_targets` híváskor — ezért szükséges a teljes tartomány DELETE+INSERT.

---

## `insert_predictions(conn, df)`

**Célja:** Predikciók beírása append-only módban. A séma a DB-ből olvasódik (nem a DataFrame-ből inferálva).

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `conn` | `duckdb.DuckDBPyConnection` | Nyitott írható kapcsolat |
| `df` | `pl.DataFrame` | Predikció sorok (`open_time`, `close`, `label_end_ts`, `trg_*`, `long_pred`, `short_pred`) |




---

<!-- Source: 1120_duckdb_query.md -->



# duckdb_query.py — Lekérdezések

`src/database/store/duckdb_query.py`

Read-only lekérdezési réteg. Minden függvény önálló db_path-on dolgozik — nem tart fenn nyitott kapcsolatot. Pandas és Polars kimenet egyaránt elérhető.

---

## `_connect(db_path)`

**Célja:** Read-only DuckDB kapcsolat megnyitása.

**Visszatérési érték:** `duckdb.DuckDBPyConnection | None` — `None` ha a fájl nem létezik.

Minden publikus lekérdező függvény ezt hívja belül, és gracefully kezeli a hiányzó DB esetet (üres DataFrame / None visszatérés).

---

## `query_range(db_path, dataset, start, end, columns)`

**Célja:** Időtartomány-alapú lekérdezés pandas DataFrame-ként.

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `db_path` | `str` | DuckDB fájl elérési útja |
| `dataset` | `str` | Tábla neve (`ohlcv`, `feat_ohlcv_quant`, stb.) |
| `start` | `str \| None` | Kezdő timestamp (`YYYY-MM-DD HH:MM:SS`), `None` = teljes history |
| `end` | `str \| None` | Záró timestamp, `None` = legújabbig |
| `columns` | `list[str] \| None` | Lekérdezett oszlopok, `None` = összes |

**Visszatérési érték:** `pd.DataFrame` — üres DataFrame ha DB hiányzik vagy nincs találat.

**SQL:** `SELECT {cols} FROM {dataset} WHERE open_time BETWEEN ? AND ? ORDER BY open_time`

---

## `query_range_pl(db_path, dataset, start, end, columns)`

**Célja:** Ugyanaz mint `query_range`, de **Polars DataFrame** kimenettel (zero-copy DuckDB → Arrow → Polars).

**Visszatérési érték:** `pl.DataFrame`

A feature computation (`sync_features.py`) ezt használja a OHLCV adatok betöltéséhez Polars pipeline-ba.

---

## `dataset_columns(db_path, dataset)`

**Célja:** Tábla oszlopneveinek lekérdezése.

**Visszatérési érték:** `list[str]` — üres lista ha DB vagy tábla hiányzik.

**Felhasználás:** `insert_predictions` hívja, hogy a séma a DB-ből olvasódjon (nem DataFrame-ből inferálva).

---

## `dataset_exists(db_path, dataset)`

**Célja:** Ellenőrzi, hogy a tábla létezik-e ÉS van-e benne legalább 1 sor.

**Visszatérési érték:** `bool`

**Felhasználás:** Sync függvények ellenőrzik a szülő tábla meglétét (pl. sync_features ellenőrzi, hogy az ohlcv tábla létezik) mielőtt futnak.

---

## `row_count(db_path, dataset)`

**Célja:** Sorok száma egy táblában.

**Visszatérési érték:** `int` — `0` ha DB vagy tábla hiányzik.

---

## `asof_join_predictions_features(db_path, feature_cols, start, end)`

**Célja:** ASOF JOIN a `predictions` és `feat_ohlcv_quant` táblák között, modeling-ready snapshothoz.

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `db_path` | `str` | DuckDB fájl elérési útja |
| `feature_cols` | `list[str]` | Feature oszlopok listája (`feat_*`) |
| `start` | `str \| None` | Időtartomány kezdete |
| `end` | `str \| None` | Időtartomány vége |

**Visszatérési érték:** `pd.DataFrame`

**SQL logika:**
```sql
SELECT p.open_time AS prediction_ts,
       f.available_ts AS lookback_end_ts,
       f.feat_col1, f.feat_col2, ...
FROM predictions p
ASOF LEFT JOIN feat_ohlcv_quant f
    ON p.open_time >= f.available_ts
WHERE p.open_time BETWEEN ? AND ?
ORDER BY prediction_ts
```

Az `available_ts` biztosítja, hogy minden predikció sorhoz csak már elérhető feature-ök csatlakoznak.

---

## `latest_open_time(db_path, dataset)`

**Célja:** A tárolt adatok legújabb timestampje.

**Visszatérési érték:** `pd.Timestamp | None` — `None` ha a tábla üres vagy hiányzik.

**Felhasználás:** Minden sync függvény innen határozza meg, hol folytassa az inkrementális szinkront.

---

## OHLCV shortcut függvények

Kényelmi wrapperek az `ohlcv` táblához:

| Függvény | Visszatérési érték |
|----------|-------------------|
| `ohlcv_dataset_exists(db_path)` | `bool` |
| `ohlcv_row_count(db_path)` | `int` |
| `ohlcv_latest_open_time(db_path)` | `str \| None` — `YYYY-MM-DD HH:MM:SS` formátum |
| `ohlcv_time_stats(db_path)` | `tuple[int, str \| None, str \| None]` — `(count, min_ts, max_ts)` |

---

## Kapcsolat kezelés

Minden lekérdező függvény belül:
1. `_connect(db_path)` → kapcsolat vagy `None`
2. Ha `None`: üres/default visszatérési értékkel tér vissza (nem dob hibát)
3. Minden esetben `conn.close()` a `finally` blokkban

A read-only mód (`read_only=True`) megakadályozza, hogy a lekérdező kód véletlenül módosítsa az adatbázist. Párhuzamos olvasást támogat több process-ből.




---

<!-- Source: 1130_duckdb_stats.md -->



# duckdb_stats.py — DB Statisztikák és Audit

`src/database/store/duckdb_stats.py`

DB egészség-ellenőrzés: sorok, időtartományok, null arányok és lekérdezési teljesítmény minden táblára. Dataset integritás audit és logolás. A `01_validate_stats.py` CLI script és a `02_sync_pipeline.py` hívja.

---

## Dataclass-ok

### `TableStats`

Egy tábla pillanatképe:

| Mező | Típus | Leírás |
|------|-------|--------|
| `table` | `str` | Tábla neve |
| `status` | `str` | `"OK"`, `"EMPTY"`, `"SKIP_TABLE_MISSING"`, `"SKIP_DB_MISSING"` |
| `row_count` | `int` | Sorok száma |
| `min_open_time` | `str \| None` | Legkorábbi `open_time` |
| `max_open_time` | `str \| None` | Legújabb `open_time` |
| `column_count` | `int` | Oszlopok száma |
| `null_ratios` | `dict[str, float]` | Null arány max 5 nem-`open_time` oszlopra (0.0–1.0) |

---

### `TimedMetric`

Egy benchmark lekérdezés eredménye:

| Mező | Típus | Leírás |
|------|-------|--------|
| `label` | `str` | Leírás (pl. `"range_ohlcv_1d"`) |
| `status` | `str` | `"OK"` vagy `"SKIP_EMPTY"` |
| `elapsed_ms` | `float` | Futási idő milliszekundumban |
| `row_count` | `int \| None` | Visszaadott sorok száma |
| `detail` | `str` | Részlet (pl. `"rows=1440"`) |

---

### `DuckDBStatsReport`

Teljes riport:

| Mező | Típus | Leírás |
|------|-------|--------|
| `db_path` | `str` | DuckDB fájl elérési útja |
| `tables` | `list[TableStats]` | Táblák stat listája |
| `timings` | `list[TimedMetric]` | Benchmark eredmények |

---

## `collect_duckdb_stats_report(db_path, tables)`

**Célja:** Teljes `DuckDBStatsReport` összeállítása a megadott táblákra.

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `db_path` | `str` | DuckDB fájl elérési útja |
| `tables` | `list[str] \| None` | Vizsgálandó táblák. Alap: `["ohlcv", "target", "feat_ohlcv_quant", "predictions"]` |

**Visszatérési érték:** `DuckDBStatsReport`

**Gyűjtött adatok táblánként:**
- `COUNT(*)` → `row_count`
- `MIN(open_time)` / `MAX(open_time)` → `min_open_time` / `max_open_time`
- Null arány max 5 (nem `open_time`) oszlopra: `COUNT(*) WHERE col IS NULL / COUNT(*)`
- Oszlopszám az `information_schema.columns`-ból

**Timing smoke benchmarkok** (ha `ohlcv` tábla elérhető):
- `range_ohlcv_1d` — utolsó 1 nap COUNT
- `range_ohlcv_1w` — utolsó 1 hét COUNT
- `range_ohlcv_1mo` — utolsó 1 hónap COUNT
- `range_ohlcv_full` — teljes COUNT(*)
- `groupby_ohlcv_year` — éves bontás GROUP BY

Ha a DB fájl vagy a tábla hiányzik, a riport `SKIP_DB_MISSING` / `SKIP_TABLE_MISSING` / `EMPTY` státuszokat ad vissza — nem dob kivételt.

---

## `raw_manifest_audit(db_path, dataset)`

**Célja:** Nyers dataset integritás audit — sorok, időtartomány, null timestamp-ek, duplikált timestamp-ek logolása.

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `db_path` | `str` | DuckDB fájl elérési útja |
| `dataset` | `str` | Vizsgálandó tábla neve (`'ohlcv'`, `'target'`, `'feat_ohlcv_quant'`, `'predictions'`) |

**Visszatérési érték:** `None` — kizárólag `logging.info` / `logging.warning` kimenet.

**Ellenőrzött metrikák (DuckDB native SQL):**

| Kulcs | Leírás |
|-------|--------|
| `row_count` | Összes sor száma (`COUNT(*)`) |
| `min_ts` | Legkorábbi `open_time` |
| `max_ts` | Legújabb `open_time` |
| `null_ts` | Null `open_time` értékű sorok (`SUM(CASE WHEN open_time IS NULL)`) |
| `dup_ts` | Duplikált `open_time` értékek (`COUNT(*) - COUNT(DISTINCT ...)`) |

**Viselkedés:**
- Ha a DB fájl hiányzik: `logger.warning` és korai visszatérés
- Ha a tábla nem létezik: `logger.warning` és korai visszatérés
- Ha `null_ts > 0` vagy `dup_ts > 0`: `logger.warning` (gyanús adat)
- Egyébként: `logger.info` OK üzenet

**Felhasználás:** Deployment utáni ellenőrzés, adatintegritás gyanú esetén. A `log_dataset_check` hívja minden dataset után.

---

## `log_dataset_check(db_path, dataset)`

**Célja:** Sor szám, időtartomány és `raw_manifest_audit` logolása egy dataset-re.

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `db_path` | `str` | DuckDB fájl elérési útja |
| `dataset` | `str` | Dataset neve: `'ohlcv'`, `'target'`, `'feat_ohlcv_quant'`, `'predictions'` |

**Visszatérési érték:** `None` — kizárólag `logging.info` / `logging.warning` kimenet.

**Belső logika:**
- `ohlcv` esetén: `ohlcv_dataset_exists` + `ohlcv_time_stats` (gyors path)
- Egyéb táblák: `dataset_exists` + `query_range_pl(columns=["open_time"])` → min/max számítás
- Mindkét ágban: `raw_manifest_audit(db_path, dataset)` meghívva a részletes audithoz

---

## `format_duckdb_stats_report(report)`

**Célja:** `DuckDBStatsReport` ember által olvasható szöveggé alakítása.

**Visszatérési érték:** `str` — többsoros riport stdout-ra vagy logba.

**Példa kimenet:**
```
DuckDB statistics smoke report
db_path: database/solusdt/solusdt.duckdb
informational: timing metrics do not fail validation by themselves

Tables:
- ohlcv: status=OK rows=1234567 min=2022-01-01 00:00:00 max=2026-06-14 23:59:00 cols=10
  null_ratios: open=0.000, high=0.000, low=0.000, close=0.000, volume=0.000
- target: status=OK rows=1234567 ...

Timings:
- range_ohlcv_1d: status=OK elapsed_ms=12.345 row_count=1440 rows=1440
- range_ohlcv_1w: status=OK elapsed_ms=45.678 row_count=10080 rows=10080
- groupby_ohlcv_year: status=OK elapsed_ms=23.100 row_count=5 rows=5
```




---

<!-- Source: 1140_validate.md -->



# validate.py — Integritás Ellenőrzés

`src/database/store/validate.py`

Adatintegritás invariánsok ellenőrzése. Minden függvény `AssertionError`-t dob ha az ellenőrzés sikertelen. A `check_*` függvények a `01_validate_stats.py` pipeline részeként futnak és CI-ban is futtathatók.

---

## `assert_zero(con, sql, msg)`

**Célja:** Alap ellenőrző primitív — SQL-t futtat, `AssertionError`-t dob ha a visszaadott szám nem nulla.

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `con` | `duckdb.DuckDBPyConnection` | Nyitott kapcsolat |
| `sql` | `str` | SQL lekérdezés, egyetlen COUNT(*) értéket ad vissza |
| `msg` | `str` | Hiba üzenet `AssertionError`-ban |

**Viselkedés:**
- Ha az SQL 0-t ad vissza: átmegy (sikeres)
- Ha az SQL > 0-t ad vissza: `AssertionError(f"{msg}: {count} sor sért feltételt")`

**Felhasználás:** A `check_*` függvények mind erre épülnek.

---

## `check_no_future_features(db_path)`

**Célja:** Ellenőrzi, hogy a `feat_ohlcv_quant` táblában `available_ts <= open_time` minden sorra teljesül.

**Invariáns:** Ha `available_ts > open_time`, az azt jelenti, hogy egy feature értéke egy jövőbeli bartól függ — ez lookahead bias.

**SQL belül:**
```sql
SELECT COUNT(*) FROM feat_ohlcv_quant
WHERE available_ts > open_time
```

**Hiba esetén:** `AssertionError` — a pipeline megáll és a hibát javítani kell.

---

## `check_target_no_current_bar(db_path)`

**Célja:** Ellenőrzi, hogy a `target` tábla utolsó `horizon` (=60) sorában a target oszlopok `NULL`-ok.

**Invariáns:** Az utolsó 60 bar jövőbeli záróára még nem ismert — a labelek `NULL`-ok kell legyenek.

**Hiba esetén:** `AssertionError` — az utolsó sorok NULL kell legyenek.

---

## `check_sample_table(db_path, sample_id, expected_feat_cols=None)`

**Célja:** Integritás ellenőrzések a materializált `sample_<sample_id>` DuckDB táblán, mielőtt a modellező pipeline felhasználja.

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `db_path` | `str` | Abszolút path a .duckdb fájlhoz |
| `sample_id` | `str` | Sample azonosító (`sample_<sample_id>` a tábla neve) |
| `expected_feat_cols` | `list[str] \| None` | Opcionális: elvárt feat_* oszlopok listája (pl. `feature_set.json`-ból) |

**Ellenőrzések:**

| # | Invariáns | Hiba |
|---|-----------|------|
| 1 | Nincs duplikált `(open_time, fold_id, segment)` sor | `AssertionError: duplicate (open_time, fold_id, segment) rows` |
| 2 | A `test` sorok minden `non-test` sort időben megelőznek | `AssertionError: non-test rows not strictly before test rows` |
| 3 | Legalább egy `purge` sor létezik (embargo buffer jelen van) | `AssertionError: no purge rows found` |
| 4 | A `train` és `valid` soroknál `long_mfe_fw60` és `short_mfe_fw60` nem NULL | `AssertionError: NULL target columns in train/valid rows` |
| 5 | Ha `expected_feat_cols` megadva: minden oszlop jelen van a táblában | `AssertionError: expected feature columns missing: [...]` |

**Raises:**
- `FileNotFoundError` — ha a .duckdb fájl nem létezik, vagy a tábla hiányzik
- `AssertionError` — ha bármely invariáns sérül

**Felhasználás:**
```python
from database.store.validate import check_sample_table

# Alap ellenőrzés
check_sample_table("database/solusdt/solusdt.duckdb", "solusdt_fw60_yearly_2024")

# Feature set konzisztencia-ellenőrzéssel
check_sample_table(
    "database/solusdt/solusdt.duckdb",
    "solusdt_fw60_yearly_2024",
    expected_feat_cols=["feat_rsi_14", "feat_roc_14", ...],
)
```

---

## Futtatás

```bash
# Standalone (01_validate_stats.py részeként)
uv run python src/database/01_validate_stats.py

# Direkt hívás Python-ból
from database.store.validate import (
    check_no_future_features,
    check_quant_train_no_duplicates,
    check_sample_table,
    check_target_no_current_bar,
)
check_no_future_features("database/solusdt/solusdt.duckdb")
check_quant_train_no_duplicates("database/solusdt/solusdt.duckdb")
check_sample_table("database/solusdt/solusdt.duckdb", "solusdt_fw60_yearly_2024")
```




---

<!-- Source: 1150_toolkit.md -->



# toolkit.py — Dataset Inspekciós Segédek

`src/database/store/toolkit.py`

DS workflow segédek gyors dataset inspekciókhoz. Nem üzleti logika — kényelmi wrapperek notebook és REPL használatra.

---

## `resolve_db_path(asset_id)`

**Célja:** Asset ID-ból DuckDB fájl elérési út lekérése a configból.

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `asset_id` | `str \| None` | Asset azonosító (`solusdt`), `None` = config default |

**Visszatérési érték:** `str` — abszolút db_path

**Belső hívás:** `utils.load_asset_config(asset_id)["database"]["db_path"]`

---

## `list_datasets(asset_id)`

**Célja:** Az asset DB-ben adatot tartalmazó dataset-ek listázása.

**Visszatérési érték:** `list[str]` — pl. `["ohlcv", "feat_ohlcv_quant", "predictions", "target"]`

**Logika:** `dataset_exists(db_path, dataset)` hívás minden ismert táblára.

---

## `get_dataset_columns(asset_id, dataset)`

**Célja:** Dataset oszlopnevei.

**Visszatérési érték:** `list[str]`

---

## `get_row_count(asset_id, dataset)`

**Célja:** Sorok száma egy datasetben.

**Visszatérési érték:** `int`

---

## `get_time_range(asset_id, dataset)`

**Célja:** Az adott dataset időtartományának lekérdezése.

**Visszatérési érték:** `tuple[pd.Timestamp | None, pd.Timestamp | None]` — `(min_ts, max_ts)`

---

## `print_summary(asset_id)`

**Célja:** Gyors összefoglaló az összes dataset állapotáról stdout-ra.

**Példa kimenet:**
```
=== solusdt DB Summary ===
ohlcv:
  Rows: 1,234,567
  Range: 2022-01-01 00:00:00 → 2026-06-14 23:59:00

feat_ohlcv_quant:
  Rows: 1,230,000
  Range: 2022-01-15 00:01:00 → 2026-06-14 23:59:00

target:
  Rows: 1,234,507
  Range: 2022-01-01 00:00:00 → 2026-06-14 22:59:00

predictions:
  Rows: 1,180,000
  Range: 2023-06-01 00:00:00 → 2026-06-14 23:59:00
```

**Felhasználás:**
```python
from database.store.toolkit import print_summary
print_summary("solusdt")
```




---

<!-- Source: 1200_sync_tables.md -->



# sync_tables/ — Sync Pipeline

A `src/database/sync_tables/` könyvtár felelős az összes adat mozgatásáért és transzformálásáért: Binance API → OHLCV → features → targets → predictions. Minden funkció idempotens — biztonságosan újrafuttatható.

---

## Pipeline áttekintés

```{mermaid}
sequenceDiagram
    participant BIN as Binance API
    participant OHLCV as ohlcv tábla
    participant FEAT as feat_ohlcv_quant
    participant TARGET as target tábla
    participant PRED as predictions

    Note over BIN,OHLCV: sync_ohlcv.py
    BIN->>OHLCV: 1000 klines/batch (append-only)

    Note over OHLCV,FEAT: sync_features.py
    OHLCV->>FEAT: query → compute_features_polars → insert

    Note over OHLCV,TARGET: sync_targets.py
    OHLCV->>TARGET: DuckDB window SQL → DELETE+INSERT

    Note over FEAT,PRED: sync_predictions.py
    FEAT->>PRED: ASOF join → model.predict / predict_proba → insert
    TARGET-.->PRED: label join (ha elérhető)
```

---

## Fájlok

### [sync_ohlcv.py](0221_sync_ohlcv.md)

Binance 1-perces kline szinkron. Paginálás 1000/batch, stale guard, gap check.

**Belépési pont:** `sync_ohlcv(open_time_ms_from, asset_id)`

**Kritikus invariáns:** 10 oszlop tárolva (Binance 12-ből — `close_time` és `ignore` elhagyva).

---

### [sync_features.py](0222_sync_features.md)

Feature számítás Polars LazyFrame pipeline-nal. t-1 lag minden OHLCV-alapú feature-re.

**Belépési pont:** `sync_features(start_time, lookback_bars, end_time, asset_id)`

**Kritikus invariáns:** `available_ts = open_time` (t-1 lag az `_apply_t1_lag_pl` biztosítja).

---

### [sync_predictions.py](0223_sync_predictions.md)

Champion modellek betöltése és inference futtatása. Unified long+short output egy sorban.

**Belépési pont:** `sync_predictions(start_time, end_time, asset_id)`

**Kulcs lépések:** `champion_models_for_asset` → `_load_model_artifacts` → ASOF join features → `_run_inference` → `insert_predictions`

---

### [sync_targets.py](0224_sync_targets.md)

Bináris target labelek számítása DuckDB window SQL-lel. Teljes rebuild szemantika.

**Belépési pont:** `sync_targets(asset_id)`

**Kritikus invariáns:** `ROWS BETWEEN 1 FOLLOWING AND 60 FOLLOWING` — az aktuális bar NEM szerepel a forward window-ban.

---

### [_features_polars.py](0225_features_polars.md)

Feature computation engine — 30+ indikátor csoport Polars LazyFrame API-val.

**Belépési pont:** `compute_features_polars(df_ohlcv, indicators, feat_prefix, available_activity, targets_cfg)`

**Kritikus invariáns:** `T_MINUS_1_SKIP` frozenset — P2 időindexes feature-ök ki vannak zárva a t-1 lag alól.

---

## Függőségi sorrend

```{mermaid}
flowchart LR
    OHLCV --> FEAT
    OHLCV --> TARGET
    FEAT --> PRED
    TARGET -.-> PRED
```

A `02_sync_pipeline.py` ezt a sorrendet garantálja:
1. `sync_targets` (csak `ohlcv`-t olvas)
2. `sync_features` (csak `ohlcv`-t olvas)
3. `sync_predictions` (`feat_ohlcv_quant`-ot ASOF join-nal olvas)

A `target` tábla NEM blokkolja a features rebuild-et — a predictions viszont a features meglétét igényli.

---

## Idempotencia

Minden sync függvény a `latest_open_time(db_path, dataset)` alapján meghatározza a szinkron kezdőpontját. Ha a kért tartomány már be van töltve, a függvény 0 beírással lép ki.

| Függvény | Gap kezelés |
|----------|-------------|
| `sync_ohlcv` | Stale guard + gap check, warning logolás |
| `sync_features` | `MAX(open_time)` a feat táblában |
| `sync_predictions` | `MAX(open_time)` a predictions táblában |
| `sync_targets` | Mindig teljes rebuild (DELETE+INSERT) |

---

## Lookahead bias megakadályozás

```{mermaid}
flowchart TD
    OT["open_time\n(bar t)"]
    AV["available_ts\n(= open_time, t-1 lag után)"]
    PRED_OT["prediction open_time\n(bar t)"]
    JOIN["ASOF: pred.open_time >= feat.available_ts"]

    OT --> AV
    AV --> JOIN
    PRED_OT --> JOIN
```

A feature-öket a `compute_features_polars` hívásban `shift(1)` tolja el. Az `available_ts` mezőt a `sync_features` állítja be `open_time`-ra az eltolás után. Az ASOF join a `predictions` táblát `feat_ohlcv_quant.available_ts <= predictions.open_time` feltétellel köti össze — így minden bar csak már elérhető feature-t lát.




---

<!-- Source: 1210_sync_ohlcv.md -->



# sync_ohlcv.py — Binance OHLCV Szinkron

`src/database/sync_tables/sync_ohlcv.py`

Inkrementális Binance 1-perces kline szinkronizálás. Minden futás az utolsó tárolt bar utántól kezdi, 1000 klines/batch lapozással.

---

## `sync_ohlcv(open_time_ms_from, asset_id)`

**Célja:** Binance klines lekérése a megadott időponttól és `insert_ohlcv` beírás.

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `open_time_ms_from` | `int` | Szinkron kezdete epoch milliszekundumban |
| `asset_id` | `str \| None` | Asset azonosító (config default ha `None`) |

---

## Belső folyamat

```{mermaid}
sequenceDiagram
    participant CALLER as 02_sync_pipeline.py
    participant SYNC as sync_ohlcv()
    participant DB as DuckDB (ohlcv)
    participant BIN as Binance API

    CALLER->>SYNC: open_time_ms_from, asset_id
    SYNC->>DB: latest_open_time(db_path, "ohlcv")
    DB-->>SYNC: last_ts (vagy None)
    SYNC->>SYNC: stale guard (last_ts < open_time_ms_from?)

    loop 1000 klines/batch
        SYNC->>BIN: GET /klines?symbol=SOLUSDT&interval=1m&startTime=...&limit=1000
        BIN-->>SYNC: list[list] (12 elemű sorok)
        SYNC->>SYNC: parse → DataFrame (10 oszlop)
        SYNC->>DB: insert_ohlcv(conn, df)
        SYNC->>SYNC: utolsó open_time → következő batch start
        alt kevesebb mint 1000 sor
            SYNC->>SYNC: break (nincs több adat)
        end
    end

    SYNC->>SYNC: gap check (folytonos 1-perces cadence?)
```

---

## Stale Guard

Ha az utolsó tárolt timestamp az adott `open_time_ms_from` kértnél régebbi lenne, a függvény figyelmeztet és a **DB-ben tárolt legutóbb érték utántól indul** (nem a kért indulóponttól). Ez megakadályozza a hiányos adatsort.

---

## Gap Check

Az összes batch betöltése után az utolsó N bar `open_time` értékeit ellenőrzi — egymást követő 1-perces időközök-e. Ha gap van (hiányzó bar), `logging.warning` hívással jelzi. Nem állítja meg a futást.

---

## Binance kline formátum

Binance 12-elemű raw list → 10 oszlop:

| Index | Binance mező | Tárolt oszlop |
|-------|-------------|---------------|
| 0 | open_time (ms) | `open_time` (TIMESTAMP) |
| 1 | open | `open` |
| 2 | high | `high` |
| 3 | low | `low` |
| 4 | close | `close` |
| 5 | volume | `volume` |
| 6 | close_time (ms) | **elhagyva** (redundáns) |
| 7 | quote_volume | `quote_volume` |
| 8 | trades | `trades` |
| 9 | taker_buy_base | `taker_buy_base` |
| 10 | taker_buy_quote | `taker_buy_quote` |
| 11 | ignore | **elhagyva** (deprecated) |

Az `open_time` ms → TIMESTAMP konverzió: `pd.to_datetime(open_time_ms, unit="ms", utc=True).tz_localize(None)`.

---

## Futtatás

Az `sync_ohlcv` a `02_sync_pipeline.py` unified CLI-n keresztül hívható:

```bash
# OHLCV szinkron egy konkrét kezdőponttól
uv run python src/database/02_sync_pipeline.py --start "2024-01-01 00:00:00" --tables ohlcv --asset-id solusdt

# Az utolsó tárolt sortól indul (alapértelmezett, OHLCV + derived táblák)
uv run python src/database/02_sync_pipeline.py

# Csak OHLCV (derived rebuild nélkül)
uv run python src/database/02_sync_pipeline.py --tables ohlcv
```




---

<!-- Source: 1230_sync_predictions.md -->



# sync_predictions.py — Inference és Predikció Beírás

`src/database/sync_tables/sync_predictions.py`

Champion modellek betöltése, feature snapshot összeállítása ASOF join-nal, inference futtatása, unified long+short predikciók beírása.

---

## `sync_predictions(start_time, end_time, asset_id)`

**Célja:** Predikciók generálása a megadott időtartományra és beírás a `predictions` táblába.

**Paraméterek:**

| Paraméter | Típus | Alap | Leírás |
|-----------|-------|------|--------|
| `start_time` | `str` | — | Inference kezdete (`YYYY-MM-DD HH:MM:SS`) |
| `end_time` | `str \| None` | `None` | Inference vége (`None` = legújabb feature bar) |
| `asset_id` | `str \| None` | `None` | Asset azonosító |

---

## Belső folyamat

```{mermaid}
sequenceDiagram
    participant CALLER as sync_predictions()
    participant UTILS as utils (config)
    participant FEAT as feat_ohlcv_quant
    participant TARGET as target tábla
    participant MODEL as model.pkl
    participant PRED as predictions tábla

    CALLER->>UTILS: load_models_config()
    UTILS-->>CALLER: model konfiguráció
    CALLER->>UTILS: champion_models_for_asset(model_cfg, asset_id)
    UTILS-->>CALLER: long_model_id, long_meta, short_model_id, short_meta

    CALLER->>CALLER: _load_model_artifacts(long_model_id, long_meta)
    CALLER->>CALLER: _load_model_artifacts(short_model_id, short_meta)

    CALLER->>FEAT: query_range_pl(feat_ohlcv_quant, start, end)
    FEAT-->>CALLER: pl.DataFrame (sorok × feature_cols)

    CALLER->>TARGET: query_range_pl(target, start, end)
    TARGET-->>CALLER: pl.DataFrame fw60 outcome értékek (opcionális)
    CALLER->>CALLER: feat_df.join(target_df, on=open_time, how=left)

    CALLER->>MODEL: _run_inference(df, feature_list, long_model, long_meta)
    MODEL-->>CALLER: long_pred (1D array — predict vagy predict_proba[:, 1])
    CALLER->>MODEL: _run_inference(df, feature_list, short_model, short_meta)
    MODEL-->>CALLER: short_pred (1D array — predict vagy predict_proba[:, 1])

    CALLER->>PRED: insert_predictions(conn, unified_df)
```

---

## `_load_model_artifacts(model_id, model_meta)`

**Célja:** Model pickle és features.json betöltése lemezről.

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `model_id` | `str` | Model azonosító (`<model_id>`) |
| `model_meta` | `dict` | Model konfiguráció (paths, trainer, stb.) |

**Visszatérési érték:** `tuple[Any, list[str]]` — `(model_object, feature_list)`

**Paths:**
- `models/<model_id>/model.pkl` → `pickle.load`
- `models/<model_id>/features.json` → JSON lista (feature nevekkel)

---

## `_run_inference(df, feature_list, model, model_meta)`

**Célja:** Egyetlen modell inference futtatása.

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `df` | `pl.DataFrame` | Feature sorok (Polars) |
| `feature_list` | `list[str]` | Feature oszlopok sorrendben (model elvárja) |
| `model` | `Any` | Betöltött model objektum |
| `model_meta` | `dict` | `predict.method` = `"predict_proba"` vagy `"predict"` |

**Visszatérési érték:** `np.ndarray` — 1D vektor, soronként egy predikciós érték.

**Numpy konverzió:** `df.select(feature_list).fill_nan(0.0).fill_null(0.0).to_numpy()` — LightGBM numpy array-t kap, pandas-mentes.

**Predict módok:**
- `predict_proba` → `model.predict_proba(X)[:, 1]` (positive class valószínűsége)
- `predict` → `model.predict(X)` (direkt output)

---

## `_feature_list_for_prediction(features_data, model, trainer)`

**Célja:** A feature lista végleges sorrendjének meghatározása.

**Prioritás:**
1. `model.feature_names_in_` (sklearn standard) — ha elérhető
2. `features.json` tartalom — fallback
3. `features_data` oszlopok — last resort

Ez biztosítja, hogy a feature sorrend pontosan egyezzen a training kori sorrénddel.

---

## Output szerkezet

Az `insert_predictions` a következő egyesített DataFrame-et kapja:

| Oszlop | Forrás |
|--------|--------|
| `open_time` | `feat_ohlcv_quant.open_time` |
| `close` | `feat_ohlcv_quant.close` |
| `label_end_ts` | `open_time + fw_minutes` (config) |
| `long_pred` | `_run_inference(long_model)` |
| `short_pred` | `_run_inference(short_model)` |




---

<!-- Source: 1300_tests.md -->



# tests/ — Teszt Áttekintés

`src/database/tests/`

A database modul tesztjei négy szinten ellenőrzik az adatintegritást, store működést és pipeline helyességét.

---

## Teszt struktúra

```
src/database/tests/
├── store/
│   ├── conftest.py          db_path és conn fixture-ök
│   ├── smoke/               Gyors szintaktikai ellenőrzések
│   ├── sanity/              Adatintegritás invariánsok az éles DB-n
│   └── perf/                Teljesítmény benchmarkok
├── sync_tables/
│   ├── smoke/               Sync funkciók mock adattal
│   ├── sanity/              Lookahead bias és adatminőség
│   └── integration/         Cross-layer pipeline flow
└── sync_pipeline/
    └── smoke/               02_sync_pipeline.py CLI helper tesztek
```

Részletes tesztek:
- Store tesztek → [1310_store_tests.md](1310_store_tests.md)
- Pipeline tesztek → [1320_pipeline_tests.md](1320_pipeline_tests.md)

---

## pytest Markok

| Mark | Leírás | Futtatás |
|------|--------|----------|
| `smoke` | Gyors szintaktikai + funkcionális ellenőrzések | Minden commit |
| `sanity` | Adat-invariáns ellenőrzések az éles DB-n | Napi / sync után |
| `perf` | Wall-clock teljesítmény benchmarkok | Ad hoc / regresszió gyanú |
| `integration` | Cross-layer pipeline flow | Release / major refactor |

---

## Fixtures (`store/conftest.py`)

### `db_path` fixture

Betölti az éles DB elérési útját a configból (`utils.load_asset_config("solusdt")["database"]["db_path"]`).

Ha a DB fájl nem létezik, a tesztek `pytest.skip()`-pel átlépnek — nem hibáznak.

### `conn` fixture

Read-only DuckDB kapcsolat az éles DB-re. Az összes `sanity` és `perf` teszt ezt kapja dependency injectionként.

---

## Futtatás

```bash
# Összes database teszt
uv run pytest src/database/tests/ -v

# Csak smoke tesztek (gyors, CI-ban)
uv run pytest src/database/tests/ -m smoke -v

# Csak sanity (éles DB kell)
uv run pytest src/database/tests/ -m sanity -v

# Perf benchmarkok
uv run pytest src/database/tests/ -m perf -v -s

# Integration tesztek (synthetic data, mock models)
uv run pytest src/database/tests/ -m integration -v
```

A `-s` flag a `print()` kimeneteket is megjeleníti — perf teszteknél fontos a timing értékek láthatóságához.




---

<!-- Source: 1310_store_tests.md -->



# tests/store/ — Store Tesztek

`src/database/tests/store/`

Store réteg tesztjei három szinten: smoke (szintaxis + alapfunkció), sanity (éles DB adatintegritás), perf (wall-clock teljesítmény).

---

## smoke/ — Gyors Funkcionális Ellenőrzések

Szintetikus adattal futnak, nem igényelnek éles DB-t.

### `test_duckdb_store_query.py`

| Teszt | Mit ellenőriz |
|-------|---------------|
| `test_ensure_tables_creates_tables` | `ensure_tables()` létrehozza `ohlcv`, `target`, `predictions` táblákat |
| `test_insert_ohlcv_append_only` | 5 sor beírás, újrafuttatásra 0 új sor (idempotens) |
| `test_insert_ohlcv_filters_columns` | Extra oszlopok csendesen ignorálódnak |
| `test_query_range_returns_correct_rows` | `query_range()` visszaad 7 napot timestamp BETWEEN-nel |
| `test_query_range_pl_returns_polars` | `query_range_pl()` Polars DataFrame-et ad vissza |
| `test_asof_join_predictions_features` | ASOF join helyes sorpárosítást ad |
| `test_dataset_exists_false_on_empty` | Üres tábla → `dataset_exists() = False` |
| `test_latest_open_time_returns_max` | `latest_open_time()` a beírt sorok maximumát adja |

---

### `test_duckdb_stats.py`

| Teszt | Mit ellenőriz |
|-------|---------------|
| `test_collect_report_skips_missing_db` | Hiányzó DB-re `DuckDBStatsReport` üres táblákkal tér vissza |
| `test_collect_report_row_count` | 100 sor betöltés után `row_count == 100` |
| `test_collect_report_time_range` | `min_ts` és `max_ts` helyesek |
| `test_collect_report_null_ratios` | 0.0 null arány tiszta adatra |
| `test_format_report_contains_table_names` | `format_report()` tartalmazza a tábla neveket |

---

### `test_validate.py`

| Teszt | Mit ellenőriz |
|-------|---------------|
| `test_assert_zero_passes` | 0 sort visszaadó SQL → átmegy |
| `test_assert_zero_fails` | >0 sort visszaadó SQL → `AssertionError` |

---

### `test_duckdb_stats_audit.py`

`raw_manifest_audit` és `log_dataset_check` szintetikus adaton és hiányzó DB/tábla esetén.

| Teszt | Mit ellenőriz |
|-------|---------------|
| `test_raw_manifest_audit_happy_path` | Nem dob hibát ha az adat megvan |
| `test_raw_manifest_audit_missing_db` | Nem dob hibát ha a DB fájl hiányzik |
| `test_raw_manifest_audit_missing_table` | Nem dob hibát ismeretlen táblanévre |
| `test_log_dataset_check_ohlcv_happy_path` | `log_dataset_check` ohlcv-re nem dob hibát |
| `test_log_dataset_check_missing_db` | Nem dob hibát ha a DB hiányzik |
| `test_log_dataset_check_missing_dataset` | Nem dob hibát üres táblára |

---

## sanity/ — Éles DB Adat-invariánsok

**Előfeltétel:** Éles `database/solusdt/solusdt.duckdb` DB megléte. Ha hiányzik, minden teszt `pytest.skip()`.

### `test_ohlcv.py`

| Teszt | Mit ellenőriz |
|-------|---------------|
| `test_ohlcv_table_exists` | Az ohlcv tábla létezik és nem üres |
| `test_ohlcv_row_count_positive` | Legalább 1 sor |
| `test_ohlcv_required_columns` | Mind a 10 oszlop jelen van |
| `test_ohlcv_date_range_reasonable` | min_ts >= 2022-01-01, max_ts <= ma |
| `test_ohlcv_no_nulls` | Nincs null érték egyik oszlopban sem |
| `test_ohlcv_invariant` | `open`, `high`, `low`, `close` konzisztencia (high >= low, stb.) |
| `test_ohlcv_volume_positive` | `volume > 0` minden sorra |
| `test_ohlcv_1min_cadence` | Egymást követő timestampek 60s különbséggel |
| `test_ohlcv_no_duplicate_timestamps` | Nincs duplikált `open_time` |

---

### `test_features_target.py`

| Teszt | Mit ellenőriz |
|-------|---------------|
| `test_feat_ohlcv_quant_table_exists` | `feat_ohlcv_quant` tábla létezik |
| `test_feat_ohlcv_quant_required_metadata_columns` | `open_time`, `close`, `available_ts`, `lookback_end_ts` jelen van |
| `test_feat_ohlcv_quant_available_ts_no_lookahead` | `available_ts <= open_time` minden sorban |
| `test_feat_ohlcv_quant_row_count` | feature sorok száma a várható tartományban van |
| `test_target_required_columns` | `long_mfe_fw60`, `short_mfe_fw60` és fw60 outcome oszlopok jelen vannak |
| `test_target_long_mfe_range` | `long_mfe_fw60` értékek az elvárt tartományban (logreturn) |
| `test_target_short_mfe_range` | `short_mfe_fw60` értékek az elvárt tartományban (logreturn) |

---

### `test_feature_lag_invariants.py`

| Teszt | Mit ellenőriz |
|-------|---------------|
| `test_available_ts_equals_open_time` | `available_ts == open_time` minden sorra |
| `test_first_row_ohlcv_feats_null` | Az első sor OHLCV-alapú feature-jei `NULL` (t-1 lag) |
| `test_p2_features_not_null_first_row` | `T_MINUS_1_SKIP` tagjai az első sorban `NOT NULL` |
| `test_rsi_lag_correlation` | RSI t. bar korrelál az ohlcv t-1 bar close-ával |
| `test_close_position_differs_from_current_bar` | `feat_close_position` != jelenlegi bar `(close-low)/(high-low)` |

---

### `test_predictions.py`

| Teszt | Mit ellenőriz |
|-------|---------------|
| `test_predictions_table_exists` | `predictions` tábla létezik |
| `test_predictions_required_columns` | `long_pred`, `short_pred` jelen van |
| `test_predictions_score_range` | `long_pred`, `short_pred` ∈ [0, 1] |
| `test_predictions_label_end_ts` | `label_end_ts > open_time` minden sorra |
| `test_predictions_alignment_with_ohlcv` | `predictions.open_time ⊆ ohlcv.open_time` |

---

### `test_target_window.py`

| Teszt | Mit ellenőriz |
|-------|---------------|
| `test_target_1_following_boundary` | `ROWS BETWEEN 1 FOLLOWING` — az aktuális bar nincs a forward window-ban |
| `test_null_count_equals_horizon` | Pontosan 60 NULL sor az utolsó soroknál |
| `test_nulls_at_tail` | NULL-ok csak a target tábla végén vannak |
| `test_null_symmetry` | Long és short NULL count egyenlő |
| `test_synthetic_10_bar` | Szintetikus 10 barios adaton forward window ellenőrzés |

---

## perf/ — Teljesítmény Benchmarkok

Wall-clock idő mérés az éles DB-n. Skippel ha a DB hiányzik.

### `test_query_timing.py`

| Teszt | Limit |
|-------|-------|
| `test_timing_ohlcv_count` | COUNT(*) < 2s |
| `test_timing_ohlcv_range_query_7d` | 7 napos range query < 3s |
| `test_timing_ohlcv_daily_aggregation` | Napi agg (GROUP BY day) < 5s |
| `test_timing_ohlcv_rolling_sma60` | Rolling SMA60 teljes history < 15s |
| `test_timing_insert_100k_rows` | 100k sor INSERT (szintetikus temp DB) < 10s |
| `test_timing_feat_count` | feat COUNT(*) < 2s |
| `test_timing_feat_range_query_7d` | feat 7 napos range < 3s |
| `test_timing_feat_range_query_30d` | feat 30 napos range < 5s |
| `test_timing_feat_groupby_year` | feat GROUP BY year < 3s |
| `test_timing_feat_groupby_month` | feat GROUP BY month < 3s |
| `test_timing_target_count` | target COUNT(*) < 2s |
| `test_timing_target_label_groupby` | target label GROUP BY < 3s |
| `test_timing_target_range_query_30d` | target 30 napos range < 3s |
| `test_timing_predictions_count` | predictions COUNT(*) < 2s |
| `test_timing_predictions_range_query_7d` | predictions 7d range < 3s |
| `test_timing_predictions_range_query_30d` | predictions 30d range < 5s |
| `test_timing_predictions_groupby_year` | predictions GROUP BY year < 3s |
| `test_timing_predictions_groupby_month` | predictions GROUP BY month < 3s |
| `test_timing_asof_join` | ASOF JOIN predictions⋈features < 10s |




---

<!-- Source: 1320_pipeline_tests.md -->



# tests/sync_tables/ + sync_pipeline/ — Pipeline Tesztek

`src/database/tests/sync_tables/` és `src/database/tests/sync_pipeline/`

Pipeline tesztek három szinten: smoke (mock adattal), sanity (lookahead bias ellenőrzés), integration (cross-layer teljes pipeline flow). A `sync_pipeline/smoke/` a CLI belépési pont helper függvényeit teszteli.

---

## sync_tables/smoke/ — Pipeline Funkciók Mock Adattal

Minden smoke teszt szintetikus adattal és mocked Binance API-val fut. Nem igényelnek éles DB-t.

### `test_sync_ohlcv.py`

| Teszt | Mit ellenőriz |
|-------|---------------|
| `test_sync_ohlcv_inserts_rows` | Mocked Binance → 5 sor kerül az ohlcv táblába |
| `test_sync_ohlcv_idempotent` | Újrafuttatásra 0 új sor (append-only guard) |
| `test_sync_ohlcv_stale_guard` | Ha `open_time_ms_from` régebbi mint a DB max-a, a guard véd |
| `test_sync_ohlcv_column_count` | 10 oszlop kerül a táblába (nem 12) |

---

### `test_sync_features.py`

| Teszt | Mit ellenőriz |
|-------|---------------|
| `test_sync_features_expected_columns` | `available_ts` és `lookback_end_ts` jelen van az outputban |
| `test_sync_features_feat_prefix` | Minden feature oszlop `feat_` prefix-szel kezdődik |
| `test_sync_features_idempotent` | Újrafuttatásra 0 új sor (append-only guard) |
| `test_sync_features_solusdt_profile` | `solusdt_fw60` profil által előírt feature-ök jelen vannak |

---

### `test_sync_predictions.py`

| Teszt | Mit ellenőriz |
|-------|---------------|
| `test_sync_predictions_long_pred_written` | `long_pred` oszlop jelen van a táblában |
| `test_sync_predictions_short_pred_written` | `short_pred` oszlop jelen van a táblában |
| `test_sync_predictions_idempotent` | Újrafuttatásra 0 új sor |
| `test_sync_predictions_scores_in_range` | `long_pred`, `short_pred` ∈ [0, 1] a szintetikus mock modellre |

---

### `test_sync_targets.py`

| Teszt | Mit ellenőriz |
|-------|---------------|
| `test_sync_targets_writes_fw60_outcome_columns` | `long_mfe_fw60`, `short_mfe_fw60` és összes fw60 outcome oszlop jelen van; idempotens |
| `test_sync_targets_last_rows_are_null` | Az utolsó 60 sor `NULL` `long_mfe_fw60` (horizon=60) |
| `test_sync_targets_fw60_values_are_nonzero` | Valid sorokban `long_mfe_fw60` és `short_mfe_fw60` nem null és nem nulla |

---

## sync_tables/sanity/ — Lookahead Bias Ellenőrzések

Szintetikus adattal, de valódi `compute_features_polars` hívással. Az összes teszt a feature pipeline determinizmusát és lookahead mentességét ellenőrzi.

### `test_leak_prevention.py`

| Teszt | Mit ellenőriz |
|-------|---------------|
| `test_day_range_position_no_intraday_future_leak` | `feat_day_range_position` nem változik ha jövőbeli barak kerülnek hozzá |
| `test_ohlcv_features_independent_of_appended_future_bars` | OHLCV-alapú feature-ök (RSI, ROC, SMA, BB) determinisztikusak, nem változnak jövőbeli adatsoron |
| `test_deterministic_time_features_stable_across_dataset_sizes` | Timestamp-alapú feature-ök (`T_MINUS_1_SKIP` tagjai) ugyanazok különböző méretű DataFrame-en |

**Teszt módszer — future-bar append:**
1. Feature-ök számítása N soros DataFrame-en
2. Feature-ök számítása N+100 soros DataFrame-en (azonos sor 0..N-1)
3. Ellenőrzés: a közös sorok értékei numerikusan azonosak

Ez a teszt közvetlenül azt ellenőrzi, amit a t-1 lag garantál.

---

## sync_tables/integration/ — Cross-Layer Pipeline Flow

Szintetikus adattal és mocked modellel (nincs szükség éles DB-re vagy real model artifact-ra). Az összes pipeline réteg együtt fut.

### `test_pipeline_integration.py`

| Teszt | Mit ellenőriz |
|-------|---------------|
| `test_ohlcv_to_predictions_cross_layer_alignment` | Teljes pipeline: ohlcv → features → target → predictions; cross-layer timestamp egyezés |
| `test_features_close_matches_ohlcv_close` | `feat_ohlcv_quant.close == ohlcv.close` minden közös `open_time`-ra |
| `test_target_open_time_subset_of_ohlcv` | `target.open_time ⊆ ohlcv.open_time` |

**Ellenőrzött cross-layer invariánsok (`test_ohlcv_to_predictions_cross_layer_alignment`):**

```
predictions.open_time ⊆ ohlcv.open_time
predictions.open_time ⊆ feat_ohlcv_quant.open_time
predictions.close == ohlcv.close (ABS diff < 1e-8)
predictions.long_pred ∈ [0, 1]
predictions.short_pred ∈ [0, 1]
```

**Mock setup:**
- `_MockModel`: `predict_proba` mindig `[0.35, 0.65]`-öt ad vissza
- `monkeypatch` a `utils.load_asset_config`, `champion_models_for_asset`, `_load_model_artifacts` függvényekre
- 250 szintetikus OHLCV bar (2024-01-01 00:00 – 04:09)

---

## sync_pipeline/smoke/ — CLI Helper Tesztek

`02_sync_pipeline.py` belső helper függvényeinek tesztjei. Nincs DB vagy Binance hívás — tisztán logikai tesztek.

### `test_sync_pipeline_helpers.py`

`importlib` segítségével tölti be a `02_sync_pipeline.py`-t (a számos prefix miatt közvetlen import nem lehetséges).

**Tesztelt függvények:** `_monthly_chunks`, `_resolve_tables`

| Teszt | Mit ellenőriz |
|-------|---------------|
| `test_monthly_chunks_single_month` | 1 hónapnál rövidebb tartomány → 1 tuple |
| `test_monthly_chunks_splits_correctly` | 6 hónapos tartomány, chunk=3 → pontosan 2 chunk |
| `test_monthly_chunks_contiguous` | Minden chunk vége == következő chunk kezdete (nincs rés) |
| `test_monthly_chunks_equal_start_end_returns_empty` | Azonos start/end → üres lista |
| `test_monthly_chunks_returns_list_of_tuples` | Return type: `list[tuple[str, str]]` |
| `test_resolve_tables_default_returns_all` | `--tables` nélkül: teljes tábla szett |
| `test_resolve_tables_subset` | `--tables=ohlcv,features` → csak ez a kettő |
| `test_resolve_tables_skip_ohlcv_removes_ohlcv` | `--skip-ohlcv` → ohlcv nincs a szettben |
| `test_resolve_tables_unknown_table_exits` | Ismeretlen tábla → `sys.exit(1)` |
| `test_resolve_tables_whitespace_stripped` | Szóközök az elemek körül elfogadottak |




---

<!-- Source: 2000_features.md -->



﻿# 2000 — Feature Layer

A feature layer a ChronoQuant ML pipeline bemeneti adatrétege: minden modellezési döntés egy feature profilt feltételez, amelyet a `feat_ohlcv_quant` DuckDB tábla szolgáltat ki.

---

## Overview

A feature layer az `ohlcv` nyers adatból kiszámított technikai és statisztikai indikátorokból áll. Ezek a jellemzők leírják a piac állapotát a predikció időpontjában, és kizárólag olyan információt tartalmazhatnak, amely a `t` időpontnál nem újabb.

```{mermaid}
flowchart TD
  A[ohlcv tábla\nopen_time, close, vol...] --> B[sync_features\n_features_polars.py]
  B --> C[feat_ohlcv_quant tábla\n202 feat_ oszlop]
  C --> D[sampling modul\n00_create_sample.py]
  D --> E[LightGBM tanítás\n01_train_model.py]
  C --> F[sync_predictions\nlive predict / predict_proba]
```

**Aktív feature profil:** `solusdt_fw60` — 208 feature, 25 csoport, 1 perces SOLUSDT OHLCV báron.

**Implementáció:** [`src/data_handling/sync_tables/_features_polars.py`](src/data_handling/sync_tables/_features_polars.py)
**Konfiguráció:** [`config/features.json`](config/features.json)
**Kód referencia:** [`_doc_/2200_features_polars.md`](_doc_/2200_features_polars.md)

---

## Feature Csoportok

| # | Csoport | Db | Domináns ablak(ok) |
|---|---------|----|--------------------|
| 1 | Momentum | 9 | w=14 (RSI, Stoch, ADX), w=20 (CCI), w=14/140 (ROC) |
| 2 | Trend | 8 | w=14, 140 (SMA, EMA ratio), w=10 (KAMA), fast=12/slow=26/sig=9 (MACD) |
| 3 | Volatility | 12 | w=14, 140 (BB), w=14 (NATR), w=20 (hist_vol), w=10, 30, 60 (GK, Parkinson) |
| 4 | Volume | 6 | w=14 (vol_sma, OBV_roc, MFI), w=20 (CMF), kumulatív (OBV) |
| 5 | Price Action | 6 | w=14 (returns std/skew/kurt), ablak nélkül (log return, hml_range, close_pos) |
| 6 | Market Structure | 6 | w=5 (swing high/low, trend counts) |
| 7 | Activity | 11 | w=10, 30 (taker flow), w=10, 30, 60 (quote vol ratio, trade count ratio) |
| 8 | Return Distance | 15 | w=10, 30, 60 (return, return_z, dist_high, dist_low, rolling_drawdown) |
| 9 | Regime Rank | 15 | w=10, 30 (vol/quote_vol/trade rank), w=20, 60 (natr/hist_vol/bb_width rank) |
| 10 | Candle Shape | 9 | ablak nélkül (body_ratio, wick_ratio), w=10, 30 (sma variants) |
| 11 | Trend Slope | 3 | w=10, 30 (EMA slope, directional agreement) |
| 12 | Interaction | 12 | w=5, 10, 30 (RSI/ROC delta, vol_adj_return) |
| 13 | Time / Session | 12 | nincs backward ablak — determinisztikus (óra, nap, szesszió, heti nyitó) |
| 14 | Autocorrelation | 5 | lag=1/5, w=30, 60 (return autocorr), cross=10/60 (variance ratio) |
| 15 | Drawdown & Timing | 12 | w=10, 30, 60 (recovery ratio, max drawdown, time since high/low) |
| 16 | Pattern Flags | 10 | ablak nélkül (doji, hammer, engulf), w=10, 30, 60 (bull_bars_ratio) |
| 17 | Gap | 3 | ablak nélkül (gap_open), w=10, 30 (abs sma) |
| 18 | Efficiency | 3 | w=10, 30, 60 (Kaufman efficiency ratio) |
| 19 | SR Levels | 8 | w=10, 30, 60 (ATR dist high/low), **prev session H/L (1440 bar shift)** |
| 20 | Tail Risk | 9 | w=10, 30, 60 (pos/neg return mean, return asymmetry) |
| 21 | Extended Accel | 2 | w=10, 30 (return momentum delta) |
| 22 | Ichimoku | 7 | built-in: 9, 26, 52 (tenkan, kijun, senkou_b, cloud) |
| 23 | Donchian | 9 | w=10, 30, 60 (width, position, breakout) |
| 24 | Linear Regression | 9 | w=10, 30, 60 (slope, R², residual) |
| 25 | Session Relative | 4 | ablak nélkül (day_range_position, day_open_return, bars_into_session_norm, weekly_open_return) |

---

## Üzleti és módszertani háttér

### Miért kritikus ez a lépés?

A feature layer az egyetlen csatorna, amelyen keresztül a modell a piacot látja. Ha egy feature jövőbeli adatot szivárogtat be (lookahead leak), a modell in-sample kiválóan teljesít, de live éles predikción azonnal összeomlik — nincs jövőbeli close ár, amelyre az indikátor támaszkodna. Ha a warmup kezelés hibás, a null sorok torzítják az imputation-t, és a sampling hamis tanulási pontokat vonhat be a train halmazba.

Ezért a feature layer helyes implementációja kötelező kapu minden modellezési munka előtt: ha a source-of-truth kód módosul, a sampling és a tanítás pipeline-nak is újra kell futnia.

### Miért ezt a megközelítést?

```{mermaid}
flowchart LR
  Q[Feature stratégia] --> A[Raw OHLCV árak\nNO: nem-stacionárius\nNO: skewed distribution]
  Q --> B[Normalizált árszintek\nNO: ablak-függő skálázás\nNO: cross-sample inkonzisztencia]
  Q --> C[Derivált technikai indikátorok\nOK: stacionáriusabb\nOK: domain knowledge\nOK: Választott]
```

| Megközelítés | Előny | Hátrány | Státusz |
|---|---|---|---|
| Derivált technikai + statisztikai indikátorok (jelenlegi) | Stacionáriusabb, domain knowledge beépítve, széles szemantikai lefedés | Sok feature → szelekció szükséges, warmup overhead | ✅ Választott |
| Raw OHLCV ár + volume | Egyszerű, nincs warmup | Nem-stacionárius, LightGBM számára nehezen értelmezhető | ❌ Elvetett — szignálminőség hiány |
| Csak momentum + trend (szűk készlet) | Gyors warmup, olvasható | Elveszett context (volatilitás, aktivitás, struktúra) | ⚠️ Fontolóra vehető — egyszerű baseline-hoz |
| Deep learning embedding (raw OHLCV) | Automatikus feature extraction | Infrastruktúra-, adatigény, interpretálhatatlan | ❌ Elvetett — projekt scope-on kívül |

### t-1 lag: miért kell és hogyan működik?

A feature `t` időpontnál kerül kiszámításra a `t` bar close ára alapján. A modell predikciója azonban a következő bar — `t+1` — irányáról szól. Ha a modell a `t`-beli feature-t a `t`-beli targethez illeszti, nincs lookahead; de ha a modell live-on fut, a `t+1` bar open-jén vásárol, azaz a döntés és a végrehajtás között eltelt egy bar.

```{mermaid}
graph TD
  F["feature(t)\nkiszámítva close[t] alapján"]
  T["target(t)\nfw60 outcome t+1..t+60-ból"]
  L["live döntés t+1 open-jén"]
  F -->|t-1 lag eltolás| L
  T -->|label a feature-hez| F
  F -.->|tárolt feat_ oszlop| DB[feat_ohlcv_quant]
```

A `_apply_t1_lag_pl()` függvény minden feature oszlopot egységesen `shift(1)`-gyel tol el. Ez azt jelenti, hogy a tárolt `feat_` érték az előző bar indikátorát tartalmazza — ezáltal a modell tanulása és live inferenciája teljesen konzisztens.

**Kivétel:** Time/Session és Session Relative feature-ök (`T_MINUS_1_SKIP` tag) — ezek az `open_time` timestamp-ből vagy a nap/hét nyitójából deterministikusan számolódnak, és nem hordoznak jövőbeli piacadatot, ezért lag nélkül is lookahead-mentesek.

**Szabály:** Minden új feature-t `_apply_t1_lag_pl()` hatókörén belül kell definiálni, kivéve ha explicit `T_MINUS_1_SKIP` annotációval van ellátva.

### Warmup bars és az adatbiztonságos minta határai

```{mermaid}
flowchart TD
  START[Adat kezdete\nohlcv t=0] --> WU[Warmup fázis\n0 → max_warmup bars]
  WU --> VALID[Valid feature tartomány\nmax_warmup+1 → T]
  VALID --> SAMPLE[Mintavételezés\nlookback_end_ts >= max_warmup+1]

  WU -->|w=140 SMA/EMA| W140[140 bar null\n= 2 óra 20 perc]
  WU -->|Ichimoku Senkou B| W52[52 bar null]
  WU -->|prev_session H/L| W1441[1441 bar null\n= 24 óra 1 perc WARN]
```

| Feature(ek) | Warmup (bar) | Valós idő |
|---|---|---|
| Legtöbb rolling feature | 10–60 | 10–60 perc |
| Ichimoku Senkou B | 52 | 52 perc |
| SMA/EMA/BB w=140 | 140 | 2 óra 20 perc |
| **prev_session_high/low_dist** | **1441** | **24 óra 1 perc** |

A `prev_session_high/low_dist` feature 1440 bar-t shift-el (előző naptári nap max/min), majd jön a t-1 lag — összesen 1441 bar null az elején.

**Szabály:** A sampling modul `lookback_end_ts` értékének min. 1441 barral el kell tolódnia az adat kezdetétől, hogy a warmup nullák ne kerüljenek be a tanítási ablakba.

### Paraméter alapértékek és indoklásuk

| Paraméter | Érték | Indoklás |
|---|---|---|
| Domináns short ablak | `w=10` | 10 perces kontextus elegendő gyors momentum jelekhez; 5-nél zajosabb, 14-nél lassabb |
| Domináns mid ablak | `w=14` | Klasszikus technikai elemzés konvenció (RSI, ATR, ADX); széles elfogadottság |
| Domináns long ablak | `w=30` | 30 perces félóra-szintű kontextus; közelíti a kereskedési szesszió egységét |
| Széles trend ablak | `w=140` | ~2.3 óra; a nap-szintű trend kontextus közelítése 1 perces bárokon |
| Ichimoku Senkou B | `w=52` | Ichimoku szabványos beállítás (26 periódus × 2) |
| prev_session shift | `1440` | Pontosan egy napnyi (00:00–23:59 UTC) bar a shift; 1440 = 24×60 |
| Feature prefix | `feat_` | Névtérelkülönítés a target és raw OHLCV oszlopoktól |
| t-1 lag | `shift(1)` | Egy bar eltolás; a smallest production granularitás |

### Ismert kockázatok és korlátok

| Kockázat | Tünet | Mitigáció |
|---|---|---|
| `prev_session` gap kockázat | Ha a piac gappel nyit és az OHLCV nem pontosan 1440 bar/nap, a shift nem a nap határán landol | Elhagyható feature set; vagy explicit napos aggregáció indexelés alapján |
| Warmup null → imputáció torzítás | A null sorok véletlenszerű középértékkel tölthetők be, ami hamis szignált ad a modellnek | `lookback_end_ts` offset >= 1441 bar kötelező; soha ne impute-old a warmup tartományban |
| Feature multikollinearitás | Sok csoport (pl. Momentum + Interaction + Return Distance) átfedő információt hordoz | LightGBM természetes feature importance szelekciója; opcionálisan SHAP-alapú pruning |
| Feature count overhead | 202 feature → lassabb tanítás, overfitting veszély kis mintán | Szűk feature profil kísérletekhez (`config/features.json` profile-ok) |
| Live warmup hiány | Ha live deploy-ban nincs elegendő history a széles ablakokhoz, az indikátorok nullák | Deployment előtt ellenőrizni: min. 1441 bar history rendelkezésre áll-e |
| Time/Session feature timezone eltolás | `_tmp_day` UTC alapú; daylight saving nem releváns, de Binance maintenance window (UTC 00:00) érinthet teljes napokat | Ismert, kezelt; maintenance window általában < 60 perc, az indikátor csillapodik |

### Validációs checklist

- [ ] Minden `feat_` oszlopban az első 1441 sor null (vagy az adott feature saját warmup-ja, amelyik nagyobb)
- [ ] Nincs lookahead: `_apply_t1_lag_pl()` alkalmazva az összes nem-`T_MINUS_1_SKIP` feature-re
- [ ] `prev_session_high_dist` és `prev_session_low_dist` értékei az előző naptári nap max/min-jét tükrözik (nem az aktuális napét)
- [ ] A `feat_ohlcv_quant` tábla oszlopszáma megfelel a `config/features.json` `solusdt_fw60` profiljának
- [ ] A sampling `lookback_end_ts` offset >= 1441 bar — ellenőrzés: `audit_feature_table()` null count riport
- [ ] Live prediction pipeline ugyanazon feature-definícióval fut, mint a tanítási pipeline (közös `_features_polars.py` forrás)
- [ ] Új feature hozzáadásakor: `T_MINUS_1_SKIP` vagy `_apply_t1_lag_pl()` hatókörbe esik-e?




---

<!-- Source: 2010_feature_engineering.md -->



﻿# Feature Engineering — Moduláris analízis layer

**Script:** `src/modeling/01_feature_engineering.ipynb`
**Library:** `src/modeling/feature_engineering/`
**Output:** `database/<asset_id>/feature_engineering/<run_id>/`

---

## Célja

A `quant_train` DuckDB tábla `feat_*` oszlopait négy egymástól független dimenzió
mentén vizsgálja. Az eredmény egy determinisztikusan generált `feature_set.json`,
amelyet a `00_create_sample.py` és a `02_hyper_param_search.py` fogyaszt.

---

## Input

| Forrás | Tartalom |
|--------|----------|
| `quant_train` DuckDB tábla | `feat_*` oszlopok + `long_mfe_fw60`, `short_mfe_fw60` target oszlopok |
| `FeatureEngineeringConfig` | Küszöbértékek minden analízis lépéshez |

A `quant_train` tábla csak NULL-mentes target sorokat tartalmaz (a build pipeline
garantálja). Az első ~1441 sor `feat_*` értékei NULL-ok lehetnek (t-1 lag warmup),
de a sampling lookback offset kizárja ezeket a tanítási ablakból.

---

## Négy analízis lépés

### 1. Quality — `analyze_quality()`

Univariáns minőségi ellenőrzés minden `feat_*` oszlopra.

| Metrika | Döntés | Feltétel |
|---------|--------|----------|
| `null_rate` | drop | > `max_null_rate` (0.01) |
| `inf_rate` | drop | > `max_inf_rate` (0.001) |
| `variance` | drop | < `min_variance` (1e-8) |
| `outlier_ratio` | review | > `max_outlier_ratio` (0.05) |
| — | keep | minden más |

Output schema: `feature, null_rate, inf_rate, variance, outlier_ratio, decision, drop_reason`

### 2. Target Relation — `analyze_target_relation()`

Pearson (`CORR`) és Spearman (RANK-alapú) korreláció minden `feat_*` × target párra.
`signal_proxy = |ρ_spearman|`

| Döntés | Feltétel |
|--------|----------|
| `leakage` | `|ρ| > 0.95` — jövőbeli adat szivárgás gyanú |
| `weak` | `|ρ| < 0.01` — nincs érdemi szignál |
| `keep` | minden más |

Egy feature csak akkor kerül ki, ha **mindkét** targetre `weak` (vagy `leakage`).

Output schema: `feature, target, pearson_r, spearman_rho, signal_proxy, leakage_flag, decision`

### 3. Redundancy — `analyze_redundancy()`

Pearson korrelációs mátrix alapján klaszterezés (union-find algoritmus).
Ha két feature `|Pearson r| ≥ pearson_cluster_thr` (0.95), egy klaszterbe kerülnek.
Klaszterenként a legkisebb indexű feature a reprezentatív (`keep`), a többi `drop`.

A korrelációs mátrix max `redundancy_max_rows` (500 000) véletlenszerű sorból számolódik
RAM-hatékonyság érdekében.

Output schema: `feature, cluster_id, is_representative, max_pearson, decision, drop_reason`

### 4. Stability — `analyze_stability()`

Az adatot `stability_bucket_days` (90) napos, nem-átfedő időablakokra osztja.
Minden (feature, bucket) párra Spearman korreláció mindkét targettel.
`drift = |ρ_bucket − ρ_baseline|`

| Flag | Feltétel | Hatás a végeredményre |
|------|----------|-----------------------|
| `stable` | drift ≤ 0.15 | selected |
| `review` | 0.15 < drift ≤ 0.30 | review lista |
| `unstable` | drift > 0.30, nem az utolsó 2 bucket | review lista |
| `decayed` | drift > 0.30 **és** az utolsó 2 bucket | **drop** |

Output schema: `feature, bucket_idx, bucket_start, bucket_end, n, null_rate, mean, std, spearman_long, spearman_short, drift_long, drift_short, stability_flag`

---

## Output — notebook inline logika

A `feature_set.json` generálása **kizárólag** a `01_feature_engineering.ipynb`
utolsó output celláiban történik — nincs külön reporting modul.

Egy feature a **selected** listába kerül, ha mind a négy feltétel teljesül:
1. quality → `keep`
2. target_relation → legalább egy targetre `keep`
3. redundancy → `keep` (reprezentatív)
4. stability → nincs `decayed` bucket

### `feature_set.json` séma

```json
{
  "run_id": "run_20240601_120000",
  "asset_id": "solusdt",
  "created_at": "2024-06-01 12:00:00",
  "target_cols": ["long_mfe_fw60", "short_mfe_fw60"],
  "selected": ["feat_rsi_14", "feat_roc_10", ...],
  "dropped": [
    {"col": "feat_foo", "reason": "quality: null_rate=0.05 > max=0.01"}
  ],
  "review": ["feat_bar"],
  "thresholds": {
    "max_null_rate": 0.01,
    "max_inf_rate": 0.001,
    "min_variance": 1e-8,
    "max_outlier_ratio": 0.05,
    "min_spearman_abs": 0.01,
    "max_spearman_leakage": 0.95,
    "pearson_cluster_thr": 0.95,
    "redundancy_max_rows": 500000,
    "stability_bucket_days": 90,
    "max_drift_threshold": 0.3
  }
}
```

### `analyst_report.md`

Human-readable összefoglaló: selected / dropped / review listák, drop okok, paraméterek.

---

## Futtatás

```bash
# Jupyter notebook interaktívan
uv run jupyter notebook src/modeling/01_feature_engineering.ipynb
```

A notebook automatikusan generál `run_id`-t (`run_YYYYMMDD_HHMMSS` formátumban).
Az output path kiíródik a konzolra futás végén.

---

## Kapcsolódó fájlok

| Fájl | Tartalom |
|------|----------|
| `src/modeling/feature_engineering/__init__.py` | Publikus API |
| `src/modeling/feature_engineering/config.py` | `FeatureEngineeringConfig` |
| `src/modeling/feature_engineering/quality.py` | Quality analízis |
| `src/modeling/feature_engineering/target_relation.py` | Target relation analízis |
| `src/modeling/feature_engineering/redundancy.py` | Redundancia klaszterezés |
| `src/modeling/feature_engineering/stability.py` | Időbeli stabilitás |
| `src/modeling/feature_engineering/reporting.py` | Output generálás |
| `src/modeling/feature_engineering/tests/smoke/test_package.py` | Smoke tesztek |
| `_doc_/2000_features.md` | Feature layer metodológia (lookahead, t-1 lag, csoportok) |
| `_doc_/5000_modelling.md` | Modeling pipeline áttekintés |




---

<!-- Source: 2100_sync_features.md -->



# sync_features.py — Feature Számítás és Beírás

`src/database/sync_tables/sync_features.py`

OHLCV adatok beolvasása, feature pipeline futtatása Polars LazyFrame-mel, majd beírás a `feat_ohlcv_quant` táblába. t-1 lag kötelező az összes OHLCV-alapú feature-re.

---

## `sync_features(start_time, lookback_bars, end_time, asset_id)`

**Célja:** Feature-ök kiszámítása a megadott időtartományra és beírás.

**Paraméterek:**

| Paraméter | Típus | Alap | Leírás |
|-----------|-------|------|--------|
| `start_time` | `str` | — | Számítás kezdete (`YYYY-MM-DD HH:MM:SS`) |
| `lookback_bars` | `int` | config | Lookback ablak mérete barokban (warm-up periódus) |
| `end_time` | `str \| None` | `None` | Számítás vége (`None` = legújabb OHLCV bar) |
| `asset_id` | `str \| None` | `None` | Asset azonosító |

---

## Belső folyamat

```{mermaid}
sequenceDiagram
    participant CALLER as sync_features()
    participant OHLCV as ohlcv tábla
    participant POLARS as compute_features_polars()
    participant FEAT as feat_ohlcv_quant

    CALLER->>OHLCV: query_range_pl(db_path, "ohlcv", start-lookback, end)
    OHLCV-->>CALLER: pl.DataFrame (OHLCV sorok)
    CALLER->>POLARS: compute_features_polars(df_ohlcv, indicators, ...)
    POLARS-->>CALLER: pl.DataFrame (feat_* oszlopok)
    CALLER->>CALLER: available_ts = open_time (t-1 lag már megtörtént)
    CALLER->>CALLER: lookback_end_ts = open_time
    CALLER->>CALLER: df[start_time:end_time] (lookback wam-up sorok ledobása)
    CALLER->>FEAT: insert_feat_ohlcv_quant(conn, df_final)
```

---

## t-1 Lag mechanizmus

A `compute_features_polars` belül az `_apply_t1_lag_pl` függvény minden `feat_*` oszlopot 1 barral eltol (`shift(1)`), kivéve a `T_MINUS_1_SKIP` frozenset tagjait (P2 időindex feature-ök).

Az eltolás után:
- `available_ts` = `open_time` — ez azt jelenti: "a feature e bar **nyitásakor** lett elérhető"
- Az ASOF join a predictions-hoz: `predictions.open_time >= feat.available_ts`
- Eredmény: a t. bar predikciója csak t-1 bar (és korábbi) feature-öket lát

---

## `build_lag_snapshot(db_path, feature_cols, start, end)`

**Célja:** Modeling-ready snapshot készítése ASOF join alapján.

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `db_path` | `str` | DuckDB fájl elérési útja |
| `feature_cols` | `list[str]` | Feature oszlopok listája |
| `start` | `str` | Időtartomány kezdete |
| `end` | `str` | Időtartomány vége |

**Visszatérési érték:** `pd.DataFrame` — `predictions` ⋈ `feat_ohlcv_quant` ASOF join eredménye.

**Belső hívás:** `asof_join_predictions_features(db_path, feature_cols, start, end)`

**Felhasználás:** A modeling réteg (`sync_predictions.py` és training pipeline) ezt hívja a feature snapshot elkészítéséhez.




---

<!-- Source: 2200_features_polars.md -->



# _features_polars.py — Feature Computation Engine

`src/data_handling/sync_tables/_features_polars.py`

A fő feature számítási motor. 30+ indikátor csoport, Polars LazyFrame API, t-1 lag kötelezően, numpy helpers a rolling statisztikákhoz. Ez a modul a lookahead bias elleni legfontosabb védelmi vonal.

---

## `compute_features_polars(df_ohlcv, indicators, feat_prefix, available_activity, targets_cfg)`

**Célja:** OHLCV DataFrame-ből feature DataFrame generálása a megadott indikátor konfigurációk alapján.

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `df_ohlcv` | `pl.DataFrame` | Input OHLCV sorok (`open_time`, `open`, `high`, `low`, `close`, `volume`) |
| `indicators` | `dict` | Indikátor konfigurációk (`config/features.json` alapján) |
| `feat_prefix` | `str` | Feature oszlopok prefix-e (pl. `"feat_"`) |
| `available_activity` | `list` | Aktivitás metaadatok (ritkán használt) |
| `targets_cfg` | `list` | Target konfiguráció (forward window info) |

**Visszatérési érték:** `pl.DataFrame` — eredeti OHLCV oszlopok + összes `feat_*` oszlop.

**Belső lépések:**
1. LazyFrame-re konvertálás
2. Minden engedélyezett indikátor csoport `_add_*_pl()` függvénye meghívva
3. `_apply_t1_lag_pl(lf, p)` — t-1 lag alkalmazása
4. `_clean_features_pl(lf, p)` — végtelen értékek (`inf`) null-ra cserélése
5. `.collect()` → `pl.DataFrame`

---

## `T_MINUS_1_SKIP`

```python
T_MINUS_1_SKIP: frozenset[str] = frozenset({
    "feat_bars_into_session_norm",
    "feat_hour_sin",
    "feat_hour_cos",
    "feat_dayofweek_sin",
    "feat_dayofweek_cos",
    "feat_weekend",
    "feat_session_asia",
    "feat_session_europe",
    "feat_session_us",
})
```

**Célja:** P2 (időindex-alapú) feature-ök, amelyek **nem** tolódnak el 1 barral. Ezek a feature-ök kizárólag az `open_time` timestampből számítódnak — nem tartalmaznak jövőbeli OHLCV adatot, ezért nem kell a lookahead bias elleni lag.

---

## `_apply_t1_lag_pl(lf, p)`

**Célja:** Minden `feat_*` oszlop shift(1) — kivéve `T_MINUS_1_SKIP` tagjai.

**Mechanizmus:** `lf.with_columns([pl.col(c).shift(1).alias(c) for c in feat_cols if c not in T_MINUS_1_SKIP])`

Az eltolás után az első sor összes OHLCV-alapú feature-je `null` lesz.

---

## Indikátor csoportok

### Momentum (`_add_momentum_pl`)

| Feature | Leírás |
|---------|--------|
| `feat_rsi_{n}` | Relative Strength Index (RSI) |
| `feat_roc_{n}` | Rate of Change |
| `feat_stoch_k_{n}`, `feat_stoch_d_{n}` | Stochastic Oscillator |
| `feat_williams_r_{n}` | Williams %R |
| `feat_cci_{n}` | Commodity Channel Index |

---

### Trend (`_add_trend_pl`)

| Feature | Leírás |
|---------|--------|
| `feat_sma_ratio_{n}` | close / SMA(n) |
| `feat_ema_ratio_{n}` | close / EMA(n) |
| `feat_wma_ratio_{n}` | close / WMA(n) |
| `feat_kama_ratio_{n}_{fast}_{slow}` | close / KAMA (Kaufman Adaptive MA) |
| `feat_macd_{fast}_{slow}` | MACD vonal |
| `feat_macd_signal_{fast}_{slow}_{signal}` | MACD szignálvonal |
| `feat_macd_diff` | MACD hisztogram (macd - signal) |
| `feat_adx_{n}` | Average Directional Index |
| `feat_adx_pos_{n}` | +DI irány indikátor |
| `feat_adx_neg_{n}` | -DI irány indikátor |

---

### Volatility (`_add_volatility_pl`)

| Feature | Leírás |
|---------|--------|
| `feat_bb_width_{n}` | Bollinger Band szélesség: `(upper-lower)/close` |
| `feat_bb_position_{n}` | Ár pozíció a sávon belül: `(close-lower)/(upper-lower)` |
| `feat_atr_{n}` | Average True Range (Wilder EWM) |
| `feat_natr_{n}` | Normalized ATR: `atr/close` |
| `feat_hist_vol_{n}` | Historikus volatilitás: rolling std(log returns) |

---

### Volume (`_add_volume_pl`)

| Feature | Leírás |
|---------|--------|
| `feat_volume_sma_{n}` | Forgalom mozgóátlag |
| `feat_volume_ratio_{n}` | Forgalom / SMA(n) forgalom |
| `feat_obv` | On-Balance Volume (kumulatív) |
| `feat_obv_roc_{n}` | OBV Rate of Change |
| `feat_mfi_{n}` | Money Flow Index |
| `feat_ad_line` | Accumulation/Distribution vonal |
| `feat_cmf_{n}` | Chaikin Money Flow |

---

### Price Action (`_add_price_action_pl`)

| Feature | Leírás |
|---------|--------|
| `feat_returns_log` | Log return: `ln(close/prev_close)` |
| `feat_returns_sma_{n}` | Log return rolling SMA |
| `feat_returns_std_{n}` | Log return rolling STD |
| `feat_returns_skew_{n}` | Log return rolling skewness |
| `feat_returns_kurt_{n}` | Log return rolling kurtosis |
| `feat_hml_range` | `(high-low)/close` |
| `feat_ohlc_range` | `(high-low)/((open+close)/2)` |
| `feat_close_position` | `(close-low)/(high-low)` |

---

### Market Structure (`_add_market_structure_pl`)

SR szintek, drawdown, trend slope, regime rank.

---

### Activity (`_add_activity_pl`)

Kereskedési aktivitás metrikák: trades normalizálva, taker flow.

---

### Return Distance (`_add_return_distance_pl`)

Távolság SMA-tól és Bollinger Band-ektől.

---

### Regime Rank (`_add_regime_rank_pl`)

Rolling percentilis rank, `_rolling_rank_arr` numpy helper-rel.

---

### Candle Shape (`_add_candle_shape_pl`)

Gyertya morfológia: body ratio, wick ratio, doji flag.

---

### Trend Slope (`_add_trend_slope_pl`)

Lineáris regresszió slope különböző ablakokra.

---

### Interaction (`_add_interaction_pl`)

Feature kombinációk: RSI × trend, volume × momentum.

---

### Time / Session (`_add_time_session_pl`, `_add_session_relative_pl`)

**P2 feature-ök — T_MINUS_1_SKIP tagjai (nem kapnak t-1 lag-ot):**

| Feature | Leírás |
|---------|--------|
| `feat_hour_sin`, `feat_hour_cos` | Nap körkörös kódolása |
| `feat_dayofweek_sin`, `feat_dayofweek_cos` | Hét napja körkörös kódolása |
| `feat_weekend` | Hétvége flag (0/1) |
| `feat_session_asia`, `feat_session_europe`, `feat_session_us` | Kereskedési szesszió flag |
| `feat_bars_into_session_norm` | Szesszión belüli pozíció (normalizált, 0–1) |
| `feat_day_range_position` | Ár pozíció a napi expanding range-en belül |
| `feat_day_open_return` | Visszatérés a nap nyitóárához képest |
| `feat_weekly_open_return` | Visszatérés a hét nyitóárához képest |

---

### Egyéb csoportok

| Csoport | Kulcs feature-ök |
|---------|--------|
| `_add_gk_volatility_pl` | `feat_parkinson_vol_{10,30,60}`, `feat_gk_vol_{10,30,60}` |
| `_add_autocorr_pl` | `feat_return_autocorr_lag{1,5}_{30,60}`, `feat_variance_ratio_10_60` |
| `_add_drawdown_timing_pl` | `feat_recovery_ratio_{n}`, `feat_max_drawdown_{n}`, `feat_time_since_high_{n}`, `feat_time_since_low_{n}` |
| `_add_pattern_flags_pl` | `feat_doji`, `feat_hammer`, `feat_shooting_star`, `feat_inside_bar`, `feat_outside_bar`, `feat_engulf_bull`, `feat_engulf_bear`, `feat_bull_bars_ratio_{n}` |
| `_add_gap_pl` | `feat_gap_open`, `feat_gap_open_abs_sma_{10,30}` |
| `_add_efficiency_pl` | `feat_efficiency_ratio_{10,30,60}` |
| `_add_sr_levels_pl` | `feat_atr_dist_high_{n}`, `feat_atr_dist_low_{n}`, `feat_prev_session_high_dist`, `feat_prev_session_low_dist` |
| `_add_tail_risk_pl` | `feat_pos_return_mean_{n}`, `feat_neg_return_mean_{n}`, `feat_return_asymmetry_{n}` |
| `_add_extended_accel_pl` | `feat_rsi_delta_{n}`, `feat_roc_delta_{n}`, `feat_return_momentum_delta_{n}` |
| `_add_ichimoku_pl` | `feat_tenkan_ratio`, `feat_kijun_ratio`, `feat_senkou_b_ratio`, `feat_ichimoku_cloud_thickness` |
| `_add_donchian_pl` | `feat_donchian_width_{n}`, `feat_donchian_position_{n}`, `feat_donchian_breakout_{n}` |
| `_add_lr_pl` | `feat_lr_slope_{n}`, `feat_lr_r2_{n}`, `feat_lr_residual_{n}` |

---

## Numpy Helpers

Rolling statisztikák, amelyek nem elérhetők a Polars natív API-ban:

| Függvény | Leírás |
|----------|--------|
| `_rolling_rank_arr(arr, window)` | Rolling percentilis rank |
| `_time_since_high_arr(arr, window)` | Barak száma az utolsó high óta |
| `_time_since_low_arr(arr, window)` | Barak száma az utolsó low óta |
| `_rolling_skew_arr(arr, window)` | Rolling skewness |
| `_rolling_kurt_arr(arr, window)` | Rolling kurtosis |
| `_kama_numpy(arr, fast, slow, er_period)` | Kaufman Adaptive Moving Average |

Ezeket `pl.Series.to_numpy()` + `pl.lit(result)` mintával integrálja a LazyFrame pipeline-ba.

---

## `_clean_features_pl(lf, p)`

Végtelen értékek (`float("inf")`, `float("-inf")`) null-ra cserélése minden `feat_*` oszlopban a `.collect()` előtt. Megelőzi, hogy a modeling layer `inf` értékeket kapjon a DuckDB-ből.




---

<!-- Source: 3000_targets.md -->



﻿# 3000 — Target Layer

A target layer a ChronoQuant ML pipeline label-rétege: a `target` DuckDB tábla tárolja az objektív forward outcome-okat, amelyek alapján a modellek taníthatók és értékelhetők.

---

## Overview

A target layer az `ohlcv` nyers árból számított, jövőbe tekintő (forward-looking) outcome oszlopokból áll. Ezek nem feature-ök — a modell inputjaként soha nem kerülhetnek felhasználásra, kizárólag tanítási label-ként és evaluation benchmark-ként.

```{mermaid}
flowchart TD
  A[ohlcv tábla\nopen_time, close, ...] --> B[sync_targets.py\n_compute_outcome_df]
  B --> C[target tábla\n10 fw60 outcome oszlop]
  C --> D[sampling modul\n00_create_sample.py]
  D --> E[LightGBM tanítás\ntarget col = long_mfe_fw60]
  C --> F[evaluation\nbacktest → sample_oos.parquet]
```

**Aktív target oszlopok:** `long_mfe_fw60`, `short_mfe_fw60` — 60-perces forward logreturn outcome-ok.

**Implementáció:** [`src/data_handling/sync_tables/sync_targets.py`](src/data_handling/sync_tables/sync_targets.py)
**Kód referencia:** [`_doc_/3100_sync_targets.md`](_doc_/3100_sync_targets.md)

---

## Target Oszlopok

| Oszlop | Típus | Definíció | Szerep |
|--------|-------|-----------|--------|
| `close` | DOUBLE | close[t] — referencia close | kontextus |
| `fw60_close` | DOUBLE | close[t+60] — nyers forward close | auxiliary |
| `fw60_max` | DOUBLE | max(close[t+1:t+60]) | auxiliary |
| `fw60_min` | DOUBLE | min(close[t+1:t+60]) | auxiliary |
| `fw60_close_ret` | DOUBLE | close[t+60] / close[t] − 1 | auxiliary |
| `fw60_close_logret` | DOUBLE | log(close[t+60] / close[t]) | auxiliary |
| `fw60_max_ratio` | DOUBLE | max(close[t+1:t+60]) / close[t] | auxiliary |
| `fw60_min_ratio` | DOUBLE | min(close[t+1:t+60]) / close[t] | auxiliary |
| **`long_mfe_fw60`** | **DOUBLE** | **log(max(close[t+1:t+60]) / close[t])** | **LONG TARGET** |
| **`short_mfe_fw60`** | **DOUBLE** | **log(min(close[t+1:t+60]) / close[t])** | **SHORT TARGET** |

Az utolsó **60 sor** minden outcome oszlopban `NULL` — nincs elegendő jövőbeli adat a teljes horizont kiszámításához.

---

## Üzleti és módszertani háttér

### Miért kritikus ez a lépés?

A target layer dönti el, hogy a modell **mit tanul meg előrejelezni**. Ha a target definíció torz (pl. jövőbeli eloszlásból vett küszöb éget bele a múltbeli labelbe), a cross-validation score nem tükrözi a valós produkciós teljesítményt. Ha a forward window helytelen (pl. az aktuális bar beleszámít), az in-sample performance irreálisan magas lesz.

A target layer ezért alapvetően meghatározza a modell megbízhatóságát és az eredmények interpretálhatóságát.

### Miért ezt a megközelítést?

```{mermaid}
flowchart LR
  Q[Target stratégia] --> A[Full-history quantile bináris label\nNO: target-definition leakage\nNO: információvesztés\nNO: rezsimfüggő torzítás]
  Q --> B[Folytonos fw60 logreturn outcome\nOK: objektív forward measurement\nOK: nincs percentilis küszöb\nOK: Választott]
  Q --> C[Triple-barrier label\nWARN: jobb MFE/MAE szétválasztás\nde komplex konfiguráció]
  Q --> D[Fold-specifikus bináris label\nWARN: leakage-mentes binarizálás\nde kompatibilitás elvész]
```

| Megközelítés | Előny | Hátrány | Státusz |
|---|---|---|---|
| Folytonos fw60 logreturn (jelenlegi) | Nincs percentilis-torzítás, magnitude megmarad, flexibilis | Regresszor szükséges, binary baseline elvész | ✅ Választott (epic-011) |
| Full-history quantile bináris | Egyszerű classifier, stabil threshold | Target-definition leakage, rezsimfüggő torzítás, információvesztés | ❌ Eltávolítva — legacy |
| Fold-specifikus quantile bináris | Leakage-mentes binarizálás | Minden foldban különböző label → összehasonlíthatatlan metrikák | ⚠️ Fontolóra vehető derived label-ként |
| Triple-barrier | MFE + MAE egyszerre kezel, stop-loss implicit | Konfiguráció érzékeny, training instabilabb | ⚠️ Jövőbeli kísérlethez |
| Quantile regression target | Tail opportunity fókusz | Nem standard loss, nehezebben interpretálható | ⚠️ Objektív altarget |

### Miért váltottunk binárisról folytonos targetre?

A korábbi rendszerben (`epic-011` előtt) két bináris label létezett:

```
trg_long = (future_max_return >= teljes history quantile küszöb)
trg_short = (future_min_return <= teljes history quantile küszöb)
```

Ez három strukturális problémát okozott:

```{mermaid}
graph TD
  P1[Target-definition leakage\nA quantile küszöb a teljes historyból\nszámolódik beleértve a validációs\nidőszak utáni adatokat is]
  P2[Rezsimfüggő torzítás\nKésőbbi volatilis időszak\nmagasabb quantile küszöböt okoz\nvisszamenőleg kevesebb pozitív labelt]
  P3[Információvesztés\nfuture_max = 0.91% és 4.50%\nmindkettő trg=1 ha a küszöb = 0.90%]
  P1 & P2 & P3 --> EFFECT[Torzult CV score\nTorzult feature importance\nNem production-like threshold]
```

**Target-definition leakage:** Ha a quantile küszöb a teljes 2025–2026-os historyból számolódik, akkor a 2025 Q2 validációs fold targetjei már 2026-os eloszlásinformációt tartalmaznak a label definíciójában. Ez nem klasszikus feature leakage, hanem *target-definition leakage*.

**Megoldás:** Objektív, küszöb-mentes forward outcome — `log(future_max / close[t])`. Az outcome a tényleges piacmozgást méri, percentilis policy és binarizálás nélkül.

### MFE és MAE: miért kell és hogyan működik?

```{mermaid}
graph LR
  LONG["Long pozíció t-től"] --> MFE_L["long_mfe_fw60\nMaximum Favorable Excursion\n= log(max_close / close[t])\n→ pozitív ha ár felmegy"]
  LONG --> MAE_L["long_mae_fw60\n= log(min_close / close[t])\n→ negatív ha ár lemegy\n(forward audit, nem primáris target)"]
  SHORT["Short pozíció t-től"] --> MFE_S["short_mfe_fw60\nMaximum Favorable Excursion\n= log(min_close / close[t])\n→ negatív ha ár lemegy (short kedvező)"]
  SHORT --> MAE_S["short_mae_fw60\n= log(max_close / close[t])\n→ pozitív ha ár felmegy (short ellen)"]
```

- `long_mfe_fw60` **pozitív** → az ár felfelé ment → long kedvező
- `short_mfe_fw60` **negatív** → az ár lefelé ment → short kedvező
- `long_mae_fw60` = `short_mfe_fw60` (azonos numerikus érték, ellentétes szemantika)

**Szabály:** A modell `long_mfe_fw60` targetre tanul; a `short_mfe_fw60` a másik modellé. A MAE értékek nem elsődleges targetok, de az evaluation és adverse move audit során kötelezően ellenőrizendők.

### Forward window szemantika: miért zárjuk ki az aktuális bart?

```{mermaid}
flowchart LR
  T["t bar (aktuális)"] --> T1["t+1"]
  T1 --> DOTS["..."]
  DOTS --> T60["t+60"]
  T -->|KIZÁRVA| FW["forward window\nt+1 .. t+60"]
  FW -->|max| MAX["fw60_max"]
  FW -->|min| MIN["fw60_min"]
  FW -->|close| CL["fw60_close"]
```

SQL invariáns: `ROWS BETWEEN 1 FOLLOWING AND 60 FOLLOWING`

Az aktuális bar (`t`) kizárása azért kötelező, mert a predikció a `t` bar zárásakor készül — a `t+1` bar nyitásán kerül végrehajtásra. Ha `t` benne lenne a forward ablakban, az outcome egy részben már ismert értéket tükrözne.

**NULL tail:** Az utolsó 60 sor minden outcome oszlopban `NULL` — nincs 60 jövőbeli bar. **Soha ne impute-old `0`-ra** — a null sor nem megfigyelt, nem negatív esemény.

### Logreturn vs. simple return: miért logaritmikus?

| Mérőszám | Képlet | Jellemző |
|---|---|---|
| Simple return | (max − close) / close | Aszimmetrikus: +10% és −10% nem összehasonlítható |
| Logreturn | log(max / close) | Additív, szimmetrikus; kis értékeknél ≈ simple return |

A logreturn additív természete lehetővé teszi, hogy multi-period outlookokat összeadással aggregáljuk, és a long/short side értékei közvetlenül összehasonlíthatók. Kis moves esetén (<2%) a két mérőszám numerikusan közel esik egymáshoz, tehát a váltás nem rontja az értelmezhetőséget.

### Paraméter alapértékek és indoklásuk

| Paraméter | Érték | Indoklás |
|---|---|---|
| Forward horizon | `60` bar | 60 perces opportunity ablak; egyezik a trading stratégia max hold time felfogásával |
| Window logika | `t+1..t+60` | Aktuális bar kizárva; forward window pontosan 60 ismert future bar |
| NULL küszöb | `fw_bar_count >= 60` | Csak teljes forward ablakkal rendelkező sorok kapnak értéket |
| Logreturn alap | természetes logaritmus (`LN`) | DuckDB `LN()` — szimmetrikus, additív |
| Elsődleges long target | `long_mfe_fw60` | MFE = maximum favorable excursion — a legjobb elérhető long opportunity |
| Elsődleges short target | `short_mfe_fw60` | MFE short oldalon = log(min/close) — a legjobb elérhető short opportunity |
| Rebuild policy | teljes DELETE+INSERT | Minden `sync_targets()` hívás teljes újraszámítást végez — idempotens |

### Ismert kockázatok és korlátok

| Kockázat | Tünet | Mitigáció |
|---|---|---|
| NULL tail torzítás | Ha a sampling elszedi az utolsó 60 sort és `0`-ra imputálja, a modell hamis negatívokat lát | `audit_feature_table()` ellenőrzi; null sorok droppolva a dataset loaderben |
| Rezsimváltás eltérő target eloszlást okoz | Alacsony volatilitású periódusban a `long_mfe_fw60` p90 kisebb mint magas volatilitású periódusban | Expanding window CV kezeli; az expanding train hatókör követi a rezsimeket |
| `fw60_max` és `fw60_min` szimmetriája | A két outcome ugyanazon a skálán van, de long és short értelmezésük ellentétes | Dokumentált szimmetria: `long_mae_fw60` numerikusan = `short_mfe_fw60` |
| Kis log value értelmezése | `long_mfe_fw60 = 0.003` → `exp(0.003) − 1 ≈ 0.30%` — konfúzió a magnitude körül | Model card-on és reporting-ban mindig % formában is feltüntetni |
| Legacy target referencia örökség | Régi docs a bináris `trg_*` targetekre hivatkoznak | Elavultként kezelni; ground truth: `_doc_/3100_sync_targets.md` és a forráskód |

### Validációs checklist

- [ ] A `target` tábla utolsó 60 sora minden fw60 outcome oszlopban `NULL`
- [ ] `ROWS BETWEEN 1 FOLLOWING AND 60 FOLLOWING` — az aktuális bar (`t`) nem szerepel a forward ablakban
- [ ] `long_mfe_fw60` = `log(fw60_max / close)` — numerikusan ellenőrzött determinisztikus teszttel
- [ ] `short_mfe_fw60` = `log(fw60_min / close)` — numerikusan ellenőrzött determinisztikus teszttel
- [ ] `long_mfe_fw60` és `short_mfe_fw60` csak DOUBLE `NULL`, soha `0.0` — nem impute-olt
- [ ] A `target` tábla nem tartalmaz legacy `trg_*` bináris oszlopot
- [ ] `sync_targets()` teljes futás után: `computed_from`, `computed_to`, `computed_at` frissítve a `solusdt.json` metaadatban
- [ ] Dataset loader: null target sorok droppolva (`dropna` a target col alapján) — nincs `0` imputation




---

<!-- Source: 3100_sync_targets.md -->



# sync_targets.py — fw60 Forward Outcome Számítás

`src/database/sync_tables/sync_targets.py`

Minden `sync_targets` hívás teljes rebuild — DELETE+INSERT az összes tárolt OHLCV bar alapján.
A régi bináris `trg_*` target rendszer eltávolítva (epic-011).

---

## `sync_targets(asset_id)`

**Célja:** A `target` tábla teljes újraépítése az összes `ohlcv` bar alapján, 10 fw60 forward outcome oszloppal.

**Paraméterek:**

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `asset_id` | `str \| None` | Asset azonosító (config default ha `None`) |

---

## Belső folyamat

```{mermaid}
sequenceDiagram
    participant SYNC as sync_targets()
    participant DB as DuckDB
    participant META as solusdt.json

    SYNC->>DB: get_connection(db_path)
    SYNC->>DB: _compute_outcome_df(conn, horizon=60)
    DB-->>SYNC: pl.DataFrame (open_time, close, fw60_* és mfe oszlopok)
    SYNC->>DB: insert_target(conn, df)
    SYNC->>META: _update_metadata_outcomes(computed_from, computed_to)
```

---

## fw60 Forward Outcome Oszlopok

| Oszlop | Típus | Definíció |
|--------|-------|-----------|
| `close` | DOUBLE | close[t] — jelenlegi bar close ára |
| `fw60_close` | DOUBLE | close[t+60] — nyers forward close |
| `fw60_max` | DOUBLE | max(close[t+1:t+60]) — nyers max ár |
| `fw60_min` | DOUBLE | min(close[t+1:t+60]) — nyers min ár |
| `fw60_close_ret` | DOUBLE | close[t+60] / close[t] - 1 |
| `fw60_close_logret` | DOUBLE | log(close[t+60] / close[t]) |
| `fw60_max_ratio` | DOUBLE | max(close[t+1:t+60]) / close[t] |
| `fw60_min_ratio` | DOUBLE | min(close[t+1:t+60]) / close[t] |
| `long_mfe_fw60` | DOUBLE | log(max(close[t+1:t+60]) / close[t]) — **LONG TARGET** |
| `short_mfe_fw60` | DOUBLE | log(min(close[t+1:t+60]) / close[t]) — **SHORT TARGET** |

**Szemantika:**
- `long_mfe_fw60` pozitív → az ár felment → long kedvező
- `short_mfe_fw60` negatív → az ár lement → short kedvező

---

## `_compute_outcome_df(conn, horizon=60)`

**Célja:** Az összes fw60 outcome oszlop kiszámítása DuckDB window SQL-lel.

**Visszatérési érték:** `pl.DataFrame` — `open_time`, `close`, és mind a 10 fw60 oszlop.

---

## `_TARGET_SQL` — A core SQL sablon

```sql
WITH ohlcv_ordered AS (
    SELECT open_time, close FROM ohlcv ORDER BY open_time
),
forward_window AS (
    SELECT
        open_time,
        close,
        LEAD(close, {horizon}) OVER (ORDER BY open_time) AS fw_close,
        MAX(close) OVER (
            ORDER BY open_time
            ROWS BETWEEN 1 FOLLOWING AND {horizon} FOLLOWING
        ) AS fw_max,
        MIN(close) OVER (
            ORDER BY open_time
            ROWS BETWEEN 1 FOLLOWING AND {horizon} FOLLOWING
        ) AS fw_min,
        COUNT(*) OVER (
            ORDER BY open_time
            ROWS BETWEEN 1 FOLLOWING AND {horizon} FOLLOWING
        ) AS fw_bar_count
    FROM ohlcv_ordered
)
SELECT
    open_time,
    close,
    CASE WHEN fw_bar_count >= {horizon} THEN fw_close           ELSE NULL END AS fw60_close,
    ...
    CASE WHEN fw_bar_count >= {horizon} AND close > 0
         THEN LN(fw_max / close)                               ELSE NULL END AS long_mfe_fw60,
    CASE WHEN fw_bar_count >= {horizon} AND close > 0
         THEN LN(fw_min / close)                               ELSE NULL END AS short_mfe_fw60
FROM forward_window
ORDER BY open_time
```

**Kritikus invariáns:** `ROWS BETWEEN 1 FOLLOWING AND {horizon} FOLLOWING`
- Az aktuális bar (`t`) **nem szerepel** a forward window-ban
- Az utolsó `horizon=60` sor `NULL`-t kap (nincs elegendő jövőbeli adat)

---

## NULL sorok

Az utolsó `60` sor minden fw60 oszlopban `NULL` — nincs elegendő jövőbeli adat.

```
                    ┌───────────────────────────────┐
OUTCOME OSZLOPOK:  │ értékek (DOUBLE) │ NULL (60 sor) │
                   └───────────────────────────────┘
                         ↑ fw_bar_count >= 60         ↑ fw_bar_count < 60
```

---

## `_update_metadata_outcomes(...)`

**Célja:** fw60 outcome definíciók és számítási időtartomány perzisztálása audit céljából.

**Kimeneti fájl:** `database/<asset_id>/<asset_id>.json`

**Tartalom:**
```json
{
  "target_outcomes": {
    "fw60": {
      "horizon": 60,
      "window": "t+1..t+60",
      "columns": {
        "close":             "close[t] — reference bar close",
        "fw60_close":        "close[t+60] — raw forward close",
        "fw60_max":          "max(close[t+1:t+60]) — raw max price",
        "fw60_min":          "min(close[t+1:t+60]) — raw min price",
        "fw60_close_ret":    "close[t+60] / close[t] - 1",
        "fw60_close_logret": "log(close[t+60] / close[t])",
        "fw60_max_ratio":    "max(close[t+1:t+60]) / close[t]",
        "fw60_min_ratio":    "min(close[t+1:t+60]) / close[t]",
        "long_mfe_fw60":     "log(max(close[t+1:t+60]) / close[t]) — LONG TARGET",
        "short_mfe_fw60":    "log(min(close[t+1:t+60]) / close[t]) — SHORT TARGET"
      },
      "null_tail_rows": 60,
      "computed_from": "2020-09-14 07:00:00",
      "computed_to":   "2026-06-17 12:00:00",
      "computed_at":   "2026-06-17 14:30:00"
    }
  }
}
```

---

## Régi bináris target rendszer (eltávolítva)

Az epic-011 előtt a target tábla két bináris oszlopot tartalmazott:
- `trg_* BOOLEAN` — legacy quantile-bináris target oszlopok (eltávolítva)

Ezek eltávolítva. Az `ensure_tables` migráció automatikusan felváltja a régi sémát az újra.




---

<!-- Source: 4000_quant_train.md -->



﻿# 4000 — quant_train Table

A `quant_train` tábla a modeling pipeline egyetlen stabil belépési pontja: a `feat_ohlcv_quant` és a `target` tábla INNER JOIN-jából épül fel, és kizárólag ad-hoc rebuild-del frissül — soha nem a live sync pipeline részént.

---

## Overview

```{mermaid}
flowchart TD
  F[feat_ohlcv_quant\nopen_time + feat_* + metaadatok]
  T[target\nopen_time + fw60 outcome-ok]
  QT[(quant_train\nopen_time + feat_* + long_mfe_fw60 + short_mfe_fw60)]
  FE[01_feature_engineering.ipynb\nfeature szelekció + minőség]
  SM[00_create_sample.py\nyearly random-hour sampling]
  LGBM[03_fit_model.py\nLightGBM tanítás]

  F -->|INNER JOIN\non open_time\nNULL target kizárva| QT
  T -->|INNER JOIN\non open_time| QT
  QT --> FE
  QT --> SM
  SM --> LGBM
```

A `quant_train` kizárja az `available_ts`, `lookback_end_ts`, `close`, és minden auxiliary fw60 oszlopot — csak a `feat_*` prefix és a két elsődleges target kerül bele.

**Kód referencia:** [`_doc_/4100_quant_train.md`](_doc_/4100_quant_train.md)
**Implementáció:** [`src/data_handling/sync_tables/sync_quant_train.py`](src/data_handling/sync_tables/sync_quant_train.py)
**CLI:** `uv run python src/data_handling/03_build_quant_train.py [--start YYYY-MM-DD] [--end YYYY-MM-DD]`

---

## Üzleti és módszertani háttér

### Miért kritikus ez a lépés?

A `quant_train` az ML pipeline stabil handoff-pontja az adat-réteg és a modellező réteg között. Ha ez a tábla sérült (pl. NULL targeteket tartalmaz, vagy hiányzó feature oszlopai vannak), az összes downstream munka — feature engineering, sampling, LightGBM tanítás — helytelen vagy félrevezető eredményt ad, és a hiba a későbbi artifactokban jelenik meg, nem a tábla építésekor.

A NULL target sorok kizárása ezen a szinten kritikus: ha az utolsó 60 sor (ahol nincs elegendő forward data) bekerülne a tanítóba, a modell `0`-ként tanulná meg ezeket, holott ezek ismeretlen állapotú sorok.

### Miért ezt a megközelítést?

```{mermaid}
flowchart LR
  Q[quant_train felépítése] --> A[Live sync pipeline\nNO: folyamatosan frissülő join\nNO: tanítás alatt változhat\nNO: nem reprodukálható]
  Q --> B[Ad-hoc rebuild DuckDB-ben\nOK: tanítás előtt lefuttatott\nOK: teljes újraépítés = determinisztikus\nOK: Választott]
  Q --> C[Parquet snapshot\nWARN: reprodukálható, de nehéz frissíteni\nWARN: inkompatibilis a live query-vel]
```

| Megközelítés | Előny | Hátrány | Státusz |
|---|---|---|---|
| Ad-hoc DuckDB rebuild (jelenlegi) | Determinisztikus, lekérdezhető, frissíthető range-rebuild-del | Manuálisan kell futtatni tanítás előtt | ✅ Választott |
| Live sync pipeline | Mindig aktuális | Tanítás alatt változhat → nem reprodukálható; overhead minden sync-nél | ❌ Elvetett — pipeline inkonzisztencia kockázata |
| Parquet snapshot | Reprodukálható snapshot | Nem frissíthető inkrementálisan; DuckDB query nem tud rá csatlakozni | ⚠️ Csak archív célra |

### INNER JOIN szemantika: miért nem LEFT JOIN?

```{mermaid}
graph TD
  F[feat_ohlcv_quant\n~összes perc] --> J{INNER JOIN\non open_time}
  T[target\nnull az utolsó 60 sorban] --> J
  J --> QT[quant_train\nnull-free sorok]
  J -->|kizárva| NL[null target sorok\nazaz az utolsó 60 perc]
```

Az INNER JOIN dupla védelmet nyújt:
1. Csak azok a sorok kerülnek be, ahol a target tábla row-ja létezik
2. A `WHERE long_mfe_fw60 IS NOT NULL AND short_mfe_fw60 IS NOT NULL` feltétel kizárja a NULL forward outcome-okat

LEFT JOIN esetén NULL targetű sorok bekerülnének, és a dataset loadernek minden betöltésnél kellene szűrni — ez hibalehetőség, és a NULL-ok véletlenül 0-vá imputálódhatnának.

### Full vs. Range rebuild: mikor melyik?

```{mermaid}
flowchart TD
  A{Rebuild típus?} -->|"--start és --end\nnem adott meg"| B[Full rebuild\nCREATE OR REPLACE TABLE\ndeterminisztikus]
  A -->|"--start és --end\nmegadva"| C[Range rebuild\nDELETE + INSERT\naz adott ablakra]
  B --> E[teljes tábla frissítve]
  C --> E
  E -->|mindkét mód| F[idempotens eredmény]
```

| Mód | Mikor | SQL |
|-----|-------|-----|
| **Full rebuild** | Első feltöltés, teljes újraépítés, schema változás után | `CREATE OR REPLACE TABLE quant_train AS SELECT ...` |
| **Range rebuild** | Inkrementális frissítés (pl. új OHLCV adat érkezett) | `DELETE ... WHERE open_time BETWEEN start AND end` + `INSERT INTO ...` |

Mindkét mód **idempotens** — biztonságos többszöri futtatásra.

### Kizárt oszlopok: miért nem kerül be az összes?

A `quant_train` szándékosan szűk scope-ú:

| Kizárt oszlop | Forrás | Miért kizárt |
|---|---|---|
| `close` | feat_ohlcv_quant | Nem feature — árszint, non-stacionárius; nem kell a modellnek |
| `available_ts`, `lookback_end_ts` | feat_ohlcv_quant | Metadata — sampling és pipeline kontroll, nem ML input |
| `fw60_close`, `fw60_max`, `fw60_min`, stb. | target | Auxiliary outcome-ok — csak `long_mfe_fw60` és `short_mfe_fw60` az aktív target |
| `long_pred`, `short_pred` | predictions | Predikciók nem kerülhetnek vissza tanítóba — feedback loop |

### Paraméter alapértékek és indoklásuk

| Paraméter | Érték | Indoklás |
|---|---|---|
| Target oszlopok | `long_mfe_fw60`, `short_mfe_fw60` | Aktív fw60 logreturn target páros; bővítés esetén új rebuild szükséges |
| NULL szűrés | `IS NOT NULL` mindkét targetre | Védi a modellt a forward-edge soroktól; soha ne imputálj 0-val |
| Rebuild mód default | Full rebuild | Biztonságos és determinisztikus; range rebuild csak ha tanítás-előtti inkrementális frissítés szükséges |
| DuckDB táblanév | `quant_train` | Fix — a downstream pipeline (sampling, training) erre a névre hivatkozik |

### Ismert kockázatok és korlátok

| Kockázat | Tünet | Mitigáció |
|---|---|---|
| Stale tábla (nem rebuild-elt tanítás előtt) | Hiányzó legfrissebb adatok; model training régi datán tanul | Kötelező futtatni a `03_build_quant_train.py`-t tanítás előtt — checklist pontja |
| Schema drift (új feature oszlop `feat_ohlcv_quant`-ban) | Hiányzó feature oszlop a `quant_train`-ben | Full rebuild kötelező ha `sync_features` megváltozott |
| Range rebuild időablak túl szűk | Overlap az INNER JOIN-nal, NULL sorok maradnak | Rebuild a target sync teljes tartományán futtasd, ne csak az OHLCV tartományán |
| `feat_ohlcv_quant` és `target` eltérő adathatárok | Kevesebb sor a quant_train-ben mint várható | Ellenőrizd mindkét tábla `MAX(open_time)`-ját; futtasd mindkét sync-et az újraépítés előtt |
| `quant_train` vs yearly sample artifact konfúzió | A sample parquet csak `open_time`, `segment`, `fold_id` és target adatot ad; a feature mátrix DuckDB-ből töltődik vissza | Training célra ne feltételezz materializált sample DuckDB táblát |

### Validációs checklist

- [ ] `quant_train` tábla létezik: `SHOW TABLES` tartalmazza
- [ ] `SELECT COUNT(*) FROM quant_train WHERE long_mfe_fw60 IS NULL` → 0 (nincs NULL target)
- [ ] `SELECT COUNT(*) FROM quant_train WHERE short_mfe_fw60 IS NULL` → 0
- [ ] `SELECT MAX(open_time) FROM quant_train` ≥ `SELECT MAX(open_time) FROM target WHERE long_mfe_fw60 IS NOT NULL` − 1 perc
- [ ] Oszlopok: `open_time`, összes `feat_*`, `long_mfe_fw60`, `short_mfe_fw60` — semmi más (pl. `close`, `available_ts`, `trg_*`)
- [ ] Rebuild futott le a legutóbbi `sync_features` és `sync_targets` után
- [ ] `DESCRIBE quant_train` feat_ oszlopszáma megegyezik `feat_ohlcv_quant`-éval

---

## Alfejezetek

| Szám | Fájl | Tartalom |
|------|------|----------|
| 4100 | [4100_quant_train.md](4100_quant_train.md) | Részletes séma, rebuild szemantika, CLI referencia |




---

<!-- Source: 4100_quant_train.md -->



﻿# 4100 — quant_train Table

Model-ready join tábla: `feat_ohlcv_quant` + `target` → tanítási adatforrás.

---

## Áttekintés

A `quant_train` tábla az ML pipeline egyetlen stabil belépési pontja. Feature engineering, sampling és LightGBM tanítás ebből dolgozik — nem a nyers `feat_ohlcv_quant` + `target` join-ból.

```{mermaid}
flowchart LR
  A[feat_ohlcv_quant\nopen_time + feat_*] -->|INNER JOIN\non open_time| C[quant_train\nopen_time + feat_* +\nlong_mfe_fw60 +\nshort_mfe_fw60]
  B[target\nopen_time + fw60 outcomes] -->|INNER JOIN\non open_time| C
  C --> D[00_create_sample.py\nyearly random-hour\nsegment assign]
  D --> E[database/.../samples/<sample_id>/\nmetadata.json + audit.json +\nsample_train_valid.parquet]
  E --> F[03_fit_model.py\nLightGBM]
```

**NULL target policy:** Az INNER JOIN automatikusan kizárja azokat a sorokat, ahol `long_mfe_fw60 IS NULL OR short_mfe_fw60 IS NULL`. Ezek a sorok sosem kerülnek be a `quant_train`-be.

**Nem pipeline:** A `quant_train` nem része a live sync pipeline-nak (`02_sync_pipeline.py`). Kizárólag ad-hoc rebuild — tanítás előtt futtatandó.

---

## Yearly sample artifact handoff

A `00_create_sample.py` (`create_yearly_sample`) az éves mintavétel után statikus
parquet/json artifactokat ír a `database/<asset>/samples/<sample_id>/` könyvtárba.
Az aktív yearly sampling pipeline nem hoz létre `sample_<id>` DuckDB táblát.

**Könyvtár példa:** `database/solusdt/samples/solusdt_fw60_yearly_2024/`

**`sample_train_valid.parquet` oszlopok:**

| Oszlop | Típus | Leírás |
|--------|-------|--------|
| `open_time` | `TIMESTAMP` | Bar nyitási ideje |
| `segment` | `VARCHAR` | `train`, `valid`, vagy `purge` |
| `fold_id` | `BIGINT \| NULL` | 0-alapú index a validációs héthez; NULL ha nem valid sor |
| `long_mfe_fw60` | `DOUBLE` | Long target |
| `short_mfe_fw60` | `DOUBLE` | Short target |

Feature oszlopok nem kerülnek a sample parquetba; a modeling lépések a szükséges
feature-öket DuckDB-ből töltik vissza a sample `open_time` soraival joinolva.

**Artefaktok szerepe:**
- `sample_train_valid.parquet`: elsődleges yearly sample handoff — ebből jönnek az `open_time`, `segment`, `fold_id` és target sorok
- `metadata.json`: konfigurációs és auditálási metaadatok; tartalmazza a `selected_valid_weeks` listát
- `audit.json`: adatminőségi metrikák

---

## Séma

| Oszlop | Típus | Forrás | Leírás |
|--------|-------|--------|--------|
| `open_time` | `TIMESTAMP` (PK) | `feat_ohlcv_quant` | Bar nyitási ideje, UTC. INNER JOIN garantálja az egyediséget. |
| `feat_*` | `DOUBLE` | `feat_ohlcv_quant` | Összes `feat_` prefixű feature oszlop. T-1 lag már alkalmazva. |
| `long_mfe_fw60` | `DOUBLE` | `target` | `log(max_price_fw60 / close[t])` — fw60 long outcome. |
| `short_mfe_fw60` | `DOUBLE` | `target` | `log(min_price_fw60 / close[t])` — fw60 short outcome. |

**Kizárt oszlopok:** `close`, `available_ts`, `lookback_end_ts` (feat táblából), `fw60_close`, `fw60_max`, `fw60_min` és egyéb fw60 oszlopok (target táblából), `long_pred`, `short_pred` (predictions tábla).

**Legacy naming:** A régi `trg_*` boolean elnevezés NEM szerepel ebben a rétegben. A target oszlopok kizárólag `long_mfe_fw60` és `short_mfe_fw60`.

---

## Rebuild szemantika

```{mermaid}
flowchart TD
  A{rebuild típus?} -->|full| B[CREATE OR REPLACE TABLE quant_train\nAS SELECT ...]
  A -->|range| C[DELETE FROM quant_train\nWHERE open_time BETWEEN start AND end]
  C --> D[INSERT INTO quant_train\nSELECT ... WHERE open_time BETWEEN start AND end]
  B --> E[determinisztikus eredmény]
  D --> E
```

| Mód | SQL | Mikor |
|-----|-----|-------|
| **Full rebuild** | `CREATE OR REPLACE TABLE quant_train AS SELECT ...` | Kezdeti feltöltés, teljes újraépítés |
| **Range rebuild** | `DELETE + INSERT` a megadott `open_time` ablakra | Inkrementális frissítés |

Mindkét mód **idempotens** — többszöri futtatás azonos eredményt ad.

---

## CLI

```powershell
# Full rebuild (alapértelmezett)
uv run python src/data_handling/03_build_quant_train.py

# Range rebuild
uv run python src/data_handling/03_build_quant_train.py --start 2024-01-01 --end 2024-12-31
```

---

## Implementáció

| Fájl | Szerepe |
|------|---------|
| [`src/data_handling/store/duckdb_store.py`](src/data_handling/store/duckdb_store.py) | `rebuild_quant_train(conn, start_time, end_time)` — core rebuild logika |
| [`src/data_handling/sync_tables/sync_quant_train.py`](src/data_handling/sync_tables/sync_quant_train.py) | `sync_quant_train(asset_id, start_time, end_time)` — asset-szintű wrapper |
| [`src/data_handling/03_build_quant_train.py`](src/data_handling/03_build_quant_train.py) | Standalone CLI |

---

## Kapcsolódó dokumentumok

- [`_doc_/1000_database.md`](_doc_/1000_database.md) — teljes DuckDB séma áttekintő
- [`_doc_/1110_duckdb_store.md`](_doc_/1110_duckdb_store.md) — store réteg
- [`_doc_/3100_sync_targets.md`](_doc_/3100_sync_targets.md) — target tábla és fw60 outcome-ok
- [`_doc_/3000_targets.md`](_doc_/3000_targets.md) — target layer módszertani háttér
- [`_doc_/5010_sampling_yearly.md`](_doc_/5010_sampling_yearly.md) — aktív yearly sampling metodológia




---

<!-- Source: 5000_modelling.md -->



﻿# 5000 — Modeling

A modeling domain a ChronoQuant ML pipeline szíve: nyers OHLCV adatokból
folytonos előrejelzéseket állít elő LightGBM regresszorokkal (fw60 MFE target).

---

## Overview

A pipeline öt lépésből áll: feature számítás → sample definíció → modell tanítás →
predikció szinkronizálás → kereskedési jelzések. Minden lépés idempotens és
önállóan újrafuttatható.

```{mermaid}
flowchart TD
  A[ohlcv táblázat] --> B[feat_ohlcv_quant]
  B --> C[00_create_sample.py\nsampling modul]
  C --> D[database/solusdt/samples/]
  D --> E[01_train_model.py\nlightgbm_model]
  E --> F[models/ artifact]
  F --> G[sync_predictions\npredict / predict_proba]
  G --> H[predictions táblázat]
  H --> I[trading/strategy.py\njelzések]
```

---

## Aktív modellek konfiguráció

### Éves modellek (naming convention v4)

Model ID minta: `lgbm_{asset}_{direction}_fw{horizon}_{year}`

| Model ID minta | Irány | Target | Évek |
|----------------|-------|--------|------|
| `lgbm_solusdt_l_fw60_{year}` | Long | `long_mfe_fw60` | 2021-2025 |
| `lgbm_solusdt_s_fw60_{year}` | Short | `short_mfe_fw60` | 2021-2025 |

- **Target szemantika:** `fw60` = 60-perces forward ablak (`t+1..t+60`); `long_mfe_fw60` = log(max future close / close[t]); `short_mfe_fw60` = log(min future close / close[t]). Folytonos regressziós target — nincs percentilis küszöb, nincs binarizálás.
- **Feature prefix:** `feat_` | **Target oszlopok:** `long_mfe_fw60`, `short_mfe_fw60`
- **t-1 lag kötelező** minden feature-ön tanítás előtt

---

## Fejezetek

| Szám | Fájl | Tartalom | Szint | Állapot |
|------|------|----------|-------|---------|
| 5010 | [5010_sampling_yearly.md](5010_sampling_yearly.md) | Yearly random-hour sampling — teljes metodológia | X100 | kész |
| 5100 | [5100_sampling_config.md](5100_sampling_config.md) | YearlySamplingConfig dataclass | X110 | kész |
| 5200 | [5200_sampling_artifacts.md](5200_sampling_artifacts.md) | write_yearly_artifacts / load_yearly_sample | X110 | kész |
| 5300 | [5300_create_sample.md](5300_create_sample.md) | create_yearly_sample orchestrator + CLI | X110 | kész |
| 2000 | [2000_features.md](2000_features.md) | Feature layer metodológia (208 feat, 25 csoport) | X100 | kész |
| 2010 | [2010_feature_engineering.md](2010_feature_engineering.md) | Feature selection — quality, target relation, redundancy, stability | X100 | kész |
| 3000 | [3000_targets.md](3000_targets.md) | Target layer metodológia (fw60 logreturn outcome-ok) | X100 | kész |
| 4000 | [4000_quant_train.md](4000_quant_train.md) | quant_train table — INNER JOIN handoff, rebuild szemantika | X100 | kész |
| — | — | LightGBM model (training, CV, hyperparameter search) | X100 | tervezett |
| — | — | Evaluation / backtest | X100 | tervezett |
| — | — | Elliott waves (kutatás, izolált) | X100 | tervezett |
| 5400 | [5400_sampling.md](5400_sampling.md) | **ARCHÍV** — expanding window CV (nem aktív) | archív | archív |
| 5410 | [5410_sampling_splits.md](5410_sampling_splits.md) | **ARCHÍV** — expanding window splits | archív | archív |
| 5420 | [5420_sampling_audit.md](5420_sampling_audit.md) | **ARCHÍV** — feature table audit | archív | archív |




---

<!-- Source: 5010_sampling_yearly.md -->



﻿# 5010 — Yearly Random-Hour Sampling

Az éves, random-óra-alapú sampling stratégia lényege: egy naptári évre pontosan egy
random percet választ óránként (~8 760 sor/év), majd 12 hónaponkénti validációs hetet
jelöl ki, és purge-ablakkal választja el a train és valid szegmenseket.

---

## Overview

```{mermaid}
flowchart TD
  QT[(DuckDB\nquant_train)] --> CS[create_yearly_sample\ncreate_sample.py]
  CS --> A[select_hourly_observations\nyearly_sampler.py]
  A --> B[select_monthly_validation_weeks\nyearly_sampler.py]
  B --> C[assign_segments\nyearly_sampler.py]
  C --> D[write_yearly_artifacts\nartifacts.py]
  D --> E[database/asset/samples/id/\nmetadata.json\naudit.json\nsample_train_valid.parquet]
```

A pipeline input-ja a `quant_train` tábla (feat_* + target oszlopok, NULL target sorok
kizárva); kimenetei:
- `database/<asset>/samples/<sample_id>/metadata.json`, `audit.json`, `sample_train_valid.parquet`

---

## Üzleti és módszertani háttér

### Miért kritikus ez a lépés?

A yearly sampling dönt arról, hogy melyik percek kerülnek tanítóba, melyik
validálásba, és melyik kap purge-jelölést. Egy rossz split → információszivárgás
train→valid irányba → a model jónak látszik backtesten, de élesben alulteljesít.

Az éves granularitás egy további célt is szolgál: minden naptári év egy önálló
megfigyelési egységként értékelhető, így a modell éven belüli stabilitása és az
évek közötti rezsimváltás hatása külön mérhető.

---

### Miért ezt a megközelítést?

```{mermaid}
flowchart LR
  Q[Sampling stratégia] --> S1[Expanding window CV\nNO: szivárgás évhatáron\nNO: nem szezonálisan izolált\nOK: max historikus adat]
  Q --> S2[Yearly random-hour\nOK: éves izoláció\nOK: random hour → kevésbé autokorrelált\nOK: kezelhető méret ~8760 sor/év]
  Q --> S3[Napi szintű sampling\nWARN: kevés obs/év\nWARN: elvész az intraday struktúra]
```

| Megközelítés | Előny | Hátrány | Státusz |
|---|---|---|---|
| **Yearly random-hour** | Éves izoláció; random hour → kevésbé autokorrelált minták; kezelhető méret | Nem maximalizálja a historikus adatot | ✅ Választott |
| Expanding window CV | Maximális historikus kontextus; hagyományos ML-CV analógia | Évhatáron átnyúló szivárgás lehetséges; nem mér éves stabilitást | ❌ Elvetett — éves izolációt nem biztosít |
| Napi mintavétel (1 bar/nap) | Minimális korreláció | Elvész az intraday mintázat; ~365 sor/év → túl kevés | ❌ Elvetett — intraday struktúra elvész |
| Minden perc (nincs mintavétel) | Maximális adatsűrűség | Erős autokorrelációval torzított metrikák | ❌ Elvetett — szivárgás kockázata magas |

---

### Random hour selection: miért kell és hogyan működik?

Az 1 perces OHLCV sorok erősen autokorreláltak — egymást követő percek közel azonos
feature-vektorokat adnak. Ha minden percet betennénk, a validációs metrikák optimistán
torzítottak lennének (a model "emlékszik" az előző percre).

Az **óránkénti random mintavétel** csökkenti ezt az autokorrelációt: az egy percnyi
ugrás + random kiválasztás biztosítja, hogy szomszédos sorok ne ugyanazon árjelből
következzenek.

**Szabály:** Egy naptári évből pontosan 1 sort választunk minden teljes óra-egységre.
Szökőévben 8 784, standard évben 8 760 sort kapunk (ha az adatbázis teljes).

**Reprodukálhatóság:** a kiválasztás `open_time.cast(Int64).hash(seed, seed+1)` alapú
— row-ordering független, azonos seed + év → azonos kiválasztás.

```{mermaid}
flowchart TD
  IN[Összes perc az évből] --> F1[Filter: YEAR == config.year]
  F1 --> H[Truncate: open_time → 1h]
  H --> HASH[Hash per sor: Int64.hash\nseed, seed+1]
  HASH --> SORT[Sort: hour + hash]
  SORT --> U[Unique: keep first per hour]
  U --> OUT[~8760 sor / év]
```

---

### Monthly validation week: miért kell és hogyan működik?

A validáció célja a generalizációs képesség mérése. Egy szezon-izolált validáció
(pl. csak Q4) elfogult lehet a piaci ciklus adott fázisára. Ezért **12 validációs
hetet** választunk — hónaponként egyet —, hogy minden naptári hónap és piaci
szezon képviselve legyen.

A hetek **teljes Monday–Sunday egységek**: ez megőrzi az intraday és intraweek
periodikus mintázatokat a validációs ablakban. Az esetleges hónaphatár-átlépés
(pl. dec. 29 → jan. 4) elfogadható — a hét integritása prioritás.

**Kiválasztás:** minden hónapban az összes hétfő listájából `random.Random(seed)`
választ egyet (determinisztikus, seed-függő).

```{mermaid}
flowchart LR
  M1[Jan] --> W1[1 random hét\nMon–Sun]
  M2[Feb] --> W2[1 random hét]
  M3[...] --> W3[...]
  M12[Dec] --> W12[1 random hét]
  W1 & W2 & W3 & W12 --> VALID[valid szegmens\n~2016 sor]
```

---

### Purge (±240 perc): miért kell és hogyan működik?

A `feat_ohlcv_quant` feature-ök rolling ablakokkal számítottak. Ha egy validációs
hét elején lévő sor feature-vektora egy train-beli percből "visszanéz" az előző
ablakba, az implicit tudást hordoz a train adatból — ez szivárgás.

A **purge** ezt kezeli: a validációs hét előtti és utáni `purge_minutes` percben lévő
sorokat se trainbe, se validba nem tesszük.

```
... [train] ... [purge 240 perc] [valid hét Mon–Sun] [purge 240 perc] [train] ...
```

**Miért 240 perc?**
A `features.json`-ban a leghosszabb rolling ablak 140 bar (= 140 perc 1m chart-on).
A 240 perces purge ~71%-os biztonsági margót ad a 140 perces max lookback fölé.
Ez biztosítja, hogy még a leghosszabb feature-ablak is biztosan a train-en belül
marad, nem "néz bele" a válida ablak előtti percekbe.

**Szabály:** Purge sorok sem train, sem valid set-be nem kerülhetnek.

---

### Segment értékek és definíciók

| Érték | Leírás |
|-------|--------|
| `train` | Minden sor, amely nincs valid vagy purge ablakban |
| `valid` | Pontosan a 12 validációs hét (Mon 00:00 → Sun 23:59) |
| `purge` | ±240 perces zóna minden validációs hét határán (nem fed át validdal) |

Prioritási sorrend az assign_segments logikájában: valid > purge > train.

---

### Paraméter alapértékek és indoklásuk

| Paraméter | Alapérték | Indoklás |
|-----------|-----------|---------|
| `purge_minutes` | `240` | Max feature lookback = 140 perc; 240 perc ~71%-os biztonsági margó; biztonságos default a jövőbeli feature-bővítésekre is |
| `seed` | `42 + year` | Évenként eltérő seed → különböző óra- és hétválasztás; reprodukálható, dokumentálható; 42 konvencionális ML alap |
| `target_cols` | `("long_mfe_fw60", "short_mfe_fw60")` | Aktív target páros a v4 modellekhez; tuple → immutable config |
| `feature_cols` | `()` | Üres tuple = minden `feat_*` oszlop auto-discovery quant_train-ből futásidőben |
| `sample_id` | `{asset_id}_fw60_yearly_{year}` | Emberi olvashatóság + programmatikus parse-olhatóság; egyértelműen azonosítja az évet és stratégiát |

---

### Ismert kockázatok és korlátok

| Kockázat | Tünet | Mitigáció |
|---|---|---|
| Év-határon átnyúló validation week (pl. dec. 29 → jan. 4) | 2025-ös évben valid=1920 (nem 2016) — jan. adatok hiányoznak | Elfogadott viselkedés; audit.json tartalmaz `missing_hours` mezőt; alert ha valid < 1800 |
| Szökőév (pl. 2024) módosítja a purge számot | 2024-ben purge=84 (nem 96) — év-határon átnyúló purge ablakok rövidülnek | Elfogadott; dokumentált; modell-összehasonlításhoz az éves row count-okat rögzíteni kell |
| Hiányzó DB adatok az évben | `missing_hours > 0` az audit-ban | Ellenőrizd az audit.json-t minden generált sample-nál; ne használd ha `missing_hours > 500` |
| Random hour selection nem fed le minden intraday mintát | Szisztematikus intraday anomáliák (pl. funding hour spike) alulreprezentáltak | Elfogadott — a hash-based random egyenletes eloszlást közelít; manuális audit ajánlott ha intraday pattern ismert |
| Régi expanding window pipeline nem kompatibilis a yearly formátummal | `lightgbm_model.py` `load_sample_definition` (`folds.json`) szintaxist vár | Az új training pipeline (yearly format aware) külön epic feladata; addig ne futtass train-t yearly sample-en a régi pipeline-nal |

---

### Validációs checklist

- [ ] `sample_train_valid.parquet` létezik a sample könyvtárban
- [ ] `metadata.json` tartalmaz: `year`, `seed`, `selected_valid_weeks` (12 elem), `row_counts` szegmensenként
- [ ] `audit.json` tartalmaz: `missing_hours`, `total_quant_train_rows_in_year`, `actual_hourly_rows`
- [ ] Standard évben: `valid ≈ 2016` (12 hét × 168 óra), `purge ≈ 96` (kb.)
- [ ] Szökőévben (2024): `valid ≈ 2016`, `purge ≤ 96`
- [ ] Nincs `open_time` átfedés train és valid szegmens között
- [ ] Purge sorok sem trainben, sem validban nem szerepelnek
- [ ] `segment` oszlop értékkészlete: `{"train", "valid", "purge"}` — semmi más
- [ ] Azonos seed + év → azonos `sample_train_valid.parquet` (reprodukálhatósági teszt)
- [ ] `missing_hours < 500` (ha felette van: vizsgáld meg az adatbázis hiányait)

---

## Kapcsolódó fájlok

| Szám | Fájl | Tartalom |
|------|------|----------|
| 5100 | [5100_sampling_config.md](5100_sampling_config.md) | YearlySamplingConfig dataclass |
| 5200 | [5200_sampling_artifacts.md](5200_sampling_artifacts.md) | write_yearly_artifacts / load_yearly_sample |
| 5300 | [5300_create_sample.md](5300_create_sample.md) | create_yearly_sample orchestrator + CLI |
| 5400 | [5400_sampling.md](5400_sampling.md) | **LEGACY** — expanding window CV (archív, nem aktív) |




---

<!-- Source: 5100_sampling_config.md -->



﻿# 5100 — YearlySamplingConfig

Immutable dataclass, amely az összes paramétert tartalmazza egy éves random-óra sample
generálásához. Forrás: [sampling/config.py](../src/modeling/sampling/config.py)

---

## Overview

```{mermaid}
classDiagram
  class YearlySamplingConfig {
    +str sample_id
    +str asset_id
    +int year
    +int seed
    +int purge_minutes = 240
    +tuple target_cols = ("long_mfe_fw60", "short_mfe_fw60")
    +tuple feature_cols = ()
  }
```

---

## Mezők

| Mező | Típus | Default | Leírás |
|------|-------|---------|--------|
| `sample_id` | `str` | — | Egyedi azonosító; ez lesz a `samples/` alkönyvtár neve |
| `asset_id` | `str` | — | Asset kulcs a `config/assets.json`-ból (pl. `solusdt`) |
| `year` | `int` | — | Naptári év, amelyből a sample készül (pl. `2021`) |
| `seed` | `int` | — | Véletlenszám-generátor seedje; minden óra- és hétválasztás ebből származik |
| `purge_minutes` | `int` | `240` | Purge zóna szélessége percben minden validációs hét határán |
| `target_cols` | `tuple[str, ...]` | `("long_mfe_fw60", "short_mfe_fw60")` | Target oszlopok, amelyek a sample_train_valid.parquet-be kerülnek |
| `feature_cols` | `tuple[str, ...]` | `()` | Feature oszlopok; üres tuple = minden `feat_*` auto-discovery futásidőben |

### `purge_minutes` default indoklása

A `feat_ohlcv_quant`-ban a leghosszabb rolling ablak 140 bar (= 140 perc 1m chart-on).
A 240 perces default ~71%-os biztonsági margót ad, hogy a jövőbeli feature-bővítések
is biztonságban legyenek purge csökkentés nélkül.

### `feature_cols` üres tuple szemantikája

Ha `feature_cols` üres, a `create_yearly_sample` orchestrator futásidőben felfedezi
az összes `feat_*` oszlopot a `quant_train` sémájából — ez az ajánlott működési mód.
Explicit lista csak akkor szükséges, ha feature-szelekcióval korlátozott sample kell.

---

## Példa inicializálás

```python
from modeling.sampling.config import YearlySamplingConfig

config = YearlySamplingConfig(
    sample_id = "solusdt_fw60_yearly_2021",
    asset_id  = "solusdt",
    year      = 2021,
    seed      = 42 + 2021,   # 2063
)
```

A `frozen=True` miatt a dataclass példányosítás után nem módosítható — minden
paraméter-változtatáshoz új `YearlySamplingConfig` példányt kell létrehozni.

---

## Kapcsolódó fájlok

| Fájl | Tartalom |
|------|----------|
| [5010_sampling_yearly.md](5010_sampling_yearly.md) | Yearly sampling teljes metodológiája |
| [5300_create_sample.md](5300_create_sample.md) | create_yearly_sample orchestrator és CLI |




---

<!-- Source: 5200_sampling_artifacts.md -->



﻿# 5200 — Sampling Artifacts

Artifact IO modul: yearly formátumhoz `write_yearly_artifacts` / `load_yearly_sample`;
legacy expanding-window formátumhoz `write_sample_artifacts` / `load_sample_definition`
(visszafele kompatibilitás). Forrás: [sampling/artifacts.py](../src/modeling/sampling/artifacts.py)

---

## Yearly formátum (aktív)

### `write_yearly_artifacts()`

Kiírja a sample könyvtárba: `metadata.json`, `audit.json`, `sample_train_valid.parquet`.
Automatikusan létrehozza a könyvtárat ha nem létezik.

```{mermaid}
flowchart TD
  A[write_yearly_artifacts\nsample_dir, metadata, segment_df, audit] --> B[metadata.json\n+ generated_at]
  A --> C[audit.json]
  A --> D[sample_train_valid.parquet\nZSTD tömörítve]
  B & C & D --> E[database/asset_id/samples/sample_id/]
```

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `sample_dir` | `Path` | Célkönyvtár (létrehozza ha nincs) |
| `metadata` | `dict` | Sample metadata (sample_id, year, seed, selected_valid_weeks, row_counts, …) |
| `segment_df` | `pl.DataFrame` | Polars DataFrame open_time + target oszlopok + `segment` + `fold_id` |
| `audit` | `dict` | Forrásadat minőségi metrikák (missing_hours, actual_hourly_rows, …) |

**`generated_at` injektálás:** automatikusan bekerül az aktuális UTC ISO timestamp.

---

### `load_yearly_sample()`

Beolvassa a `metadata.json`-t és ellenőrzi, hogy a `sample_train_valid.parquet` létezik.

```python
sample = load_yearly_sample("database/solusdt/samples/solusdt_fw60_yearly_2021")
# sample["sample_parquet_path"] → "database/.../sample_train_valid.parquet"
```

**Raises:** `FileNotFoundError` ha `metadata.json` vagy `sample_train_valid.parquet` hiányzik.

---

## Artifact fájlok sémája

### `metadata.json`

```json
{
  "sample_id"           : "solusdt_fw60_yearly_2021",
  "asset_id"            : "solusdt",
  "year"                : 2021,
  "seed"                : 2063,
  "purge_minutes"       : 240,
  "target_cols"         : ["long_mfe_fw60", "short_mfe_fw60"],
  "feature_cols"        : ["feat_rsi_14", "feat_vol_200"],
  "selected_valid_weeks": [
    {"start": "2021-01-04", "end": "2021-01-10"},
    "..."
  ],
  "row_counts"          : {"train": 7012, "valid": 2016, "purge": 96},
  "generated_at"        : "2025-06-01T10:00:00+00:00"
}
```

### `audit.json`

```json
{
  "total_quant_train_rows_in_year": 525600,
  "source_rows_with_valid_targets": 525480,
  "expected_hours"                : 8760,
  "actual_hourly_rows"            : 8760,
  "missing_hours"                 : 0
}
```

### `sample_train_valid.parquet`

| Oszlop | Típus | Leírás |
|--------|-------|--------|
| `open_time` | `Datetime` | Timestamp (UTC) |
| `feat_*` | `Float64` | Kvantitatív feature-ök |
| `long_mfe_fw60` | `Float64` | Long target |
| `short_mfe_fw60` | `Float64` | Short target |
| `segment` | `Utf8` | `train` / `valid` / `purge` |
| `fold_id` | `Int16` (nullable) | Validációs hét indexe (0-based); train sorokra null |

---

## Legacy formátum (expanding window — archív)

A `write_sample_artifacts` / `load_sample_definition` / `validate_sample_definition`
funkciók az expanding window CV sample formátumhoz tartoznak. Ezek csak visszafele
kompatibilitás miatt maradnak a kódban — új munkában ne használd.

| Fájl | Leírás |
|------|--------|
| `metadata.json` | Expanding window paraméterek (min_train_days, valid_days, …) |
| `folds.json` | `{"folds": [...], "test": {...}}` — fold határok |
| `audit.json` | Feature table audit (data_start_safe, data_end_safe, gap_count, …) |

---

## Kapcsolódó fájlok

| Fájl | Tartalom |
|------|----------|
| [5010_sampling_yearly.md](5010_sampling_yearly.md) | Yearly sampling teljes metodológiája |
| [5100_sampling_config.md](5100_sampling_config.md) | YearlySamplingConfig dataclass |
| [5300_create_sample.md](5300_create_sample.md) | create_yearly_sample orchestrator |




---

<!-- Source: 5300_create_sample.md -->



﻿# 5300 — create_yearly_sample Orchestrator és CLI

A sampling orchestrator összefogja a DB load → hourly select → segment assign → write
lépéseket egyetlen `create_yearly_sample(config)` hívásba. Csak ez a modul importálja
a `utils`-t és DuckDB-t — az altmodulok (yearly_sampler, artifacts) projekt-agnosztikusak.

Forrás:
- [sampling/create_sample.py](../src/modeling/sampling/create_sample.py)
- [00_create_sample.py](../src/modeling/00_create_sample.py)

---

## Overview

```{mermaid}
sequenceDiagram
  participant CLI as 00_create_sample.py
  participant CS as create_yearly_sample()
  participant U as utils.load_asset_config
  participant DB as DuckDB (quant_train)
  participant HS as select_hourly_observations
  participant MV as select_monthly_validation_weeks
  participant AS as assign_segments
  participant W as write_yearly_artifacts

  CLI ->> CS: YearlySamplingConfig
  CS ->> U: config.asset_id
  U -->> CS: db_path
  CS ->> DB: SELECT feat_* + target FROM quant_train WHERE year = config.year
  DB -->> CS: pl.DataFrame (~525 000 sor)
  CS ->> HS: df, config.year, config.seed
  HS -->> CS: hourly_df (~8 760 sor)
  CS ->> MV: hourly_df, config.year, config.seed
  MV -->> CS: 12 (week_start, week_end) tuple
  CS ->> AS: hourly_df, valid_weeks, config.purge_minutes
  AS -->> CS: segment_df (train/valid/purge + fold_id)
  CS ->> CS: audit dict + metadata dict összeállítása
  CS ->> W: sample_dir, metadata, segment_df, audit
  W -->> CS: kész (metadata.json, audit.json, sample_train_valid.parquet)
  CS -->> CLI: return None
  CLI ->> CLI: load_yearly_sample → print összefoglaló
```

---

## `create_yearly_sample(config)`

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `config` | `YearlySamplingConfig` | Frozen dataclass az összes paraméterrel |

### Lépések

1. **Útvonalak feloldása** — `utils.load_asset_config(asset_id)` → `db_path`; `sample_dir` = `database/<asset_id>/samples/<sample_id>/`
2. **DB betöltés** — `quant_train`-ből év-szűrt sorok, NULL target sorok kizárva
3. **Feature column feloldás** — `config.feature_cols` ha nem üres; különben auto-discovery minden `feat_*` oszlop
4. **Óránkénti kiválasztás** — `select_hourly_observations` → ~8 760 sor/év
5. **Validációs hetek** — `select_monthly_validation_weeks` → 12 hét (mind a 12 hónapból)
6. **Szegmens hozzárendelés** — `assign_segments` → `train` / `valid` / `purge` + `fold_id`
7. **Audit** — `missing_hours`, `actual_hourly_rows`, `total_quant_train_rows_in_year`
8. **Kiírás** — `write_yearly_artifacts` → `metadata.json`, `audit.json`, `sample_train_valid.parquet`

**Raises:**
- `ValueError` ha a `quant_train`-nek nincs sora érvényes targettel az adott évre
- `RuntimeError` ha a `quant_train` tábla nem létezik

---

## CLI — `00_create_sample.py`

### Argumentumok

| Argument | Kötelező | Default | Leírás |
|----------|----------|---------|--------|
| `--year` | igen | — | Naptári év (pl. `2021`) |
| `--asset-id` | igen | — | Asset kulcs (`config/assets.json`-ból) |
| `--seed` | nem | `42 + year` | Véletlenszám seed |

### Példa CLI hívás

```bash
uv run python src/modeling/00_create_sample.py --year 2021 --asset-id solusdt
uv run python src/modeling/00_create_sample.py --year 2022 --asset-id solusdt --seed 100
```

### Output summary

Sikeres futás után a CLI összefoglalót nyomtat:

```
OK: Sample created at database/solusdt/samples/solusdt_fw60_yearly_2021
    year         = 2021
    seed         = 2063
    valid_weeks  = 12
    feature_cols = 208
    total_rows   = 9124
      train      = 7012
      valid      = 2016
      purge      = 96
```

---

## Miért csak az orchestratorban van `utils` import?

Az `yearly_sampler` és `artifacts` modulok szándékosan projekt-agnosztikusak —
tesztelhetők és újrafelhasználhatók projekt kontextus nélkül. Csak az orchestrator
ismeri a projekt-specifikus path-konvenciókat és config formátumot.

---

## Kapcsolódó fájlok

| Fájl | Tartalom |
|------|----------|
| [5010_sampling_yearly.md](5010_sampling_yearly.md) | Yearly sampling teljes metodológiája |
| [5100_sampling_config.md](5100_sampling_config.md) | YearlySamplingConfig dataclass |
| [5200_sampling_artifacts.md](5200_sampling_artifacts.md) | write_yearly_artifacts / load_yearly_sample |




---

<!-- Source: 5400_sampling.md -->



﻿# 5400 — Sampling (LEGACY — Expanding Window CV)

> **ARCHÍV DOKUMENTUM.** Ez a fájl az expanding window CV alapú sampling megközelítést
> írja le, amelyet felváltott a yearly random-hour sampling.
> Az aktív metodológia: [5010_sampling_yearly.md](5010_sampling_yearly.md)
>
> A kódban a legacy funkciók (`write_sample_artifacts`, `load_sample_definition`,
> `validate_sample_definition`, `audit_feature_table`, `build_expanding_window_splits`)
> visszafele kompatibilitás miatt megmaradnak, de új munkában ne használd őket.

---

A sampling almodul időalapú keresztvalidációs sample definíciókat generál. Kimenet:
három JSON fájl (metadata, folds, audit) és egy Parquet fájl (`sample.parquet`) a
`database/<asset>/samples/<sample_id>/` könyvtárban — ezeket olvassa be a modell
tanítási pipeline és az elemzési notebookok.

---

## Overview

Az expanding window CV lényege: minden fold-ban a training ablak visszanyúlik az
adatok elejéig (nem csúszóablak), a validációs ablak pedig előre gördül. Embargo
(embargó gap) biztosítja, hogy a feature-ök `target_horizon_minutes` percnyi
előrenézési ablaka ne szivárogjon át train→valid határon.

```{mermaid}
flowchart TD
  A[SamplingConfig] --> F[create_sample\ncreate_sample.py]
  F --> B[audit_feature_table\naudit.py]
  B --> C[build_expanding_window_splits\nsplits.py]
  C --> D[write_sample_artifacts\nartifacts.py]
  D --> E[database/asset/samples/id/\nmetadata.json\nfolds.json\naudit.json]
  C --> G[_write_sample_parquet\ncreate_sample.py]
  G --> H[database/asset/samples/id/\nsample.parquet]
```

---

## Üzleti és módszertani háttér

### Miért kritikus lépés a sampling?

A sampling dönti el, hogy:

1. Milyen adat áll rendelkezésre az adott assetre és targetre?
2. Melyik rész használható modell- és hiperparaméter-szelekciójára?
3. Melyik rész marad érintetlen végső holdout-ként?
4. Melyik perzisztált sample definíciót kell az összehasonlítható modelleknek újra felhasználni?

A helytelen sampling a leggyakoribb forrása a félrevezető backtest-eredményeknek.
Egy szétcsúszott train/valid határ vagy kimaradt embargo torzított metrikákat ad
— a modell jónak látszik, de élesben nem teljesít.

---

### Miért kronologikus split? (Nem random CV)

Pénzügyi idősorokon a standard random k-fold keresztvalidáció **szivárgáshoz vezet**:
egy 2024-es bar kerülhet a train set-be, miközben a 2023-as szomszédja a validation-be
— így a modell implicit módon jövőbeli információhoz fér hozzá tanítás során.

A piac nem i.i.d. folyamat: trend, volatilitás-rezsim, korreláció időben változik.
A cronológikus split imitálja azt a helyzetet, amit live kereskedésben tapasztalunk:
a model csak a múltat látja, és a jövőre van kiértékelve.

**Szabály:** Mindig kronologikus splitet használj. Shuffled CV csak akkor megengedett,
ha az adott kísérlet dokumentáltan igazolja, hogy biztonságos.

---

### Miért expanding window? (Nem sliding/rolling)

Két lehetséges CV stratégia:

| Stratégia | Train ablak | Hátránya |
|-----------|-------------|----------|
| Sliding (rolling) window | Fix hosszú, előre csúszik | Korai adatok kiesnek; kevesebb stabil becslés |
| **Expanding window** | Mindig az adatok elejétől, csak vége nő | Maximális historikus kontextus, stabil tanítás |

Az expanding window azért preferált, mert:
- Több historikus adat → jobb szignál/zaj arány
- Az első fold már 2+ éves historikus háttérrel rendelkezik (`min_train_days = 730`)
- Intraday kereskedési szignálok esetén a korai évek (pl. 2020–2021 bull market) is releváns rezsim-kontextust adnak

---

### Embargo: miért kell és hogyan működik?

A `feat_ohlcv_quant` feature-ök egy részét rolling ablakokkal számítjuk. A
`long_mfe_fw60` target egy 60 perces előre néző ablakot vesz figyelembe — ez azt
jelenti, hogy az ablak határán lévő feature sorok **implicit módon tartalmazzák
jövőbeli információt** (pl. az átlag kiszámításához a target ablak áraihoz is nyúlik
a rolling window).

Ha a train vége és a valid eleje között nincs gap, ezek a sorok nem szivárogthatnak
— de pont az átmeneti zónában lévő sorok kerülnének oda. Az **embargo** a megoldás:

```
train_end = valid_start − embargo_minutes − 1 perc
```

Az embargo mérete alapértelmezetten a `target_horizon_minutes` értéke (azaz 60 perc
a fw60 targetnél), ami garantálja, hogy a target ablak által érintett percek mindig
kiesnek a training set végéről.

---

### Egész hónapok szabálya

A sampling mindig teljes naptári hónapokra korlátozódik — nem az audit által visszaadott
pontos timestampre. Az indok: a hónaphatáros adatszeletelés megkönnyíti az összehasonlítást
modellek és verziók között, mivel az időszak emberi kommunikációban is egyértelmű.

**Szabály:**

- `data_start` → felfelé kerekítés a következő hónap első napjára (ha a tényleges start nem hó eleje)
- `data_end` → lefelé kerekítés az előző hónap utolsó percére

**Példa:**

| Audit eredmény | Kerekítés után |
|----------------|---------------|
| `data_start_safe = 2020-09-14 07:00:00` | `2020-10-01 00:00:00` |
| `data_end_safe   = 2026-06-12 18:20:00` | `2026-05-31 23:59:00` |

**Sample névkonvenció:** `solusdt_fw60_YYMM_YYMM` ahol YYMM a kerekített start és end hónap.
Pl.: `solusdt_fw60_2010_2605` (2020-10 → 2026-05).

---

### Final holdout: az "érettségi vizsga"

A végső holdout (test set) a **legfrissebb 365 nap** (alapértelmezetten). Ez:

- **Nem kerül felhasználásra** feature-, modell-, hiperparaméter- vagy trigger-szelekciójához
- **Egyetlen alkalommal** van kiértékelve: amikor a kutatási döntések megvannak
- **Nem eldobott adat:** ha egy kandidáns átment a holdout-ellenőrzésen, az összes
  jóváhagyott adat (beleértve a holdup-ot) felhasználható a promóciós fitre

A holdout gondolata: a kutatási döntések meghozataláig az "examinál" adat ismeretlen.
Ha a holdoton is átmegy a modell, az erős jele annak, hogy nem overfit-elt a
kutatási fázisra.

---

### Promotion fit

Promotion (élesítés) előtt a modell újrataníthato az összes jóváhagyott adaton:

```
Kutatási fázis:  data_start → pre-holdout data  (fold CV + trigger selection)
Final holdout:   újabb 365 nap (csak egyszer kiértékelve)
Promotion fit:   data_start → latest safe timestamp (holdout jóváhagyás után)
```

A promotion fit nem véletlenszerű túlillesztés: a döntések a kutatási fázisban
születtek, a holdout csak validál. Az összes adaton való refitelés a live deployment
hatékonyságát növeli.

---

## Sample ID policy

### Mikor lehet ugyanazt a `sample_id`-t újra felhasználni?

Reuse ugyanazon `sample_id`-vel, ha az összehasonlíthatóság szükséges:

- Ugyanarra az assetre és horizonra épülő long és short modellek
- Ugyanarra a targetre szánt LightGBM kandidáns verziók
- Baseline vs. champion összehasonlítás

### Mikor kell új `sample_id`?

| Ok | Magyarázat |
|----|-----------|
| Asset vagy forrástábla változott | Az adatok összehasonlíthatatlanok |
| Target horizon változott | A fold határok más perceket jelölnek |
| Label definíció változott | A target értékek eltérnek |
| Feature tábla rebuild | Ha az elérhető dátumtartomány jelentősen változott |
| Split paraméterek változtak | A fold határok eltérnek |
| Adatminőségi javítás | Ha a minta érdemben megváltozott |

**Elv:** Az összehasonlítható modelleknek azonos `sample_id`-vel kell rendelkezniük —
különben az összehasonlítás érvénytelen (más train/valid határokon értékelve).

---

## Startup adatellenőrzés

Minden modell-fejlesztési ciklus előtt futtasd az audit-ot az alábbi szempontokra:

- Feature tábla első és utolsó elérhető timestampja
- Sorok száma a feature táblában
- Szükséges target oszlopok meglétét az adott horizonhoz
- Target és feature null arányok
- Duplikált `open_time` értékek
- Időbeli hézagok (gap) a feature táblában
- Target horizon és embargo igény

Az audit eredménye határozza meg a `data_start_safe` értékét, ami a splitek
generálásának alapja.

---

## Target NULL szemantika

A feature tábla target oszlopai (`long_mfe_fw60`, `short_mfe_fw60`) folytonos fw60 outcome-okat tartalmaznak:

| Érték | Jelentés |
|-------|---------|
| `DOUBLE` | Számszerű fw60 outcome a `target` táblából |
| `NULL` | Forward adat még nem elérhető — az utolsó `rolling_window` bar-ban vagyunk |

### Miért fontos a NULL?

A target egy fordított rolling ablakkal számított folytonos outcome. Az utolsó
`rolling_window` bar-nál (pl. 60 bar a fw60-nál) az ablak nem tartalmaz teljes
forward adatot — ezért ezek a sorok ismeretlen állapotban vannak.

**Alapelv:** Ne imputáld a NULL targeteket 0-val. A NULL valóban ismeretlen, nem
egy semleges outcome. A modell-pipeline `dropna(subset=[target_col])` szűréssel
kezeli őket.

Az utolsó `rolling_window` bar-ban lévő, nem-nulla NULL arány normális és helyes.
Ha ezen kívül is NULL értékek vannak, az adatpipeline-ban van hiba.

---

## Paraméter alapértékek és indoklásuk

| Paraméter | Alapérték | Indoklás |
|-----------|-----------|---------|
| `min_train_days` | 730 (2 év) | Elegendő historikus kontextus az első validációs foldhoz; rövidebb historikus adat instabil modellhez vezet |
| `valid_days` | 180 (6 hó) | Érdemi időtáv a modell stabilitásának méréséhez, de nem annyira hosszú, hogy kevés fold keletkezzen |
| `step_days` | 180 (6 hó) | Megegyezik a `valid_days`-zel — non-overlapping validációs ablakok biztosítva |
| `test_days` | 365 (1 év) | Legalább egy teljes éves holdout; piaci szezonalitást lefedje |
| `embargo_minutes` | `None` → `target_horizon_minutes` | Automatikusan a target horizon értéke; ha nincs megadva, nem lehet tévesen nulla |

Rövidebb holdout indokolt, ha az asset historikusan kevés adattal rendelkezik.
Hosszabb holdout indokolt rezsim-érzékeny kutatásnál, de csökkenti a kutatási fázis
rendelkezésre álló adatát.

---

## Expanding window CV — kulcsfogalmak

| Fogalom | Leírás |
|---------|--------|
| Expanding train | Train ablak mindig `data_start_safe`-tól indul, csak a vége tolódik |
| Fixed valid | Minden fold-ban azonos hosszú validációs ablak |
| Embargo | `embargo_minutes` perces gap train vége és valid eleje között |
| Test set | Fix végső holdout — az összes fold után, embargóval elválasztva |

---

## Artifact output

Minden `create_sample` futtatás négy fájlt ír a `database/<asset_id>/samples/<sample_id>/` alá:

| Fájl | Tartalom |
|------|----------|
| `metadata.json` | sample_id, asset_id, target_col, paraméterek, adathatárok, generated_at |
| `folds.json` | `{"folds": [...], "test": {...}}` — időhatárok fold-onként |
| `audit.json` | Feature tábla minőségi metrikák (gap, null arány, sorok száma) |
| `sample.parquet` | Összes sor (feat_ohlcv_quant + target + `segment` label), ZSTD tömörítve |

### sample.parquet struktúra

| Oszlop | Forrás | Leírás |
|--------|--------|--------|
| `open_time` | `feat_ohlcv_quant` | Timestamp (YYYY-MM-DD HH:MM:SS) |
| `feat_*` (208 db) | `feat_ohlcv_quant` | Kvantitatív feature-ök |
| `long_mfe_fw60` | `target` | Long fw60 outcome |
| `short_mfe_fw60` | `target` | Short fw60 outcome |
| `segment` | generált | Szegmens azonosító (ld. lent) |

**`segment` oszlop értékkészlete:**

| Érték | Leírás |
|-------|--------|
| `fold_1_train` … `fold_N_train` | Az N. fold training adatai |
| `fold_1_valid` … `fold_N_valid` | Az N. fold validációs adatai |
| `test` | Végső holdout |

Az expanding window miatt az azonos sor több `fold_K_train` szegmensben is szerepelhet
(pl. 2021-es adat fold_1_train-ben és fold_2_train-ben is). Szegmentált olvasáshoz
használj Polars lazy frame-t:

```python
import polars as pl

df = (
    pl.scan_parquet("database/solusdt/samples/<sample_id>/sample.parquet")
    .filter(pl.col("segment") == "fold_1_train")
    .collect()
)
```

---

## Validációs checklist

- [ ] `metadata.json`, `folds.json`, `audit.json` és `sample.parquet` létezik a `sample_id`-hoz
- [ ] Fold határok kronologikusak és nem fednek át
- [ ] Embargo elválasztja a train és valid/test sorokat
- [ ] A végső holdout nem volt felhasználva modell-, feature-, hiperparaméter- vagy trigger-szelekciójára
- [ ] Összehasonlítható kandidánsok azonos `sample_id`-t használnak
- [ ] A minta dátumtartománya megfelel az aktuális modellezési kérdésnek
- [ ] Nincs NULL target a forward-edge-en kívül
- [ ] `sample.parquet` `segment` oszlopában minden elvárt szegmens jelen van (`fold_1_train`…`fold_N_valid`, `test`)

---

## Alfejezetek

| Szám | Fájl | Tartalom |
|------|------|----------|
| 5100 | [5100_sampling_config.md](5100_sampling_config.md) | SamplingConfig dataclass |
| 5410 | [5410_sampling_splits.md](5410_sampling_splits.md) | build_expanding_window_splits |
| 5420 | [5420_sampling_audit.md](5420_sampling_audit.md) | audit_feature_table |
| 5200 | [5200_sampling_artifacts.md](5200_sampling_artifacts.md) | write / load / validate artifacts |
| 5300 | [5300_create_sample.md](5300_create_sample.md) | create_sample orchestrator + CLI |




---

<!-- Source: 5410_sampling_splits.md -->



﻿# 5410 — Expanding Window Splits

Pure date-math modul: nincs IO, nincs adatbázis-hozzáférés, nincs projekt-import.
Bemenete és kimenete `YYYY-MM-DD HH:MM:SS` formátumú stringek.
Forrás: [sampling/splits.py](../src/modeling/quantitative/sampling/splits.py)

---

## Overview

```{mermaid}
flowchart TD
  A[data_start\ndata_end] --> B[test_start = data_end - test_days]
  B --> C[cv_end = test_start - embargo]
  A --> D[valid_start_0 = data_start + min_train_days]
  D --> E{valid_start + valid_days <= cv_end?}
  E -- igen --> F[fold létrehozása\ntrain_end = valid_start - embargo - 1min]
  F --> G[valid_start += step_days]
  G --> E
  E -- nem --> H[folds lista kész]
  H --> I[return dict\nfolds + test]
```

---

## `build_expanding_window_splits()`

Expanding training ablakokat generál fix hosszú validációs ablakokkal és egy végső
test set-tel.

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `data_start` | `str` | Legkorábbi felhasználható timestamp (`YYYY-MM-DD HH:MM:SS`) |
| `data_end` | `str` | Utolsó címkézett timestamp |
| `min_train_days` | `int` | Minimális training ablak naptári napban |
| `valid_days` | `int` | Validációs ablak hossza naptári napban |
| `step_days` | `int` | Egymást követő fold-ok közti lépés napban |
| `test_days` | `int` | Végső holdout ablak hossza napban |
| `embargo_minutes` | `int \| None` | Embargó percben; `None` → 0 perc (nincs gap) |

```{mermaid}
sequenceDiagram
  participant C as create_sample
  participant S as build_expanding_window_splits
  C ->> S: data_start, data_end, min_train_days, valid_days, step_days, test_days, embargo_minutes
  S ->> S: parse ISO strings → datetime
  S ->> S: számíts test_start, cv_end, valid_start_0
  loop minden fold
    S ->> S: számíts train_end = valid_start - embargo - 1min
    S ->> S: fold dict hozzáadása a listához
    S ->> S: valid_start += step_days
  end
  S -->> C: {"folds": [...], "test": {...}}
```

### Return dict struktúra

```json
{
  "folds": [
    {
      "fold": 1,
      "train_start": "2021-01-01 00:00:00",
      "train_end":   "2022-12-31 22:59:00",
      "valid_start": "2022-12-31 23:00:00",
      "valid_end":   "2023-06-29 23:59:00"
    }
  ],
  "test": {
    "start": "2024-01-01 00:00:00",
    "end":   "2024-12-31 23:59:00"
  }
}
```

Minden fold mezői:

| Mező | Leírás |
|------|--------|
| `fold` | 1-indexelt fold sorszám |
| `train_start` | Mindig `data_start` — az expanding window nem változtatja |
| `train_end` | `valid_start - embargo - 1 perc` |
| `valid_start` | Aktuális fold validációs ablakának kezdete |
| `valid_end` | `valid_start + valid_days - 1 perc` |

### Embargo logika

Az embargo gap biztosítja, hogy a training set utolsó példányai és a validation set
első példányai között legalább `embargo_minutes` perc kihagyás legyen:

```
train_end = valid_start - embargo_td - 1 minute
```

Ez kizárja a forward-return ablak által érintett perceket a validációból, megakadályozva
az adatszivárgást ahol egy feature a target kiszámításához felhasznált jövőbeli
adatot is tartalmaz.

### ValueError feltételek

| Feltétel | Hibaüzenet |
|----------|------------|
| `data_end <= data_start` | `"data_end must be after data_start"` |
| Egy fold sem generálható | `"No folds generated. Reduce min_train_days, valid_days, or test_days."` |




---

<!-- Source: 5420_sampling_audit.md -->



﻿# 5420 — Feature Table Audit

A feature tábla auditor meghatározza a biztonságos adathatárokat és minőségi
metrikákat a `feat_ohlcv_quant` és `target` táblákra. Csak read-only DuckDB
lekérdezéseket futtat — nincs Polars, nincs pandas.
Forrás: [sampling/audit.py](../src/modeling/quantitative/sampling/audit.py)

---

## Overview

```{mermaid}
sequenceDiagram
  participant CS as create_sample
  participant A as audit_feature_table
  participant DS as dataset_columns
  participant DB as DuckDB
  participant R as _run_audit

  CS ->> A: db_path, target_col
  A ->> DS: db_path, "feat_ohlcv_quant"
  DS -->> A: all_cols lista
  A ->> A: szűr feat_* oszlopokra
  A ->> DB: duckdb.connect(db_path, read_only=True)
  DB -->> A: conn
  A ->> R: conn, feat_cols, target_col
  R ->> DB: row/uniqueness stats query
  R ->> DB: data_start_safe query (MIN WHERE all feat_ NOT NULL)
  R ->> DB: data_end_safe query (MAX WHERE target NOT NULL)
  R ->> DB: target_null_count query
  R ->> DB: feature_null_summary query (single aggregation)
  R ->> DB: gap detection query (LAG window function)
  DB -->> R: eredmények
  R -->> A: audit dict
  A -->> CS: audit dict
```

---

## `audit_feature_table(db_path, target_col)`

Auditálja a `feat_ohlcv_quant` és `target` táblákat a biztonságos sampling határok
meghatározásához.

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `db_path` | `str` | Abszolút útvonal az asset `.duckdb` fájlhoz |
| `target_col` | `str` | Target oszlop neve (pl. `long_mfe_fw60`) |

### Return dict

| Kulcs | Típus | Leírás |
|-------|-------|--------|
| `data_start_safe` | `str \| None` | Első `open_time` ahol az összes `feat_*` oszlop NOT NULL |
| `data_end_safe` | `str \| None` | Utolsó `open_time` ahol a `target_col` NOT NULL |
| `row_count` | `int` | Összes sor a `feat_ohlcv_quant`-ban |
| `unique_timestamps` | `int` | Egyedi `open_time` értékek száma |
| `duplicate_count` | `int` | `row_count - unique_timestamps` |
| `target_null_count` | `int` | Sorok száma ahol `target_col IS NULL` |
| `feature_null_summary` | `dict[str, float]` | Null-arány minden `feat_*` oszlopra |
| `gap_count` | `int` | 1 percnél nagyobb időbeli ugrások száma |
| `gap_minutes_total` | `int` | Összes hiányzó perc az ugrások felett |

### `data_start_safe` és `data_end_safe` — miért fontosak

- **`data_start_safe`**: az első perc ahol az összes feature NOT NULL. A gördülő
  ablakos feature-ök (pl. 200-perces moving average) `min_periods` hosszú
  null-periódussal indulnak — ezeket ki kell zárni a training-ből.

- **`data_end_safe`**: az utolsó perc ahol a target NOT NULL. A `fw60` target az
  utolsó 60 percben szükségszerűen NULL (nincs elég jövőbeli adat) — ez jelzi
  a tényleges adatvégét.

---

## `_run_audit()` — belső függvény

Nem publikus, de a logika megértéséhez fontos. Egyetlen nyílt connection-ön fut
minden SQL lekérdezést.

### Gap detection SQL

```sql
WITH lagged AS (
    SELECT
        open_time,
        LAG(open_time) OVER (ORDER BY open_time) AS prev_time
    FROM feat_ohlcv_quant
)
SELECT
    COUNT(*) AS gap_count,
    COALESCE(SUM(DATEDIFF('minute', prev_time, open_time) - 1), 0) AS gap_minutes_total
FROM lagged
WHERE prev_time IS NOT NULL
  AND DATEDIFF('minute', prev_time, open_time) > 1
```

A LAG window function minden sorhoz meghatározza az előző `open_time`-ot. A `WHERE`
feltétel szűri ki azokat a sorpárokat ahol a különbség > 1 perc — ezek a hiányzó
1-perces gyertyák. A `SUM(diff - 1)` adja a ténylegesen hiányzó percek számát.

### ValueError feltétel

Ha a `feat_ohlcv_quant` táblában nincs egyetlen `feat_*` oszlop sem:
```
ValueError: No feat_* columns found in feat_ohlcv_quant: <db_path>
```




---

<!-- Source: 5500_hyper_param_search.md -->



# 5500 — LightGBM Hyperparameter Search

A hyperparameter search az éves sample és a feature_engineering kimenete alapján
keresi az optimális LightGBM modell paraméterkészletet. Optuna TPE (vagy seeded
random fallback) alapú keresés, stability-penalized RMSE célértékkel.

---

## Overview

```{mermaid}
flowchart TD
  FE[artifact_dir/feature_engineering/\nfeature_set.json] --> S[lgbm_search.run_search]
  SP[sample_dir/\nsample_train_valid.parquet\nmetadata.json] --> S
  DB[(feat_ohlcv_quant\nDuckDB)] --> S
  S --> O1[search/search_best.json]
  S --> O2[search/best_params.json]
  S --> O3[search/search_trials.jsonl\nsearch_summary.csv\ntrial_logs/\ntrial_curves/]
```

**Entry point:**
```bash
uv run python src/modeling/02_hyper_param_search.py --model lgbm_solusdt_l_fw60_2021 --stage smoke
uv run python src/modeling/pipeline.py --model lgbm_solusdt_l_fw60_2021 --step search --stage explore
```

---

## Input

| Forrás | Tartalom | Hogyan töltődik be |
|--------|----------|-------------------|
| `artifact_dir/feature_engineering/feature_set.json` | `selected` lista — a feature engineering által kiválasztott feature-ök | `_load_feature_cols(artifact_dir)` |
| `sample_dir/sample_train_valid.parquet` | `open_time`, `segment`, `target_col` — az éves hourly sample | `load_yearly_sample` + polars |
| `sample_dir/metadata.json` | `selected_valid_weeks` (12 hét), `year` | `load_yearly_sample` |
| DuckDB `feat_ohlcv_quant` | Feature értékek az adott évre | `query_range_pl` |

**Fontos:** a search **kizárólag** a `feature_set.json["selected"]` listán szereplő
feature-öket használja — nem fut újabb feature auditet, nem hasznal hardcoded listát.

---

## Target

A `long_mfe_fw60` és `short_mfe_fw60` target oszlopok folytonos értékek (log-return).
A search ezeket **közvetlenül** használja regressziós targetként — nincs binarizálás,
nincs percentilis küszöb.

| Model típus | Target oszlop | Modell típusa |
|------------|--------------|---------------|
| Long (`_l_`) | `long_mfe_fw60` | `LGBMRegressor` |
| Short (`_s_`) | `short_mfe_fw60` | `LGBMRegressor` |

A `long_mfe_fw60` értéke pozitív ha az ár felfelé ment (long kedvező); a
`short_mfe_fw60` értéke negatív ha az ár lefelé ment (short kedvező).

---

## CV struktúra

A CV a `metadata.json["selected_valid_weeks"]` alapján épül — 12 fold, hónaponként egy.

```
Fold 1: train=összes "train" szegmens sor  |  valid="2021-01-11"–"2021-01-17"
Fold 2: train=összes "train" szegmens sor  |  valid="2021-02-08"–"2021-02-14"
...
Fold 12: train=összes "train" szegmens sor |  valid="2021-12-13"–"2021-12-19"
```

**Kulcspont:** A training set minden foldban UGYANAZ — az összes `train` szegmens sor.
Ez szándékos: a random-hour yearly sampling már elvégezte a strukturális szeparációt
(purge zóna + segment assignment). A search feladata a generalizáció mérése 12
különböző validációs héten keresztül, nem a CV folds train-valid időbeli szeparációja.

**Purge sorok** (`segment == "purge"`) semmilyen foldba nem kerülnek be — sem train,
sem valid oldalra.

---

## Search Stages

| Stage | Trials | Folds | Célja |
|-------|--------|-------|-------|
| `smoke` | 5 | 2 | Pipeline sanity check — nem keresési eredmény |
| `explore` | 60 | mind (12) | Széles régió feltérképezés |
| `refine` | 30 | mind (12) | Legjobb régiók pontosítása |

A `row_stride` paraméter alapértéke **1** minden stage-nél (a sample már hourly
→ ~8 760 sor/év, nem szükséges tovább ritkulni). Manuálisan felülírható.

---

## Search Objective

A search az alábbi penalizált célfüggvényt minimalizálja (lower = better):

```
score = mean(valid_rmse)
      + 0.25 × std(valid_rmse)         # stabilitás penalizálás
      + 0.10 × max(0, gap - 0.03)      # overfitting penalizálás
```

ahol `gap = mean(valid_rmse) - mean(train_rmse)`.

**Miért stabilitást bünteti?** Egy magas variance-ű modell (jó néhány foldon, rossz
másokon) az éles kereskedésben megbízhatatlan. A std(valid_rmse) büntetés preferálja
a konzisztensen közepes modelleket az ingadozó jókkal szemben.

**Miért gap-et bünteti?** Egy 0.03-nál nagyobb train-valid rés overfittingre utal.
A gap penalizálás a regularizált, általánosítható megoldásokat kedvezi.

---

## Search Engine

| Elérhetőség | Engine | Megjegyzés |
|------------|--------|-----------|
| `optuna` csomag elérhető | **Optuna TPE** | Multivariate TPE, seed=42, 20 startup trial |
| `optuna` nem telepítve | Seeded random fallback | `np.random.default_rng(seed=42)`, crude TPE-guide a legjobb quartile alapján |

Az Optuna Sqlite-ba perzisztálja a study-t (`search/optuna_study.db`), így
megszakítás után folytatható (`--resume` nem szükséges — automatikus).

---

## Parameter Space

| Paraméter | Tér | Típus |
|-----------|-----|-------|
| `num_leaves` | [3, 63] (smoke: 31) | log-int |
| `max_depth` | {-1, 2, 3, 4, 5, 6, 8} | kategória |
| `min_child_samples` | [200, 8 000] | log-int |
| `min_child_weight` | [1e-4, 1e-1] | log-float |
| `min_split_gain` | 0 (20% valószínűség) vagy [1e-5, 0.1] | vegyes |
| `reg_alpha` | [1e-3, 10] | log-float |
| `reg_lambda` | [1, 100] | log-float |
| `subsample` | [0.45, 0.95] | uniform |
| `colsample_bytree` | [0.35, 0.95] | uniform |
| `learning_rate` | [0.005, 0.05] | log-float |
| `max_bin` | {63, 127} | kategória |
| `path_smooth` | [1e-3, 10] | log-float |
| `extra_trees` | {True, False} | kategória |

Rögzített (nem keresett): `objective=regression`, `metric=rmse`,
`n_estimators=3000`, `early_stopping=100`, `n_jobs=4`.

---

## Fold metrikák

| Metrika | Leírás |
|---------|--------|
| `rmse` | Root mean squared error — elsődleges optimalizálási metrika |
| `mae` | Mean absolute error — referencia metrika |

---

## Output Artifacts

| Fájl | Tartalom |
|------|----------|
| `search/search_best.json` | Teljes best trial rekord: params, metrics, fold summary |
| `search/best_params.json` | Csak a tunable paraméter dict — az ugyanazon `model_id` fit lépésének inputja |
| `search/search_trials.jsonl` | Compact rekord minden befejezett trialhoz |
| `search/search_summary.csv` | CSV: trial_no, objective_score, params_* — elemzéshez |
| `search/trial_logs/trial_NNNN.json` | Teljes trial rekord fold metricsekkel |
| `search/trial_curves/trial_NNNN_fold_MM.json` | LightGBM eval görbék (tömörítve) |
| `search/failed_trials.jsonl` | Hibás trialok logja |
| `search/optuna_study.db` | Optuna SQLite study (ha optuna telepítve) |

---

## Resume és Dedup

A search **automatikusan folytatható** — minden futtatás beolvassa az előző session
completed/failed hash-eit, és kihagyja a már látott paraméterkombinációkat.
`--retry-failed` flag újra lefuttatja a korábban hibás trialokat.

---

## Kapcsolódó fájlok

| Szám | Fájl | Tartalom |
|------|------|----------|
| 5000 | [5000_modelling.md](5000_modelling.md) | Modeling domain overview |
| 5010 | [5010_sampling_yearly.md](5010_sampling_yearly.md) | Yearly sample struktúra és selected_valid_weeks |
| 2010 | [2010_feature_engineering.md](2010_feature_engineering.md) | Feature selection — feature_set.json generálás |




---

<!-- Source: 6000_trading.md -->



# src/trading/ — Trading Domain

A `src/trading/` modul feladata a kalibrált predikciós modellek alapján kereskedési stratégiák tesztelése, finomhangolása és élő futtatása. A calibration almodul az OOS predikciókból stratégia-artefaktumot állít elő; a live almodul ezt az artefaktumot felhasználva percenkénti ciklusban dönt, majd végrehajtja és naplózza a kereskedéseket.

---

## Adatfolyam

```{mermaid}
flowchart TD
    OOS["artifacts/<model_id>/\nsample_oos.parquet"]
    CAL["00_calibrate_strategy.py\nrun_calibration()"]
    ART["artifacts/<model_id>/strategy/\nstrategy_artifact.json"]
    SWEEP["01_sweep_strategy.py\n(opcionális paraméter sweep)"]
    SVC["02_run_service.py\nTradingService"]
    DB["solusdt.duckdb\n(predictions)"]
    LOOP["Live loop\n~60s/cycle"]
    EXEC["BinanceFuturesClient\nopen/close_long/short"]
    JOURNAL["trading.db\nDuckDB journal"]

    OOS --> CAL
    CAL --> ART
    OOS --> SWEEP
    SWEEP -.->|manuális döntés| ART
    ART --> SVC
    DB --> LOOP
    SVC --> LOOP
    LOOP --> EXEC
    EXEC --> JOURNAL
    LOOP --> JOURNAL
```

---

## Modul struktúra

```
src/trading/
├── calibration/
│   ├── backtest.py         load_oos_frame(), simulate_long/short, summarize_trades, write_backtest_report
│   ├── calibrate.py        run_calibration() orchestrátor
│   └── artifacts.py        write_strategy_artifact(), load_strategy_artifact()
├── live/
│   ├── service.py          TradingService — főciklus, life cycle
│   ├── exchange.py         BinanceFuturesClient — Binance Futures API wrapper
│   ├── journal.py          DuckDB journal (5 tábla)
│   ├── state.py            TradingState dataclass
│   └── strategy.py         evaluate() state machine
├── 00_calibrate_strategy.py   CLI: single-pass kalibrálás
├── 01_sweep_strategy.py       CLI: paraméter sweep
└── 02_run_service.py          CLI: live/dry_run service indítás
```

---

## Entry point scriptek

| Script | Paraméterek | Kimenet |
|--------|-------------|---------|
| `00_calibrate_strategy.py` | `--model <model_id>` (kötelező), `--start YYYY-MM-DD`, `--end YYYY-MM-DD` | `artifacts/<model_id>/strategy/strategy_artifact.json`, `trades.csv`, `equity_curve.csv`, `report.html` |
| `01_sweep_strategy.py` | `--model <model_id>` (kötelező), `--start YYYY-MM-DD`, `--end YYYY-MM-DD`, `--top-n N` (alap: 20) | `artifacts/<model_id>/strategy/sweep_results.csv`, stdout tábla |
| `02_run_service.py` | `--mode dry_run\|live` (alap: config-ból) | Futó TradingService, `trading.db` journal |

---

## Fejezetek

| Fájl | Tartalom |
|------|----------|
| [6100_calibration.md](6100_calibration.md) | Calibration almodul — backtest, artifacts, run_calibration |
| [6200_live_service.md](6200_live_service.md) | Live service — TradingService, state machine, journal, exchange |




---

<!-- Source: 6100_calibration.md -->



# calibration/ — Kalibrációs Almodul

A `src/trading/calibration/` könyvtár olvassa az OOS predikciós fájlokat, szimulál egy visszatesztelést, kiszámítja a teljesítménymutatókat, és lementi a stratégia-artefaktumot.

---

## Áttekintés

```{mermaid}
sequenceDiagram
    participant CLI as 00_calibrate_strategy.py
    participant CAL as calibrate.py
    participant BT  as backtest.py
    participant ART as artifacts.py
    participant FS  as Fájlrendszer

    CLI->>CAL: run_calibration(model_id, start, end)
    CAL->>FS: manifest.json olvasás → side
    CAL->>BT: load_oos_frame(model_id, start, end)
    BT->>FS: sample_oos.parquet olvasás
    BT->>FS: DuckDB OHLCV lekérdezés
    BT-->>CAL: frame (DataFrame)
    CAL->>BT: simulate_long_probability_strategy(frame, cfg)
    BT-->>CAL: trades_df, equity_df, summary
    CAL->>BT: write_backtest_report(out_dir, ...)
    CAL->>ART: write_strategy_artifact(model_id, side, cfg, summary, oos_period)
    ART->>FS: strategy_artifact.json írás
    ART-->>CAL: artifact_path
    CAL-->>CLI: result dict
```

---

## backtest.py

### `load_oos_frame(model_id, start, end)`

OOS predikciók betöltése és join OHLCV adatokkal.

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `model_id` | `str` | Artifact könyvtár neve, pl. `lgbm_solusdt_l_fw60_2021` |
| `start` | `str \| None` | OOS kezdő dátum (`YYYY-MM-DD`), None = legkorábbi sor |
| `end` | `str \| None` | OOS záró dátum (`YYYY-MM-DD`), None = legutolsó sor |

**Visszatérési érték:** `pd.DataFrame` — oszlopok: `open_time, open, high, low, close, target, prediction`

**Működés:**
1. Beolvassa `artifacts/<model_id>/manifest.json`-t — meghatározza a side-ot (`long`/`short`) és a predikciós oszlop nevét (`pred_long` / `pred_short`).
2. Betölti `artifacts/<model_id>/sample_oos.parquet`-ot, szűri a dátumtartományra.
3. Betölti a DuckDB OHLCV tábla megfelelő tartományát (`duckdb_query.query_range`).
4. Inner join-t végez `open_time`-on, deduplication + sort után adja vissza.

**Kivételek:** `FileNotFoundError` (hiányzó parquet vagy manifest), `ValueError` (ismeretlen `target_name`, üres tartomány, hiányzó pred oszlop).

---

### `simulate_long_probability_strategy(frame, strategy_cfg)`

Valószínűségi küszöb alapú LONG/FLAT stratégia szimulátora. Ha `strategy_cfg["side"] == "short"`, automatikusan a short szimulátorra (`_simulate_short_probability_strategy`) delegál.

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `frame` | `pd.DataFrame` | `load_oos_frame` kimenete |
| `strategy_cfg` | `dict` | Stratégia paraméterek (lásd lent) |

**Visszatérési érték:** `tuple[pd.DataFrame, pd.DataFrame, dict]` — `(trades_df, equity_df, summary_dict)`

**`strategy_cfg` kulcsok:**

| Kulcs | Alap | Leírás |
|-------|------|--------|
| `side` | `"long"` | `"long"` vagy `"short"` |
| `entry_threshold` | — (kötelező) | Belépési valószínűségi küszöb |
| `rearm_threshold` | = `entry_threshold` | Újrafegyverkezési küszöb (rearm) |
| `exit_threshold` | `-1.0` | Kilépési valószínűségi küszöb |
| `min_hold_minutes` | `0` | Minimális tartási idő |
| `max_hold_minutes` | `240` | Maximális tartási idő |
| `take_profit_pct` | `0.0` | Take profit ár % (0 = letiltva) |
| `stop_loss_pct` | `0.0` | Stop loss ár % (0 = letiltva) |
| `trailing_activation_pct` | `0.0` | Trailing stop aktiválási % |
| `trailing_stop_pct` | `0.0` | Trailing stop követési % |
| `cooldown_minutes` | `0` | Kilépés utáni várakozás percekben |
| `fee_bps_per_side` | `0.0` | Kereskedési díj bázispontban (oldalanként) |
| `slippage_bps_per_side` | `0.0` | Csúszás bázispontban (oldalanként) |
| `initial_equity` | `10000.0` | Kezdeti tőke |

**Belépési logika (LONG):**
- Az előző bar `prediction >= entry_threshold` → belépés a jelenlegi bar `open`-ján + slippage
- Rendszer csak `armed == True` esetén lép be
- Cooldown lejárta + `prediction <= rearm_threshold` esetén a rendszer újra fegyverkezik

**Kilépési logika (LONG) prioritás sorrendben:**
1. `low <= hard_stop` → `stop_loss`
2. `high >= take_profit` → `take_profit`
3. Trailing stop aktiválva + `low <= trailing_stop_price` → `trailing_stop`
4. `hold_minutes >= max_hold_minutes` → `max_hold`
5. `hold_minutes >= min_hold_minutes` és `prediction <= exit_threshold` → `probability_exit`

**`trades_df` oszlopok:** `entry_time, entry_signal_time, exit_time, exit_reason, hold_minutes, entry_price, exit_price, entry_prediction, entry_target, gross_return, net_return, equity_after`

---

### `_simulate_short_probability_strategy(frame, strategy_cfg)` (belső)

A long szimulátor tükörképe — SHORT/FLAT stratégiát szimulál. A P&L pozitív, ha az ár a belépés után esik.

Megegyező paraméterek és visszatérési típus mint a long verziónál. Különbségek:
- Belépéskor a slippage az ár **alá** tolódik (short slippage iránya fordított)
- Stop loss: `high >= entry_price * (1 + stop_loss_pct)`
- Take profit: `low <= entry_price * (1 - take_profit_pct)`
- Trailing: `lowest_price` és `trailing_cover_price` nyomkövetés

---

### `summarize_trades(trades_df, equity_df, initial_equity, start_time, end_time)`

Trade-szintű és equity-szintű backtest metrikák kiszámítása.

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `trades_df` | `pd.DataFrame` | Trade rekordok |
| `equity_df` | `pd.DataFrame` | Equity görbe (`open_time`, `equity`) |
| `initial_equity` | `float` | Kezdeti tőke |
| `start_time` | `pd.Timestamp` | Első bar időbélyege |
| `end_time` | `pd.Timestamp` | Utolsó bar időbélyege |

**Visszatérési érték:** `dict` — metrikák:

| Kulcs | Leírás |
|-------|--------|
| `trade_count` | Kötések száma |
| `winning_trades` / `losing_trades` | Nyerő/vesztes kötések |
| `win_rate` | Nyerési arány (0.0–1.0) |
| `avg_net_return` / `median_net_return` | Átlag/medián nettó hozam |
| `best_trade` / `worst_trade` | Legjobb/legrosszabb kötés |
| `profit_factor` | Bruttó nyereség / bruttó veszteség |
| `total_return` | Összesített hozam |
| `profit` | Abszolút nyereség USDT-ben |
| `max_drawdown` | Maximális drawdown (negatív) |
| `avg_hold_minutes` / `median_hold_minutes` | Tartási idők |
| `avg_minutes_between_entries` | Átlag idő kötések között |
| `exposure_pct` | Piaci expozíció aránya (tartási / összes idő) |
| `exit_reasons` | `dict` — kilépési okok és darabszámok |

---

### `write_backtest_report(output_dir, strategy_id, strategy_cfg, summary, trades_df)`

HTML backtest riport írása.

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `output_dir` | `Path` | Célkönyvtár |
| `strategy_id` | `str` | Riport cím / modell azonosító |
| `strategy_cfg` | `dict` | Stratégia konfiguráció |
| `summary` | `dict` | `summarize_trades` kimenete |
| `trades_df` | `pd.DataFrame` | Trade rekordok |

**Kimenet:** `output_dir/report.html` — tartalmaz strategy config, summary tábla, exit reasons, utolsó 50 trade.

---

## calibrate.py

### `run_calibration(model_id, start, end)`

Egymenetes kalibrációs orchestrátor.

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `model_id` | `str` | Artifact könyvtár neve |
| `start` | `str \| None` | OOS kezdő dátum override |
| `end` | `str \| None` | OOS záró dátum override |

**Visszatérési érték:** `dict` — kulcsok: `model_id, side, oos_period, strategy, metrics, artifact_path`

**Alapértelmezett stratégia paraméterek (`_DEFAULT_STRATEGY`):**

| Paraméter | Érték |
|-----------|-------|
| `entry_threshold` | `0.45` |
| `rearm_threshold` | `0.18` |
| `exit_threshold` | `0.10` |
| `min_hold_minutes` | `5` |
| `max_hold_minutes` | `120` |
| `take_profit_pct` | `0.0` |
| `stop_loss_pct` | `0.0` |
| `trailing_activation_pct` | `0.0` |
| `trailing_stop_pct` | `0.0` |
| `cooldown_minutes` | `60` |
| `fee_bps_per_side` | `10.0` |
| `slippage_bps_per_side` | `2.0` |
| `initial_equity` | `10000.0` |

**Folyamat:**
1. Manifest alapján meghatározza a side-ot
2. `load_oos_frame` → frame
3. `simulate_long_probability_strategy` → trades_df, equity_df, summary
4. Ha vannak kötések: `trades.csv`, `equity_curve.csv`, `report.html` mentés
5. `write_strategy_artifact` → `strategy_artifact.json`

---

## artifacts.py

### `write_strategy_artifact(model_id, side, strategy_cfg, summary, oos_period)`

Stratégia-artefaktum írása JSON formátumban.

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `model_id` | `str` | Artifact könyvtár neve |
| `side` | `str` | `"long"` vagy `"short"` |
| `strategy_cfg` | `dict` | Stratégia paraméterek |
| `summary` | `dict` | `summarize_trades` kimenete |
| `oos_period` | `dict` | `{"start": "YYYY-MM-DD", "end": "YYYY-MM-DD"}` |

**Visszatérési érték:** `Path` — a megírt `strategy_artifact.json` elérési útja.

**Helyszín:** `artifacts/<model_id>/strategy/strategy_artifact.json` (felülírja ha létezik)

---

### `load_strategy_artifact(model_id)`

Betölti a `strategy_artifact.json` fájlt.

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `model_id` | `str` | Artifact könyvtár neve |

**Visszatérési érték:** `dict` — a teljes artifact JSON tartalma.

**Kivételek:** `FileNotFoundError` ha a fájl hiányzik.

---

## Strategy artifact JSON schema

| Mező | Típus | Szemantika |
|------|-------|-----------|
| `model_id` | `str` | Modell azonosító |
| `side` | `str` | `"long"` vagy `"short"` |
| `calibrated_at` | `str` | ISO 8601 UTC timestamp (kalibrálás időpontja) |
| `oos_period.start` | `str` | OOS periódus kezdete (`YYYY-MM-DD`) |
| `oos_period.end` | `str` | OOS periódus vége (`YYYY-MM-DD`) |
| `strategy` | `dict` | Teljes `strategy_cfg` (entry_threshold, max_hold_minutes stb.) |
| `metrics.trade_count` | `int` | Kötések száma az OOS periódusban |
| `metrics.win_rate` | `float \| null` | Nyerési arány |
| `metrics.total_return` | `float` | Összesített hozam (arány) |
| `metrics.profit` | `float \| null` | Abszolút nyereség USDT-ben |
| `metrics.profit_factor` | `float \| null` | Profit factor |
| `metrics.max_drawdown` | `float` | Maximális drawdown (negatív) |
| `metrics.avg_hold_minutes` | `float \| null` | Átlag tartási idő |
| `metrics.exposure_pct` | `float` | Piaci expozíció aránya |
| `metrics.initial_equity` | `float` | Kezdeti tőke |
| `metrics.final_equity` | `float \| null` | Záró tőke |
| `metrics.exit_reasons` | `dict` | Kilépési okok darabszámmal |

**Példa elérési út:** `artifacts/lgbm_solusdt_l_fw60_2021/strategy/strategy_artifact.json`




---

<!-- Source: 6200_live_service.md -->



# live/ — Live Trading Service

A `src/trading/live/` könyvtár kezeli az élő (és dry_run) kereskedést: percenkénti adatszinkront végez, predikció alapján dönt a stratégia state machine segítségével, megbízásokat ad Binance Futures-ön, és minden eseményt naplóz egy DuckDB journalba.

---

## Áttekintés — állapotgép

```{mermaid}
stateDiagram-v2
    [*] --> FLAT : startup / reconcile

    FLAT --> LONG  : pred_long >= entry_threshold\n(armed == True)
    FLAT --> SHORT : pred_short >= entry_threshold\n(armed == True, long nem triggerelt)
    FLAT --> FLAT  : HOLD (below threshold / not armed)

    LONG --> COOLDOWN : EXIT_LONG\n(max_hold / probability_exit / opposite_signal)
    SHORT --> COOLDOWN : EXIT_SHORT\n(max_hold / probability_exit / opposite_signal)

    COOLDOWN --> FLAT : cooldown lejárt ÉS\nboth predictions below rearm_threshold
    COOLDOWN --> COOLDOWN : HOLD (várakozás)
```

---

## service.py — TradingService

### `__init__(config)`

Inicializálja a service-t a trading config dict alapján.

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `config` | `dict` | Teljes trading konfiguráció (`config/trading.json`) |

**Inicializált attribútumok:**

| Attribútum | Leírás |
|-----------|--------|
| `mode` | `"dry_run"` vagy `"live"` |
| `asset_id` | Kereskedett asset azonosítója |
| `long_cfg` / `short_cfg` | Stratégia paraméter dict-ek (`config/strategies.json`-ból) |
| `long_pred_col` / `short_pred_col` | Predikció oszlop nevek: `"long_pred"`, `"short_pred"` |
| `exchange` | `BinanceFuturesClient` példány |
| `state` | `TradingState \| None` (startup után töltődik) |
| `run_id` | Futás azonosítója (startup után töltődik) |

---

### `start()`

Elindítja a trading loop-ot háttérszálban (`daemon=True`, szál neve: `chronoquant-trading`).

---

### `stop()`

Jelzést küld a graceful leállításhoz (`_stop_event.set()`). Az aktuális ciklus befejezése után a loop megáll.

---

### `is_running()`

**Visszatérési érték:** `bool` — `True` ha a service még nem lett leállítva.

---

### `_startup()` (belső)

1. `journal.ensure_tables` — táblák létrehozása ha hiányoznak
2. Unique `run_id` generálás (`run_YYYYMMDD_HHMMSS_<6hex>`)
3. `journal.insert_run` — futás naplózása
4. `journal.get_open_position` — nyitott pozíció reconcile DB-ből
5. `TradingState.from_db` — state rekonstrukció
6. `exchange.set_leverage` — tőkeáttétel beállítása

---

### `_cycle()` (belső)

Egy 1-perces bar feldolgozása. Sorrendben:

1. `_sync_data()` — OHLCV + features + predictions szinkron (`run_database_sync`)
2. `_read_latest_bar()` → `(bar_open_time, pred_long, pred_short, close)` vagy `None`
3. `_apply_cooldown_rearm()` — COOLDOWN → FLAT átmenet és `armed` flag kezelése
4. `strategy.evaluate()` → `(decision, reason)`
5. `_execute()` — pozíció nyitás/zárás az exchange-en
6. `journal.insert_signal()` — jel naplózása
7. `state.consecutive_errors = 0` — hiba számláló reset

---

### `_shutdown()` (belső)

1. `journal.mark_run_stopped` — futás lezárása
2. Ha `journal_cfg["export_on_stop"]`: `journal.export_run` → CSV export

---

## strategy.py — State Machine

### `evaluate(state, pred_long, pred_short, long_cfg, short_cfg, now)`

Egy lezárt bar stratégiai értékelése. **Nem módosítja a state-et** — a hívó alkalmazza a döntést.

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `state` | `TradingState` | Jelenlegi állapot |
| `pred_long` | `float` | Long modell predikciós valószínűsége |
| `pred_short` | `float` | Short modell predikciós valószínűsége |
| `long_cfg` | `dict` | Long stratégia paraméterek |
| `short_cfg` | `dict` | Short stratégia paraméterek |
| `now` | `datetime \| None` | Referencia UTC idő (alapért.: `datetime.now(UTC)`) |

**Visszatérési érték:** `tuple[str, str]` — `(decision, reason)`

**Decision értékek:** `HOLD`, `ENTER_LONG`, `ENTER_SHORT`, `EXIT_LONG`, `EXIT_SHORT`

**Döntési logika állapotonként:**

| Állapot | Feltétel | Döntés |
|---------|----------|--------|
| `COOLDOWN` | `now < cooldown_until` | `HOLD` "cooldown Nmin remaining" |
| `COOLDOWN` | lejárt + `pred_long <= rearm` és `pred_short <= rearm` | `HOLD` "rearm_triggered" |
| `COOLDOWN` | lejárt, de nem rearm | `HOLD` "waiting_rearm" |
| `FLAT` | `not state.armed` | `HOLD` "not_armed" |
| `FLAT` | `pred_long >= entry_threshold` | `ENTER_LONG` |
| `FLAT` | `pred_short >= entry_threshold` (long nem triggerelt) | `ENTER_SHORT` |
| `LONG` | `hold_min >= max_hold_minutes` | `EXIT_LONG` "max_hold" |
| `LONG` | `pred_short >= short entry_threshold` | `EXIT_LONG` "opposite_signal" |
| `LONG` | `hold_min >= min_hold` és `pred_long <= exit_threshold` | `EXIT_LONG` "probability_exit" |
| `SHORT` | `hold_min >= max_hold_minutes` | `EXIT_SHORT` "max_hold" |
| `SHORT` | `pred_long >= long entry_threshold` | `EXIT_SHORT` "opposite_signal" |
| `SHORT` | `hold_min >= min_hold` és `pred_short <= exit_threshold` | `EXIT_SHORT` "probability_exit" |

---

## state.py — TradingState

### `TradingState` dataclass

Mutable runtime állapot egy trading service futáshoz.

| Mező | Típus | Leírás |
|------|-------|--------|
| `status` | `str` | Jelenlegi állapot: `FLAT`, `LONG`, `SHORT`, `COOLDOWN` |
| `armed` | `bool` | True = kész belépni új pozícióba |
| `cooldown_until` | `datetime \| None` | Legkorábbi újrafegyverkezési idő |
| `position_id` | `str \| None` | Aktív pozíció azonosítója (journalból) |
| `side` | `str \| None` | Jelenlegi pozíció oldala (`LONG` / `SHORT`) |
| `entry_time` | `datetime \| None` | Pozíció nyitásának UTC időpontja |
| `entry_price` | `float \| None` | Belépési végrehajtási ár |
| `quantity` | `float \| None` | Pozíció mérete (base asset egységben) |
| `run_id` | `str \| None` | Futás azonosítója |
| `daily_trade_count` | `int` | Mai nyitott kötések száma |
| `daily_loss_usdt` | `float` | Mai kumulált veszteség USDT-ben |
| `consecutive_errors` | `int` | Egymást követő hibák száma |
| `last_trade_date` | `str \| None` | Utolsó kötés dátuma (`YYYY-MM-DD`) |

**Állandók:** `FLAT = "FLAT"`, `LONG = "LONG"`, `SHORT = "SHORT"`, `COOLDOWN = "COOLDOWN"`

**Metódusok:**

| Metódus | Leírás |
|---------|--------|
| `hold_minutes(now)` | Jelenlegi pozíció tartási ideje percben |
| `clear_position()` | Összes pozíció mező nullázása |
| `record_trade_result(pnl_usdt)` | Napi risk számlálók frissítése, napi reset ha új nap |
| `from_db(run_id, open_position)` | Classmethod — state rekonstrukció DB sorból |

---

## journal.py — DuckDB Journal

Az összes live trading eseményt egy külön `trading.db` DuckDB fájlba írja. Minden írás tranzakcionális (`BEGIN` / `COMMIT` / `ROLLBACK`).

**DB elérési út:** `utils.load_trading_config()["db_path"]` → `trading_db_path()`

### Táblák

#### `trading_runs`

| Oszlop | Típus | Leírás |
|--------|-------|--------|
| `run_id` | `TEXT PK` | Egyedi futás azonosító |
| `started_at` | `TEXT` | Indulás UTC timestamp |
| `stopped_at` | `TEXT \| NULL` | Leállás UTC timestamp |
| `mode` | `TEXT` | `"dry_run"` vagy `"live"` |
| `asset_id` | `TEXT` | Kereskedett asset |
| `long_strategy_id` | `TEXT` | Long stratégia azonosítója |
| `short_strategy_id` | `TEXT` | Short stratégia azonosítója |
| `config_json` | `TEXT` | Teljes config JSON |

#### `trading_signals`

| Oszlop | Típus | Leírás |
|--------|-------|--------|
| `id` | `BIGINT PK` | Auto-increment (sequence) |
| `run_id` | `TEXT` | Futás azonosítója |
| `bar_open_time` | `TEXT` | Bar nyitási időpontja |
| `pred_long` | `REAL` | Long predikciós valószínűség |
| `pred_short` | `REAL` | Short predikciós valószínűség |
| `state_before` | `TEXT` | Állapot a döntés előtt |
| `decision` | `TEXT` | Döntés (HOLD, ENTER_LONG stb.) |
| `reason` | `TEXT` | Olvasható indoklás |
| `processed_at` | `TEXT` | Feldolgozás UTC timestamp |

#### `trading_positions`

| Oszlop | Típus | Leírás |
|--------|-------|--------|
| `position_id` | `TEXT PK` | Egyedi pozíció azonosító |
| `run_id` | `TEXT` | Futás azonosítója |
| `side` | `TEXT` | `"LONG"` vagy `"SHORT"` |
| `status` | `TEXT` | `"OPEN"` vagy `"CLOSED"` |
| `entry_time` | `TEXT` | Nyitás UTC timestamp |
| `exit_time` | `TEXT \| NULL` | Zárás UTC timestamp |
| `entry_price` | `REAL` | Belépési ár |
| `exit_price` | `REAL \| NULL` | Kilépési ár |
| `quantity` | `REAL` | Pozíció mérete |
| `pnl_usdt` | `REAL \| NULL` | Realizált P&L USDT-ben |
| `exit_reason` | `TEXT \| NULL` | Zárás oka |
| `entry_order_id` | `TEXT \| NULL` | Belépési megbízás azonosítója |
| `exit_order_id` | `TEXT \| NULL` | Kilépési megbízás azonosítója |

#### `trading_orders`

| Oszlop | Típus | Leírás |
|--------|-------|--------|
| `order_id` | `TEXT PK` | Helyi megbízás azonosító |
| `run_id` | `TEXT` | Futás azonosítója |
| `position_id` | `TEXT \| NULL` | Kapcsolódó pozíció |
| `side` | `TEXT` | `"BUY"` vagy `"SELL"` |
| `order_type` | `TEXT` | Pl. `"MARKET"` |
| `status` | `TEXT` | Pl. `"FILLED"` |
| `client_order_id` | `TEXT \| NULL` | Kliens oldali megbízás ID |
| `binance_order_id` | `TEXT \| NULL` | Binance megbízás ID |
| `requested_qty` | `REAL \| NULL` | Kért mennyiség |
| `filled_qty` | `REAL \| NULL` | Ténylegesen teljesített mennyiség |
| `avg_price` | `REAL \| NULL` | Átlag teljesítési ár |
| `request_json` | `TEXT \| NULL` | Nyers kérés (JSON) |
| `response_json` | `TEXT \| NULL` | Nyers exchange válasz (JSON) |
| `created_at` | `TEXT` | Létrehozás UTC timestamp |

#### `trading_errors`

| Oszlop | Típus | Leírás |
|--------|-------|--------|
| `id` | `BIGINT PK` | Auto-increment (sequence) |
| `run_id` | `TEXT \| NULL` | Futás azonosítója |
| `error_time` | `TEXT` | Hiba UTC timestamp |
| `component` | `TEXT` | Komponens neve (pl. `"cycle"`, `"execute"`) |
| `error_type` | `TEXT` | Exception osztály neve |
| `message` | `TEXT` | Exception üzenet |
| `traceback` | `TEXT \| NULL` | Teljes traceback |

### Journal függvények

| Függvény | Leírás |
|----------|--------|
| `ensure_tables(db_path)` | Táblák és sequence-ek létrehozása |
| `insert_run(...)` | Új futás naplózása |
| `mark_run_stopped(db_path, run_id)` | `stopped_at` beállítása |
| `insert_signal(...)` | Jel naplózása |
| `insert_position(...)` | Nyitott pozíció beírása |
| `close_position(...)` | Pozíció lezárása (UPDATE) |
| `get_open_position(db_path)` | Legutóbbi nyitott pozíció lekérése |
| `get_latest_run(db_path)` | Legutóbbi futás lekérése |
| `insert_order(...)` | Megbízás naplózása |
| `insert_error(...)` | Hiba naplózása (soha nem dob kivételt) |
| `get_recent_signals(db_path, limit)` | Legutóbbi jelek (dashboard) |
| `get_recent_positions(db_path, limit)` | Legutóbbi pozíciók (dashboard) |
| `get_current_run_status(db_path)` | Összesített status dict (dashboard) |
| `export_run(db_path, run_id, report_dir)` | CSV export leálláskor |

---

## exchange.py — BinanceFuturesClient

### `__init__(symbol, leverage, quote_order_qty, mode)`

| Paraméter | Típus | Leírás |
|-----------|-------|--------|
| `symbol` | `str` | Binance szimbólum, pl. `"SOLUSDT"` |
| `leverage` | `int` | Futures tőkeáttétel szorzó |
| `quote_order_qty` | `float` | Nominális megbízás méret USDT-ben |
| `mode` | `str` | `"dry_run"` vagy `"live"` |

**`dry_run` mód:** valódi megbízások nélkül, mark price-on szimulált teljesítés. Az exchange API-t csak price lekérdezésre hívja.

**`live` mód:** aláírt Binance Futures MARKET megbízások a `python-binance` kliens via API kulcsok (`config/env.json`-ban konfigurált elérési út).

### Megbízás metódusok

| Metódus | Leírás | Visszatérési érték |
|---------|--------|-------------------|
| `set_leverage()` | Tőkeáttétel beállítás (dry_run: no-op) | — |
| `get_mark_price()` | Mark price lekérdezés | `float` |
| `open_long(mark_price)` | BUY MARKET megbízás | order response `dict` |
| `close_long(quantity, mark_price)` | Reduce-only SELL MARKET | order response `dict` |
| `open_short(mark_price)` | SELL MARKET megbízás | order response `dict` |
| `close_short(quantity, mark_price)` | Reduce-only BUY MARKET | order response `dict` |

**Quantity számítás:** `qty = round_down((quote_order_qty * leverage) / mark_price, step=0.1)`

**Dry fill response formátum:**
```json
{
    "orderId": "DRY_<timestamp_ms>",
    "clientOrderId": "CQ_DRY_<8hex>",
    "symbol": "SOLUSDT",
    "side": "BUY|SELL",
    "type": "MARKET",
    "status": "FILLED",
    "executedQty": "<qty>",
    "avgPrice": "<mark_price>",
    "dry_run": true
}
```

---

## 02_run_service.py CLI

```bash
uv run python src/trading/02_run_service.py [--mode dry_run|live]
```

| Argument | Alap | Leírás |
|----------|------|--------|
| `--mode` | config-ból | `"dry_run"` vagy `"live"` — felülírja a `config/trading.json` mode mezőjét |

**Live mód megerősítés:** `live` módban interaktív `"yes"` megerősítés szükséges a folytatáshoz.

**Jelkezelés:** `SIGINT` / `SIGTERM` → `service.stop()` → graceful shutdown.
